# Global SST: an additional chronological comparison

This notebook tests whether **eight global SST principal components** add information to the fixed rainfall reference model. It evaluates January 2021–December 2022, with training capped at September 2020. It does not fit a neural network.

## Why this experiment

The existing seven-block 2007–2020 comparison already used SST T−2 and training-only PCA. SST reduced pooled RMSE from **1.753399 to 1.752762 mm/day**, but increased MAE from 1.047213 to 1.048663 and signed bias from +0.004916 to +0.017217. There is little justification for repeating that run unchanged.

This extension keeps its SST method fixed and checks an additional period. **2021–2022 has already been consulted in the reference model/U-Net extension and earlier final fits. It is not an untouched holdout.** Results cannot establish future forecast skill.

## Data and temporal boundaries

Attach four Kaggle Inputs:

1. The original competition training files, including `treino_tp.nc` and the nine atmospheric files.
2. The prepared SEAS5 dataset, containing `seas5_51_manifesto.json` and its development and final-fit NetCDF partitions.
3. The prepared CFSv2 dataset, containing `cfsv2_manifesto.json` and the same two partitions.
4. The original SST snapshot: `ersstv5_4graus_198201_202411.nc`. SHA-256 must be `908034f6418833302ea432f25850ad2982d1114b747b7b298e26a350eaf2891d`.

The fourth file is the prepared NOAA ERSSTv5 4° subset, 60°S–60°N, rather than a new download of a possibly revised product. The source is [NOAA PSL ERSSTv5](https://psl.noaa.gov/data/gridded/data.noaa.ersst.v5.html). The archived NOAA index table is bundled and checked against the earlier snapshot. No API keys or Internet access are needed. CPU is sufficient; leave the accelerator disabled.

| Quantity | Rule |
|---|---|
| Rainfall training targets | October 1990–September 2020, exactly 360 months |
| Atmospheric fields | Target month T−4 |
| Ocean indices | T−3 |
| SST | T−2 |
| Seasonal initialization | T−1 |
| PCA fitting origins | August 1990–July 2020 |
| SST origins for evaluation | November 2020–October 2022 |

The ocean mask, monthly SST means, centering and PCA components are fitted only on training origins. Anomalies receive sqrt(cos(latitude)) weights; no pixelwise variance standardization is added. Missing selected ocean cells stop execution. Later rainfall labels are decoded only after both prediction files have been frozen.

These monthly lags are an availability assumption for retrospective evaluation. The archived products do **not** establish which historical revision was available on each issue date. This notebook neither emits an operational forecast nor changes the live source-receipt registry.

## Paired comparison

Both treatments use exactly the same sampled pixels, rainfall labels, base features and climatologies. Eight PC scores are appended only to the two LightGBM components: 29/31 predictors become 37/39. The local ridge is fitted once and shared. Each tree model keeps 300 trees, 31 leaves, learning rate 0.05 and seed 42; weights remain 0.375 / 0.375 / 0.25 and residual scales 0.9 / 0.875.

Budget: **one block, four LightGBM fits, one shared ridge, one PCA, zero neural fits**. Progress reports each tree component's fit and prediction time. Runtime has not been measured for this new notebook; input loading, hash checks and inference also take time.

The reconstructed reference model must match its archived 2021–2022 RMSE **1.7967836119** and the other recorded global metrics within 10⁻⁶. A mismatch is reported and blocks interpreting the run as a clean SST comparison.

## Running and reading the output

Import this notebook into a **new Kaggle notebook**, attach the four Inputs and select **Run All**. Use the same reference versions: NumPy 2.0.2, pandas 2.3.3, xarray 2025.12.0, LightGBM 4.6.0, scikit-learn 1.6.1 and SciPy 1.16.3. Versions and source hashes are recorded. Package versions are not upgraded automatically.

The first checks use synthetic fields: tiny grids for model fits and constant full-grid maps for output validation. Their printed scores are artificial. The next cell performs the real paired fit, and the last opens evaluation rainfall. Read `baseline_reproduced`, pooled RMSE, MAE, signed bias, both years and monthly/regional differences together. No weight search, PCA-size search or automatic adoption follows the score.

Download **`sst_temporal_extension_reports.zip`** and preserve the full Kaggle output. The small ZIP contains reports, source snapshots and hashes; model weights, PCA state and prediction maps remain in the full output. `complete_extension: true` describes this one 24-month extension; `complete_seven_blocks` is false by design.

An unchanged completed fit can be verified and reused. An incomplete run is preserved and requires a new `OUTPUT` directory; it is never silently overwritten.


In [ ]:
from pathlib import Path
import hashlib, json, sys, runpy, shutil
BASE = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()/'outputs'
RUN_ROOT = BASE/'sst_temporal_extension'
CODE_ROOT = RUN_ROOT/'code'
CODE_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE = {'competition/metadata/NOAA/indices_noaa.csv': 'time_origem,nino34,nino12,tna,tsa\n1976-12-01,0.64,0.25,-0.60,-0.47\n1977-01-01,0.84,0.29,-0.40,-0.53\n1977-02-01,0.55,-0.14,-0.40,-0.27\n1977-03-01,0.40,-0.38,-0.27,-0.34\n1977-04-01,-0.25,-0.68,-0.25,-0.34\n1977-05-01,-0.03,-0.45,-0.28,-0.39\n1977-06-01,0.36,0.08,-0.20,-0.41\n1977-07-01,0.45,0.03,-0.42,-0.40\n1977-08-01,0.25,-0.52,-0.62,-0.19\n1977-09-01,0.44,-0.50,-0.61,-0.12\n1977-10-01,0.89,-0.19,-0.66,-0.31\n1977-11-01,1.03,-0.29,-0.57,-0.27\n1977-12-01,1.09,-0.56,-0.44,-0.34\n1978-01-01,0.72,-0.07,-0.13,-0.52\n1978-02-01,0.51,-0.35,0.02,-0.56\n1978-03-01,0.06,-0.77,0.09,-0.80\n1978-04-01,-0.37,-0.37,0.10,-0.96\n1978-05-01,-0.50,-0.54,-0.13,-0.99\n1978-06-01,-0.45,-0.86,-0.39,-0.78\n1978-07-01,-0.48,-0.40,-0.60,-0.33\n1978-08-01,-0.62,-0.64,-0.66,-0.40\n1978-09-01,-0.60,-0.30,-0.59,-0.50\n1978-10-01,-0.34,-0.43,-0.35,-0.61\n1978-11-01,-0.15,0.04,-0.30,-0.61\n1978-12-01,0.07,0.23,-0.48,-0.43\n1979-01-01,0.05,0.23,-0.08,-0.49\n1979-02-01,0.09,-0.58,0.10,-0.51\n1979-03-01,0.13,-0.25,-0.01,-0.65\n1979-04-01,0.17,0.51,0.14,-0.30\n1979-05-01,-0.09,0.27,0.15,-0.27\n1979-06-01,0.04,0.76,0.18,-0.21\n1979-07-01,-0.20,0.45,0.01,-0.12\n1979-08-01,0.21,0.50,-0.25,0.07\n1979-09-01,0.48,1.02,-0.27,0.14\n1979-10-01,0.32,0.67,-0.32,-0.17\n1979-11-01,0.35,0.12,-0.20,-0.12\n1979-12-01,0.69,-0.34,-0.03,-0.28\n1980-01-01,0.55,-0.10,0.15,-0.48\n1980-02-01,0.33,-0.25,0.23,-0.61\n1980-03-01,0.09,0.02,0.02,-0.61\n1980-04-01,0.20,-0.00,0.40,-0.42\n1980-05-01,0.13,0.25,0.50,-0.38\n1980-06-01,0.37,0.21,0.23,-0.39\n1980-07-01,0.02,-0.44,-0.08,-0.31\n1980-08-01,-0.23,-0.44,-0.04,-0.16\n1980-09-01,-0.12,-0.11,-0.17,-0.04\n1980-10-01,-0.11,-0.52,-0.38,-0.25\n1980-11-01,0.10,-0.68,-0.28,-0.43\n1980-12-01,0.36,-1.13,-0.31,-0.86\n1981-01-01,-0.46,-1.29,0.03,-0.61\n1981-02-01,-0.45,-0.83,-0.01,-0.73\n1981-03-01,-0.02,-0.19,0.45,-0.99\n1981-04-01,-0.17,-0.31,0.18,-0.83\n1981-05-01,-0.11,-0.22,0.16,-0.75\n1981-06-01,-0.15,-0.06,-0.06,-0.53\n1981-07-01,-0.43,-0.56,-0.12,-0.48\n1981-08-01,-0.18,-0.55,-0.24,-0.07\n1981-09-01,-0.07,-0.24,-0.23,0.05\n1981-10-01,-0.03,-0.52,-0.39,0.12\n1981-11-01,-0.27,-0.55,-0.18,0.08\n1981-12-01,0.07,-0.17,-0.02,-0.16\n1982-01-01,-0.04,-0.12,-0.06,-0.30\n1982-02-01,-0.13,-0.35,-0.24,-0.47\n1982-03-01,-0.02,-0.72,-0.23,-0.56\n1982-04-01,0.24,-0.51,-0.32,-0.66\n1982-05-01,0.65,0.08,-0.24,-0.71\n1982-06-01,0.92,0.27,-0.40,-0.76\n1982-07-01,0.64,0.60,-0.54,-0.70\n1982-08-01,0.93,1.09,-0.64,-0.77\n1982-09-01,1.39,2.01,-0.66,-0.46\n1982-10-01,2.00,2.28,-0.77,-0.33\n1982-11-01,2.07,3.08,-0.77,-0.46\n1982-12-01,2.33,3.20,-0.79,-0.43\n1983-01-01,2.43,2.73,-0.61,-0.01\n1983-02-01,2.22,2.03,-0.19,0.11\n1983-03-01,1.69,2.34,0.28,-0.48\n1983-04-01,1.12,3.36,0.29,-0.56\n1983-05-01,1.12,4.10,-0.03,-0.80\n1983-06-01,0.62,4.56,0.10,-0.76\n1983-07-01,-0.11,3.93,-0.05,-0.56\n1983-08-01,-0.13,3.11,-0.33,-0.28\n1983-09-01,-0.50,2.07,-0.56,-0.11\n1983-10-01,-1.03,1.02,-0.58,-0.31\n1983-11-01,-1.13,0.56,-0.26,-0.12\n1983-12-01,-0.95,0.22,-0.06,0.12\n1984-01-01,-0.80,-0.29,-0.30,0.35\n1984-02-01,-0.53,-0.77,-0.38,0.38\n1984-03-01,-0.48,-0.41,-0.25,0.44\n1984-04-01,-0.60,-0.25,-0.34,0.19\n1984-05-01,-0.64,-1.11,-0.36,-0.08\n1984-06-01,-0.90,-0.92,-0.61,0.14\n1984-07-01,-0.40,-0.54,-0.63,0.29\n1984-08-01,-0.40,-0.52,-0.62,0.57\n1984-09-01,-0.35,-0.36,-0.62,0.69\n1984-10-01,-0.83,-0.61,-0.92,0.36\n1984-11-01,-1.18,-0.18,-0.77,0.13\n1984-12-01,-1.53,-0.50,-0.79,-0.01\n1985-01-01,-0.89,-0.77,-0.68,-0.02\n1985-02-01,-0.91,-1.13,-0.55,0.25\n1985-03-01,-0.96,-0.59,-0.77,-0.21\n1985-04-01,-0.98,-1.38,-0.81,-0.16\n1985-05-01,-0.90,-1.41,-0.91,-0.10\n1985-06-01,-0.76,-1.25,-0.54,-0.27\n1985-07-01,-0.61,-1.04,-0.32,0.02\n1985-08-01,-0.49,-1.02,-0.42,-0.10\n1985-09-01,-0.57,-0.83,-0.54,-0.13\n1985-10-01,-0.64,-0.82,-0.57,0.05\n1985-11-01,-0.43,-0.92,-0.39,0.06\n1985-12-01,-0.40,-0.75,-0.54,0.03\n1986-01-01,-0.79,-0.59,-0.86,-0.14\n1986-02-01,-0.64,-0.35,-0.80,0.20\n1986-03-01,-0.51,-0.84,-0.76,0.04\n1986-04-01,-0.41,-0.86,-0.80,-0.03\n1986-05-01,-0.47,-0.82,-0.74,-0.08\n1986-06-01,-0.08,-0.90,-0.84,0.10\n1986-07-01,0.16,-0.14,-0.65,-0.06\n1986-08-01,0.24,0.17,-0.74,0.01\n1986-09-01,0.59,-0.13,-0.34,0.16\n1986-10-01,0.90,0.14,-0.63,-0.08\n1986-11-01,1.04,0.28,-0.91,-0.35\n1986-12-01,0.98,0.53,-0.98,-0.24\n1987-01-01,1.19,0.77,-0.57,-0.32\n1987-02-01,1.17,1.02,-0.35,-0.19\n1987-03-01,1.24,1.15,0.06,-0.18\n1987-04-01,0.89,1.51,0.17,-0.27\n1987-05-01,0.91,1.35,0.17,-0.06\n1987-06-01,1.24,1.06,0.37,0.11\n1987-07-01,1.34,1.19,0.21,0.25\n1987-08-01,1.48,0.91,0.22,0.24\n1987-09-01,1.53,1.31,0.02,0.17\n1987-10-01,1.36,1.50,0.02,0.19\n1987-11-01,1.29,0.90,0.05,0.45\n1987-12-01,1.05,0.89,0.22,0.13\n1988-01-01,0.69,0.09,-0.09,0.30\n1988-02-01,0.35,-0.27,-0.20,0.39\n1988-03-01,0.29,-0.52,0.12,0.43\n1988-04-01,-0.49,-0.90,0.22,0.28\n1988-05-01,-1.05,-1.08,0.08,0.37\n1988-06-01,-1.46,-1.96,0.10,0.42\n1988-07-01,-1.54,-1.54,-0.00,0.20\n1988-08-01,-1.44,-1.59,-0.31,0.30\n1988-09-01,-1.33,-1.43,-0.32,0.16\n1988-10-01,-2.09,-1.26,-0.63,0.15\n1988-11-01,-2.18,-0.65,-0.54,-0.05\n1988-12-01,-1.98,-0.97,-0.52,-0.07\n1989-01-01,-1.95,-0.28,-0.67,0.08\n1989-02-01,-1.37,0.05,-0.66,-0.04\n1989-03-01,-1.33,-0.45,-0.82,0.00\n1989-04-01,-1.11,-0.49,-1.04,-0.02\n1989-05-01,-0.80,-1.16,-0.86,-0.01\n1989-06-01,-0.64,-0.73,-0.42,0.00\n1989-07-01,-0.47,-0.44,-0.03,0.00\n1989-08-01,-0.58,-0.46,0.05,0.19\n1989-09-01,-0.38,-0.47,-0.32,0.06\n1989-10-01,-0.40,-0.29,-0.30,0.06\n1989-11-01,-0.30,-0.38,-0.35,-0.24\n1989-12-01,-0.13,-0.43,-0.07,-0.51\n1990-01-01,0.02,-0.33,-0.33,-0.68\n1990-02-01,0.36,-0.27,-0.28,-0.14\n1990-03-01,0.20,-0.41,-0.20,-0.03\n1990-04-01,0.26,-0.33,-0.10,0.04\n1990-05-01,0.32,-0.23,-0.07,-0.40\n1990-06-01,0.00,-0.05,-0.27,-0.23\n1990-07-01,0.15,-0.87,-0.22,-0.37\n1990-08-01,0.17,-0.61,-0.04,-0.33\n1990-09-01,0.12,-0.28,0.02,-0.22\n1990-10-01,0.13,-0.73,-0.01,-0.13\n1990-11-01,0.08,-0.55,0.02,-0.01\n1990-12-01,0.32,-0.44,-0.12,-0.04\n1991-01-01,0.51,-0.09,-0.39,-0.21\n1991-02-01,0.32,-0.10,-0.29,-0.20\n1991-03-01,0.10,0.00,-0.37,-0.21\n1991-04-01,0.21,-0.68,-0.54,-0.00\n1991-05-01,0.44,-0.01,-0.62,0.14\n1991-06-01,0.65,0.24,-0.67,0.13\n1991-07-01,0.62,0.60,-0.67,-0.20\n1991-08-01,0.43,0.00,-0.62,-0.52\n1991-09-01,0.22,0.16,-0.52,-0.51\n1991-10-01,0.91,0.42,-0.53,-0.59\n1991-11-01,1.14,0.64,-0.46,-0.52\n1991-12-01,1.62,0.45,-0.47,-0.34\n1992-01-01,1.61,0.36,-0.24,-0.61\n1992-02-01,1.62,0.44,-0.14,-0.53\n1992-03-01,1.50,0.92,-0.34,-0.74\n1992-04-01,1.41,1.72,-0.44,-0.88\n1992-05-01,1.27,1.83,-0.39,-1.01\n1992-06-01,0.53,0.71,-0.36,-1.11\n1992-07-01,0.27,-0.05,-0.29,-0.86\n1992-08-01,-0.15,-0.24,-0.56,-0.86\n1992-09-01,-0.14,-0.50,-0.58,-0.70\n1992-10-01,-0.35,-0.26,-0.48,-0.74\n1992-11-01,-0.14,-0.25,-0.61,-0.97\n1992-12-01,0.04,-0.31,-0.52,-0.63\n1993-01-01,0.28,0.03,-0.50,-0.59\n1993-02-01,0.42,0.20,-0.43,-0.39\n1993-03-01,0.47,0.53,-0.38,-0.44\n1993-04-01,0.92,0.83,-0.22,-0.11\n1993-05-01,0.93,1.13,-0.25,-0.32\n1993-06-01,0.64,0.88,-0.18,-0.34\n1993-07-01,0.33,0.32,-0.44,-0.21\n1993-08-01,0.16,0.21,-0.45,-0.16\n1993-09-01,0.26,0.12,-0.50,0.30\n1993-10-01,0.36,0.32,-0.56,0.47\n1993-11-01,0.28,-0.23,-0.65,0.57\n1993-12-01,0.19,-0.29,-0.70,0.24\n1994-01-01,0.04,-0.07,-0.83,-0.01\n1994-02-01,-0.16,-0.33,-0.78,-0.14\n1994-03-01,-0.06,-1.13,-0.76,-0.10\n1994-04-01,0.03,-1.22,-0.62,-0.13\n1994-05-01,0.14,-0.91,-0.70,-0.23\n1994-06-01,0.27,-0.59,-0.67,-0.49\n1994-07-01,0.16,-0.82,-0.67,-0.52\n1994-08-01,0.52,-0.90,-0.63,-0.37\n1994-09-01,0.40,-0.05,-0.62,-0.30\n1994-10-01,0.90,0.65,-0.50,-0.27\n1994-11-01,1.14,0.59,-0.34,-0.11\n1994-12-01,1.21,0.68,-0.42,-0.07\n1995-01-01,1.10,0.91,-0.30,0.07\n1995-02-01,0.87,0.24,-0.22,0.34\n1995-03-01,0.49,-0.57,-0.21,0.39\n1995-04-01,0.27,-0.99,-0.04,0.29\n1995-05-01,0.03,-1.01,0.10,0.24\n1995-06-01,0.08,-0.53,0.23,0.33\n1995-07-01,0.03,-0.25,0.31,0.09\n1995-08-01,-0.38,-0.52,0.25,-0.26\n1995-09-01,-0.57,-0.28,0.03,-0.19\n1995-10-01,-0.72,-0.41,-0.06,-0.27\n1995-11-01,-0.78,-0.26,0.20,-0.06\n1995-12-01,-0.72,-0.69,0.17,-0.26\n1996-01-01,-0.65,-0.61,0.49,-0.15\n1996-02-01,-0.66,-0.32,0.22,0.16\n1996-03-01,-0.48,-0.07,0.07,0.12\n1996-04-01,-0.34,-1.28,0.15,0.34\n1996-05-01,-0.40,-0.81,0.16,0.43\n1996-06-01,-0.12,-1.07,-0.06,0.56\n1996-07-01,-0.14,-0.98,-0.15,0.41\n1996-08-01,-0.30,-0.46,-0.36,0.15\n1996-09-01,-0.34,-0.53,-0.22,-0.01\n1996-10-01,-0.28,-0.80,-0.34,-0.10\n1996-11-01,-0.30,-0.79,-0.28,-0.06\n1996-12-01,-0.43,-1.03,-0.15,-0.33\n1997-01-01,-0.43,-0.56,0.06,-0.55\n1997-02-01,-0.24,-0.03,0.16,-0.60\n1997-03-01,-0.06,0.82,-0.00,-0.82\n1997-04-01,0.34,0.98,0.12,-0.81\n1997-05-01,0.87,2.16,0.23,-0.89\n1997-06-01,1.15,3.17,0.29,-0.98\n1997-07-01,1.60,3.63,0.06,-0.53\n1997-08-01,1.94,3.98,-0.17,-0.29\n1997-09-01,2.10,4.05,-0.12,0.06\n1997-10-01,2.29,4.00,0.16,0.47\n1997-11-01,2.42,4.12,0.27,0.60\n1997-12-01,2.30,4.27,0.15,0.76\n1998-01-01,2.42,3.55,0.19,0.38\n1998-02-01,2.08,2.88,0.59,0.47\n1998-03-01,1.49,2.88,0.59,0.34\n1998-04-01,0.90,3.00,0.43,0.23\n1998-05-01,0.68,3.01,0.39,0.20\n1998-06-01,-0.39,2.00,0.42,0.35\n1998-07-01,-0.73,1.31,0.29,0.59\n1998-08-01,-0.83,0.94,0.33,0.40\n1998-09-01,-0.82,0.31,0.13,0.26\n1998-10-01,-1.19,0.12,0.07,-0.01\n1998-11-01,-1.23,-0.21,0.12,0.14\n1998-12-01,-1.51,-0.14,0.00,-0.15\n1999-01-01,-1.53,-0.23,-0.35,-0.20\n1999-02-01,-1.41,0.05,-0.38,-0.22\n1999-03-01,-0.92,0.30,-0.38,-0.02\n1999-04-01,-0.81,-0.78,-0.28,0.14\n1999-05-01,-0.87,-0.43,-0.09,0.18\n1999-06-01,-0.95,-0.55,-0.10,0.18\n1999-07-01,-0.84,-0.65,-0.10,0.38\n1999-08-01,-0.98,-0.51,0.01,0.35\n1999-09-01,-0.84,-0.97,0.01,0.09\n1999-10-01,-1.03,-0.76,-0.18,0.05\n1999-11-01,-1.41,-1.11,-0.18,0.15\n1999-12-01,-1.54,-1.06,-0.02,0.10\n2000-01-01,-1.79,-0.89,-0.42,-0.12\n2000-02-01,-1.53,-0.47,-0.28,0.12\n2000-03-01,-1.26,-0.56,-0.09,0.05\n2000-04-01,-0.80,0.46,-0.17,0.06\n2000-05-01,-0.80,-0.29,-0.27,-0.05\n2000-06-01,-0.75,-0.65,-0.40,-0.11\n2000-07-01,-0.57,-0.57,-0.27,-0.10\n2000-08-01,-0.36,-0.59,-0.19,-0.00\n2000-09-01,-0.39,-0.34,-0.36,-0.13\n2000-10-01,-0.55,-0.49,-0.42,-0.12\n2000-11-01,-0.75,-0.70,-0.43,-0.11\n2000-12-01,-0.92,-0.55,-0.49,-0.27\n2001-01-01,-0.88,-0.76,-0.38,-0.17\n2001-02-01,-0.63,-0.33,-0.32,-0.10\n2001-03-01,-0.48,0.35,-0.01,-0.06\n2001-04-01,-0.30,0.42,-0.11,0.07\n2001-05-01,-0.30,-0.64,-0.14,0.12\n2001-06-01,-0.11,-0.86,-0.01,0.04\n2001-07-01,0.01,-0.70,0.02,-0.11\n2001-08-01,-0.07,-0.77,0.00,-0.06\n2001-09-01,-0.28,-1.20,0.13,-0.04\n2001-10-01,-0.26,-1.14,0.06,-0.03\n2001-11-01,-0.28,-0.94,0.31,-0.02\n2001-12-01,-0.46,-1.12,0.37,-0.25\n2002-01-01,-0.14,-0.94,0.53,-0.19\n2002-02-01,0.00,-0.30,0.35,-0.33\n2002-03-01,0.11,0.62,0.21,-0.06\n2002-04-01,0.14,0.33,-0.18,-0.00\n2002-05-01,0.21,0.18,-0.40,0.15\n2002-06-01,0.68,-0.40,-0.38,0.02\n2002-07-01,0.57,-0.44,-0.33,0.22\n2002-08-01,0.70,-0.26,-0.38,-0.19\n2002-09-01,0.82,0.04,-0.31,-0.23\n2002-10-01,1.16,0.57,-0.07,-0.09\n2002-11-01,1.41,0.78,-0.09,-0.25\n2002-12-01,1.41,0.69,-0.09,0.08\n2003-01-01,0.98,0.10,0.02,0.22\n2003-02-01,0.64,-0.21,-0.17,0.30\n2003-03-01,0.48,-0.49,-0.11,0.28\n2003-04-01,-0.03,-0.90,-0.15,0.11\n2003-05-01,-0.52,-1.35,-0.14,0.02\n2003-06-01,-0.19,-0.91,-0.02,-0.21\n2003-07-01,0.14,-0.58,0.09,0.14\n2003-08-01,0.05,0.08,0.26,0.30\n2003-09-01,0.15,-0.43,0.28,0.34\n2003-10-01,0.46,-0.07,0.38,0.31\n2003-11-01,0.39,0.29,0.17,0.36\n2003-12-01,0.32,0.31,0.20,0.30\n2004-01-01,0.26,-0.03,0.25,0.07\n2004-02-01,0.17,-0.20,0.34,-0.05\n2004-03-01,-0.10,-0.34,0.20,-0.36\n2004-04-01,0.06,-0.45,0.30,-0.57\n2004-05-01,0.10,-0.96,-0.03,-0.47\n2004-06-01,0.14,-0.69,0.02,-0.39\n2004-07-01,0.41,-0.70,0.18,-0.18\n2004-08-01,0.66,-0.44,0.35,0.02\n2004-09-01,0.67,-0.10,0.21,0.18\n2004-10-01,0.73,0.11,0.20,-0.08\n2004-11-01,0.62,0.39,0.32,0.06\n2004-12-01,0.71,0.27,0.24,0.17\n2005-01-01,0.56,0.03,0.22,0.28\n2005-02-01,0.26,-0.61,0.20,0.22\n2005-03-01,0.28,-1.15,0.53,0.26\n2005-04-01,0.28,-0.36,0.65,-0.02\n2005-05-01,0.30,0.22,0.81,-0.40\n2005-06-01,0.22,-0.15,0.67,-0.57\n2005-07-01,-0.01,-0.06,0.57,-0.57\n2005-08-01,-0.04,-0.30,0.41,-0.19\n2005-09-01,-0.08,-0.61,0.35,-0.23\n2005-10-01,-0.15,-1.05,0.12,-0.22\n2005-11-01,-0.44,-1.41,0.09,-0.26\n2005-12-01,-0.75,-1.19,0.26,-0.29\n2006-01-01,-0.98,-0.69,0.16,-0.30\n2006-02-01,-0.71,0.33,0.01,-0.04\n2006-03-01,-0.73,-0.27,0.02,0.06\n2006-04-01,-0.30,-0.89,0.26,-0.00\n2006-05-01,-0.11,-0.39,0.23,-0.11\n2006-06-01,0.09,-0.19,0.28,0.10\n2006-07-01,0.03,0.18,0.09,0.05\n2006-08-01,0.37,0.51,0.20,0.13\n2006-09-01,0.62,0.78,0.31,0.13\n2006-10-01,0.76,1.20,0.31,0.01\n2006-11-01,0.98,0.91,0.40,-0.10\n2006-12-01,1.10,0.62,0.18,0.05\n2007-01-01,0.59,0.57,0.21,-0.09\n2007-02-01,0.12,0.06,0.38,-0.06\n2007-03-01,-0.15,-0.65,0.17,-0.31\n2007-04-01,-0.16,-1.05,0.24,-0.07\n2007-05-01,-0.39,-1.47,0.01,-0.06\n2007-06-01,-0.16,-1.40,-0.08,0.12\n2007-07-01,-0.37,-0.94,-0.18,-0.02\n2007-08-01,-0.57,-1.36,-0.33,-0.04\n2007-09-01,-1.04,-1.34,-0.27,0.12\n2007-10-01,-1.41,-1.65,-0.19,-0.11\n2007-11-01,-1.58,-1.95,-0.07,-0.08\n2007-12-01,-1.61,-1.63,-0.21,-0.06\n2008-01-01,-1.79,-0.80,-0.10,-0.14\n2008-02-01,-1.70,-0.16,-0.00,0.09\n2008-03-01,-1.17,0.33,0.12,0.25\n2008-04-01,-0.89,0.50,-0.06,0.33\n2008-05-01,-0.64,-0.32,0.07,0.47\n2008-06-01,-0.44,0.26,0.15,0.39\n2008-07-01,-0.04,0.47,0.17,0.40\n2008-08-01,-0.04,0.67,0.12,0.31\n2008-09-01,-0.28,0.32,0.20,0.13\n2008-10-01,-0.30,-0.17,0.01,0.05\n2008-11-01,-0.37,-0.21,-0.04,0.13\n2008-12-01,-0.90,-0.33,-0.03,0.15\n2009-01-01,-1.00,-0.17,-0.17,0.11\n2009-02-01,-0.71,-0.31,-0.55,0.08\n2009-03-01,-0.72,-0.85,-0.49,0.20\n2009-04-01,-0.25,0.00,-0.47,0.21\n2009-05-01,0.17,0.17,-0.43,0.44\n2009-06-01,0.49,0.54,-0.19,0.28\n2009-07-01,0.69,0.70,0.17,0.14\n2009-08-01,0.62,0.66,0.07,0.04\n2009-09-01,0.68,0.22,-0.03,0.04\n2009-10-01,0.96,0.22,0.20,0.23\n2009-11-01,1.49,0.14,0.20,0.11\n2009-12-01,1.81,0.49,0.25,0.43\n2010-01-01,1.43,0.36,0.55,0.46\n2010-02-01,1.18,0.04,0.90,0.41\n2010-03-01,1.07,-0.04,1.12,0.54\n2010-04-01,0.56,0.51,1.25,0.37\n2010-05-01,-0.15,0.39,1.15,0.27\n2010-06-01,-0.62,0.14,0.87,0.54\n2010-07-01,-0.89,-1.05,0.74,0.35\n2010-08-01,-1.33,-1.23,0.63,0.29\n2010-09-01,-1.56,-1.29,0.37,0.08\n2010-10-01,-1.65,-1.53,0.35,0.21\n2010-11-01,-1.57,-1.39,0.39,0.22\n2010-12-01,-1.63,-1.03,0.67,0.15\n2011-01-01,-1.70,-0.61,0.73,0.12\n2011-02-01,-1.26,0.07,0.35,0.32\n2011-03-01,-0.98,-0.43,0.25,0.38\n2011-04-01,-0.74,-0.05,0.17,0.18\n2011-05-01,-0.53,0.25,0.20,-0.04\n2011-06-01,-0.25,0.36,0.35,-0.11\n2011-07-01,-0.23,0.27,0.15,0.02\n2011-08-01,-0.66,-0.20,0.15,0.00\n2011-09-01,-0.76,-1.01,-0.01,0.09\n2011-10-01,-0.93,-0.71,0.02,0.06\n2011-11-01,-1.09,-0.80,-0.10,-0.23\n2011-12-01,-1.05,-0.68,-0.06,-0.74\n2012-01-01,-0.93,-0.39,-0.09,-0.66\n2012-02-01,-0.61,0.40,-0.19,-0.78\n2012-03-01,-0.48,-0.02,-0.41,-0.43\n2012-04-01,-0.29,0.48,-0.33,-0.64\n2012-05-01,-0.18,0.43,-0.11,-0.38\n2012-06-01,0.14,0.99,0.07,-0.27\n2012-07-01,0.44,0.82,0.02,-0.29\n2012-08-01,0.66,0.27,0.19,-0.28\n2012-09-01,0.44,0.04,0.34,-0.13\n2012-10-01,0.23,-0.39,0.35,-0.19\n2012-11-01,0.33,-0.46,0.44,-0.16\n2012-12-01,-0.13,-0.61,0.26,-0.09\n2013-01-01,-0.42,-0.70,0.19,0.05\n2013-02-01,-0.40,-0.32,0.22,-0.02\n2013-03-01,-0.14,-0.38,0.55,-0.09\n2013-04-01,-0.08,-1.32,0.37,0.04\n2013-05-01,-0.28,-1.44,0.44,0.13\n2013-06-01,-0.33,-1.44,0.07,-0.03\n2013-07-01,-0.28,-1.36,0.06,-0.03\n2013-08-01,-0.29,-0.76,0.22,0.05\n2013-09-01,-0.09,-0.77,0.23,-0.12\n2013-10-01,-0.24,-0.16,0.23,-0.18\n2013-11-01,-0.02,-0.50,0.02,-0.26\n2013-12-01,-0.09,-0.25,-0.02,-0.16\n2014-01-01,-0.42,0.37,-0.12,-0.23\n2014-02-01,-0.45,-0.67,-0.25,-0.05\n2014-03-01,-0.07,-0.51,-0.50,-0.13\n2014-04-01,0.28,-0.25,-0.36,-0.08\n2014-05-01,0.45,0.74,-0.27,0.14\n2014-06-01,0.48,1.17,-0.29,0.15\n2014-07-01,0.13,1.04,-0.13,-0.09\n2014-08-01,0.14,0.67,-0.10,0.05\n2014-09-01,0.37,0.15,0.13,0.11\n2014-10-01,0.48,0.34,0.13,-0.06\n2014-11-01,0.89,0.57,0.04,-0.50\n2014-12-01,0.77,0.14,0.05,-0.38\n2015-01-01,0.59,-0.16,-0.08,0.10\n2015-02-01,0.57,-0.62,-0.06,0.15\n2015-03-01,0.48,-0.01,-0.27,-0.11\n2015-04-01,0.90,0.64,-0.40,0.05\n2015-05-01,1.04,1.59,-0.13,0.11\n2015-06-01,1.28,2.19,-0.31,0.07\n2015-07-01,1.56,2.32,-0.12,-0.19\n2015-08-01,1.87,2.08,0.09,-0.31\n2015-09-01,2.01,2.48,0.35,-0.19\n2015-10-01,2.21,2.38,0.51,0.06\n2015-11-01,2.57,2.21,0.18,0.19\n2015-12-01,2.56,2.09,0.13,0.44\n2016-01-01,2.56,1.82,0.25,0.47\n2016-02-01,2.11,1.09,0.28,0.32\n2016-03-01,1.60,1.10,0.24,0.25\n2016-04-01,1.05,0.29,0.17,0.08\n2016-05-01,0.45,0.45,0.18,0.01\n2016-06-01,0.06,0.52,0.15,0.37\n2016-07-01,-0.25,-0.06,0.10,0.37\n2016-08-01,-0.48,0.19,0.15,0.33\n2016-09-01,-0.46,0.25,0.01,0.11\n2016-10-01,-0.75,-0.03,0.12,0.24\n2016-11-01,-0.63,-0.08,0.15,0.10\n2016-12-01,-0.51,0.02,0.23,0.41\n2017-01-01,-0.34,0.33,0.24,0.25\n2017-02-01,-0.01,0.93,0.06,-0.18\n2017-03-01,-0.09,1.56,-0.03,-0.06\n2017-04-01,0.22,0.49,0.31,0.09\n2017-05-01,0.30,0.25,0.38,0.12\n2017-06-01,0.22,0.20,0.35,0.26\n2017-07-01,0.22,-0.46,0.32,0.10\n2017-08-01,-0.18,-0.76,0.33,0.39\n2017-09-01,-0.56,-1.28,0.15,0.12\n2017-10-01,-0.52,-1.32,0.03,0.19\n2017-11-01,-0.84,-1.44,0.22,0.02\n2017-12-01,-0.85,-1.54,0.25,-0.09\n2018-01-01,-0.86,-1.25,0.03,0.29\n2018-02-01,-0.73,-0.43,-0.27,-0.27\n2018-03-01,-0.73,-1.30,-0.13,-0.18\n2018-04-01,-0.36,-0.81,-0.33,-0.03\n2018-05-01,-0.12,-0.46,-0.60,-0.07\n2018-06-01,0.12,-0.47,-0.72,0.15\n2018-07-01,0.27,-0.08,-0.48,0.12\n2018-08-01,0.05,-0.54,-0.32,0.21\n2018-09-01,0.30,0.12,-0.04,0.21\n2018-10-01,0.84,0.30,-0.13,0.31\n2018-11-01,1.00,0.69,-0.32,0.21\n2018-12-01,0.97,0.79,-0.33,0.40\n2019-01-01,0.51,0.67,-0.17,0.41\n2019-02-01,0.62,0.66,-0.04,0.41\n2019-03-01,0.81,-0.18,-0.15,0.67\n2019-04-01,0.67,0.36,-0.17,0.34\n2019-05-01,0.61,0.24,-0.08,0.31\n2019-06-01,0.66,-0.11,0.06,0.27\n2019-07-01,0.41,-0.36,0.09,0.24\n2019-08-01,0.19,-0.29,0.09,0.22\n2019-09-01,0.11,-0.68,0.20,0.33\n2019-10-01,0.55,-0.71,0.01,0.25\n2019-11-01,0.74,0.02,-0.05,0.87\n2019-12-01,0.51,-0.19,-0.04,0.80\n2020-01-01,0.64,-0.12,0.01,0.95\n2020-02-01,0.48,-0.03,0.32,0.57\n2020-03-01,0.36,-0.44,0.58,0.31\n2020-04-01,0.49,0.03,0.46,0.41\n2020-05-01,-0.12,-0.24,0.32,0.53\n2020-06-01,-0.21,-0.43,0.47,0.29\n2020-07-01,-0.04,-1.09,0.38,0.29\n2020-08-01,-0.42,-0.43,0.25,0.30\n2020-09-01,-0.66,-0.85,0.15,0.09\n2020-10-01,-1.19,-0.99,0.19,0.17\n2020-11-01,-1.01,-0.87,0.08,-0.06\n2020-12-01,-0.98,-0.76,0.17,-0.38\n2021-01-01,-1.04,-0.58,0.27,-0.12\n2021-02-01,-0.94,-0.74,0.23,-0.33\n2021-03-01,-0.72,-0.50,-0.01,-0.21\n2021-04-01,-0.55,-0.96,0.16,0.06\n2021-05-01,-0.41,-0.73,0.04,0.41\n2021-06-01,-0.06,0.08,-0.10,0.58\n2021-07-01,-0.20,-0.14,-0.02,0.86\n2021-08-01,-0.38,-0.17,-0.07,0.72\n2021-09-01,-0.50,-0.29,0.16,0.43\n2021-10-01,-0.78,-0.60,0.03,0.42\n2021-11-01,-0.88,-0.98,0.11,0.32\n2021-12-01,-1.07,-1.29,0.16,0.00\n2022-01-01,-0.77,-0.90,0.36,0.14\n2022-02-01,-0.68,-1.14,0.39,0.46\n2022-03-01,-0.84,-0.59,-0.07,0.22\n2022-04-01,-0.90,-1.04,-0.11,0.15\n2022-05-01,-1.02,-1.04,0.14,0.01\n2022-06-01,-0.77,-0.82,0.23,0.30\n2022-07-01,-0.56,-0.54,0.05,0.08\n2022-08-01,-0.96,-0.23,0.14,0.18\n2022-09-01,-1.06,-0.77,0.25,0.20\n2022-10-01,-0.99,-1.37,0.08,0.30\n2022-11-01,-0.76,-0.92,0.02,0.33\n2022-12-01,-0.86,-0.67,0.18,0.34\n2023-01-01,-0.78,-0.35,0.23,0.49\n2023-02-01,-0.62,-0.14,0.04,0.28\n2023-03-01,-0.13,1.00,0.49,0.14\n2023-04-01,0.24,2.50,0.62,0.19\n2023-05-01,0.47,1.69,0.83,0.44\n2023-06-01,0.95,2.39,1.17,0.26\n2023-07-01,1.20,2.45,1.14,0.28\n2023-08-01,1.56,2.55,1.02,0.30\n2023-09-01,1.65,2.04,1.06,0.25\n2023-10-01,1.59,1.80,0.89,0.19\n2023-11-01,2.01,1.94,0.78,0.73\n2023-12-01,1.81,1.37,0.88,1.05\n2024-01-01,1.71,1.14,1.17,0.99\n2024-02-01,1.47,1.08,1.41,1.14\n2024-03-01,1.10,-0.06,1.21,1.33\n2024-04-01,0.93,-0.11,1.17,1.10\n2024-05-01,0.41,-0.41,1.22,0.82\n2024-06-01,0.25,-0.48,1.09,0.54\n2024-07-01,0.20,-0.44,0.87,0.33\n2024-08-01,0.02,-0.62,0.79,0.19\n2024-09-01,-0.11,-0.58,0.66,0.30\n2024-10-01,-0.24,-0.47,0.61,0.28\n2024-11-01,-0.05,-0.14,0.67,0.22\n', 'research/common/baseline.py': '"""Fit and predict the fixed reference model with longer input lags, without changing its architecture."""\nfrom pathlib import Path\nimport gc\nimport numpy as np\nimport xarray as xr\nfrom .inputs import require, sha256, write_json\n\n\ndef fit(data, cutoff, folder):\n    folder = Path(folder)\n    folder.mkdir(parents=True, exist_ok=True)\n    m = data.lib\n    train = data.rain.sel(time=slice(None, cutoff))\n    X, y, ids, dates, climate, atmosphere, sums, counts, seas, cfs = m.amostrar_arvores_multissistema(\n        train, data.atmosphere, data.seas, data.cfs, cutoff)\n    models = []\n    for name, features in [(\'arvores_seas5\', m.FEATURES_SEAS5), (\'arvores_multissistema\', m.FEATURES_MULTISSISTEMA)]:\n        dataset = m.lgb.Dataset(np.ascontiguousarray(X[:, :len(features)]), label=y, feature_name=features)\n        model = m.lgb.train(m.PARAMETROS, dataset, num_boost_round=m.ARVORES)\n        require(model.feature_name() == features, \'Baseline feature order changed.\')\n        model.save_model(str(folder / (name + \'.txt\')))\n        models.append(model)\n        del dataset\n    ridge = m.mos_fit(train, data.seas, data.cfs, cutoff)\n    np.savez_compressed(folder / \'ridge.npz\', **ridge)\n    np.savez_compressed(folder / \'clima_atmosfera_indices.npz\', **atmosphere)\n    for name, field in [(\'chuva\', climate), (\'seas5\', seas), (\'cfsv2\', cfs)]:\n        field.to_netcdf(folder / (\'clima_\' + name + \'.nc\'))\n    audit = dict(training_start=str(dates[0].date()), training_end=str(dates[-1].date()),\n        training_months=len(dates), examples=len(y), parameters=m.PARAMETROS,\n        tree_count=m.ARVORES, weights=[.375, .375, .25], residual_scales=[.9, .875],\n        sample_hash=__import__(\'hashlib\').sha256(ids.tobytes()).hexdigest(), inputs=data.audit)\n    del X, y, ids\n    gc.collect()\n    write_json(folder / \'fit.json\', audit)\n    return dict(models=models, ridge=ridge, climate=climate, atmosphere=atmosphere, seas=seas, cfs=cfs, audit=audit)\n\n\ndef predict(data, state, dates):\n    m = data.lib\n    components = []\n    for model, name, scale in zip(state[\'models\'], m.NOMES_MODELOS, [.9, .875]):\n        anomalies = m.prever_par(model, name, data.atmosphere, data.seas, data.cfs,\n            state[\'atmosphere\'], state[\'climate\'], state[\'seas\'], state[\'cfs\'], dates)\n        components.append(m.reconstruir_chuva(state[\'climate\'], anomalies, scale).values.astype(\'float64\'))\n    ridge = m.mos_predict(state[\'ridge\'], data.seas, data.cfs, dates)\n    result = .75 * (.5 * components[0] + .5 * components[1]) + .25 * ridge\n    return xr.DataArray(result, dims=(\'time\', \'lat\', \'lon\'),\n        coords=dict(time=dates, lat=data.rain.lat, lon=data.rain.lon), attrs=dict(units=\'mm/day\'))\n', 'research/common/inputs.py': '"""Load the existing audited inputs; no downloads, imputation or new sources."""\nfrom pathlib import Path\nimport hashlib\nimport importlib.metadata\nimport importlib.util\nimport json\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\n\nROOT = Path(__file__).resolve().parents[2]\nLIBRARY = ROOT / \'research/lagged_sources/library.py\'\nREQUIRED = {\'numpy\': \'2.0.2\', \'pandas\': \'2.3.3\', \'xarray\': \'2025.12.0\', \'lightgbm\': \'4.6.0\'}\n\n\ndef require(ok, message):\n    if not bool(ok):\n        raise ValueError(message)\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef write_json(path, obj):\n    Path(path).write_text(json.dumps(obj, indent=2, sort_keys=True, allow_nan=False, default=str) + \'\\n\', encoding=\'utf-8\')\n\n\ndef library(path=LIBRARY):\n    # Separate module globals: callers cannot change another run\'s index table.\n    spec = importlib.util.spec_from_file_location(\'fixed_lag_library\', path)\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    return module\n\n\ndef find_unique(filename, supplied=None):\n    if supplied:\n        p = Path(supplied)\n        require((p / filename).is_file(), f\'Missing {filename} in {p}\')\n        return p.resolve()\n    candidates = sorted({p.parent.resolve() for p in Path(\'/kaggle/input\').rglob(filename)})\n    require(len(candidates) == 1, f\'Attach exactly one input containing {filename}, or set its directory explicitly. Found {len(candidates)}.\')\n    return candidates[0]\n\n\ndef preflight(official=None, seas5=None, cfsv2=None, final=False):\n    paths = dict(official=find_unique(\'treino_tp.nc\', official),\n                 seas5=find_unique(\'seas5_51_manifesto.json\', seas5),\n                 cfsv2=find_unique(\'cfsv2_manifesto.json\', cfsv2))\n    expected = json.loads((ROOT / \'research/lagged_sources/official_hashes.json\').read_text())\n    files = {}\n    for row in expected:\n        if not row[\'nome\'].startswith(\'treino_\'):\n            continue\n        p = paths[\'official\'] / row[\'nome\']\n        require(p.is_file(), f\'Missing official input: {p.name}\')\n        require(sha256(p) == row[\'sha256\'], f\'Official input hash mismatch: {p.name}\')\n        files[p.name] = row[\'sha256\']\n    for source, name in [(\'seas5\', \'seas5_51_manifesto.json\'), (\'cfsv2\', \'cfsv2_manifesto.json\')]:\n        manifest = paths[source] / name\n        files[name] = sha256(manifest)\n        m = json.loads(manifest.read_text(encoding=\'utf-8\'))\n        for phase in [\'desenvolvimento\'] + ([\'somente_ajuste_final\'] if final else []):\n            row = m[\'arquivos\'][phase]\n            p = paths[source] / row[\'arquivo\']\n            require(p.is_file() and sha256(p) == row[\'sha256\'], f\'Missing or altered {source}: {phase}\')\n            files[p.name] = row[\'sha256\']\n    versions = {n: importlib.metadata.version(n) for n in REQUIRED}\n    require(versions == REQUIRED, f\'The fixed baseline needs {REQUIRED}; found {versions}.\')\n    return paths, dict(files=files, versions=versions, final_training=final)\n\n\nclass Inputs:\n    def __init__(self, official=None, seas5=None, cfsv2=None, output=\'outputs\', final=False):\n        self.paths, self.audit = preflight(official, seas5, cfsv2, final)\n        self.lib = m = library()\n        m.PASTA = self.paths[\'official\']\n        m.PASTA_SEAS5_MANUAL = self.paths[\'seas5\']\n        m.PASTA_CFSV2_MANUAL = self.paths[\'cfsv2\']\n        m.PASTA_SEAS5, m.MANIFESTO_SEAS5 = m.localizar_seas5()\n        m.PASTA_CFSV2, m.MANIFESTO_CFSV2 = m.localizar_cfsv2()\n        m.SAIDA = Path(output)\n        m.SAIDA.mkdir(parents=True, exist_ok=True)\n        end = \'2022-12-01\' if final else \'2020-12-01\'\n        start = \'1993-01-01\' if final else \'1976-10-01\'\n        with xr.open_dataset(m.PASTA / \'treino_tp.nc\') as ds:\n            require(ds.tp.attrs.get(\'units\') == \'mm/day\', \'Unexpected rainfall units.\')\n            self.rain = ds.tp.sel(time=slice(start, end)).transpose(\'time\', \'lat\', \'lon\').load()\n        require(pd.DatetimeIndex(self.rain.time.values).equals(pd.date_range(start, end, freq=\'MS\')), \'Incomplete rainfall calendar.\')\n        require(np.array_equal(self.rain.lat, np.arange(-60, 15.25, .25)) and\n                np.array_equal(self.rain.lon, np.arange(-90, -24.75, .25)), \'Different target grid.\')\n        require(np.isfinite(self.rain.values).all() and (self.rain.values >= 0).all(), \'Invalid rainfall.\')\n        atm_start = pd.Timestamp(start) - pd.DateOffset(months=4)\n        atm_end = pd.Timestamp(end) - pd.DateOffset(months=4)\n        self.atmosphere, _ = m.carregar_atmosfera(self.rain, atm_start, atm_end)\n        index_path = ROOT / (\'competition/metadata/NOAA/indices_noaa.csv\' if final else \'research/lagged_sources/ocean_indices.csv\')\n        m.INDICES_OC = pd.read_csv(index_path, parse_dates=[\'time_origem\']).set_index(\'time_origem\')\n        m.conferir_indices(m.INDICES_OC)\n        self.audit[\'indices_sha256\'] = sha256(index_path)\n        self.audit[\'library_sha256\'] = sha256(LIBRARY)\n        phases = [\'desenvolvimento\'] + ([\'somente_ajuste_final\'] if final else [])\n        self.seas = xr.concat([m.carregar_seas5(p, self.rain) for p in phases], dim=\'time\').sel(time=slice(start, end))\n        self.cfs = xr.concat([m.carregar_cfsv2(p, self.rain) for p in phases], dim=\'time\').sel(time=slice(start, end))\n        self.audit.update(rainfall_read=[start, end], forecast_partitions=phases,\n                          test_partition_read=False, sst_used=False)\n\n\nclass Context:\n    """Training-only climatologies, with a bounded reference window for inner selection."""\n    def __init__(self, data, reference, training):\n        self.data = data\n        m = data.lib\n        self.reference = pd.DatetimeIndex(reference)\n        self.training = pd.DatetimeIndex(training)\n        require(24 <= len(self.training) <= 360 and len(self.reference) <= 360, \'Invalid training window.\')\n        require(self.training.isin(self.reference).all() and self.reference.is_monotonic_increasing and\n                self.reference.is_unique and self.training.is_unique, \'Invalid training/reference dates.\')\n        self.cutoff = self.reference[-1]\n        rain = data.rain.sel(time=self.reference).values\n        self.counts = np.array([(self.reference.month == month).sum() for month in range(1, 13)])\n        require((self.counts > 1).all(), \'At least two reference years per month are needed.\')\n        self.sums = np.stack([rain[self.reference.month == month].sum(0, dtype=np.float64) for month in range(1, 13)])\n        self.climate = xr.DataArray((self.sums / self.counts[:, None, None]).astype(\'float32\'),\n            dims=(\'month\', \'lat\', \'lon\'), coords=dict(month=np.arange(1, 13), lat=data.rain.lat, lon=data.rain.lon), attrs=dict(units=\'mm/day\'))\n        origins = m.origem_mensal(self.reference, 4)\n        self.atm = m.ajustar_clima_atmosfera(data.atmosphere, origins)\n        origins = m.origem_mensal(self.reference, 3)\n        vals = m.INDICES_OC.loc[origins, m.INDICES_NOMES].to_numpy(dtype=np.float64)\n        self.atm[\'_clima_indices\'] = np.stack([vals[origins.month == month].mean(0) for month in range(1, 13)])\n        self.seasonal = []\n        for forecasts in [data.seas, data.cfs]:\n            v = forecasts.sel(time=self.training)\n            require(pd.DatetimeIndex(v.time_origem.values).equals(m.origem_mensal(self.training, 1)), \'Wrong seasonal initialization.\')\n            means = np.stack([v.values[self.training.month == month].mean(0, dtype=np.float64) for month in range(1, 13)]).astype(\'float32\')\n            require(np.isfinite(means).all(), \'Missing seasonal reference month.\')\n            self.seasonal.append(xr.DataArray(means, dims=self.climate.dims, coords=self.climate.coords))\n\n    def features(self, target, training=False):\n        d, m = self.data, self.data.lib\n        t = pd.Timestamp(target)\n        require(t in self.training if training else t > self.cutoff, \'Month crosses the training boundary.\')\n        x = np.column_stack([m.matriz_mes(d.atmosphere, self.atm, self.climate, t),\n            m.valores_seas5(d.seas, t), m.valores_anomalia_seas5(d.seas, self.seasonal[0], t),\n            m.valores_cfsv2(d.cfs, t), m.valores_anomalia_cfsv2(d.cfs, self.seasonal[1], t)]).astype(\'float32\')\n        if training:\n            # Exclude the entire target map, including neighbors seen by convolutions.\n            y = d.rain.sel(time=t).values\n            month = t.month - 1\n            x[:, 22] = ((self.sums[month] - y.astype(\'float64\')) / (self.counts[month] - 1)).ravel().astype(\'float32\')\n        require(np.isfinite(x).all(), \'Nonfinite features.\')\n        return x.T.reshape(31, d.rain.sizes[\'lat\'], d.rain.sizes[\'lon\'])\n\n    def save(self, folder):\n        folder = Path(folder)\n        folder.mkdir(parents=True, exist_ok=True)\n        self.climate.to_netcdf(folder / \'rainfall_climatology.nc\')\n        np.savez_compressed(folder / \'references.npz\', sums=self.sums, counts=self.counts,\n                            seas5=self.seasonal[0].values, cfsv2=self.seasonal[1].values, **self.atm)\n        write_json(folder / \'context.json\', dict(reference=self.reference.strftime(\'%Y-%m\').tolist(),\n                                                training=self.training.strftime(\'%Y-%m\').tolist(), features=self.data.lib.FEATURES_MULTISSISTEMA))\n', 'research/common/presentation.py': '"""Reader-facing model names, without changing saved experiment identifiers."""\n\nMODEL_LABELS = {\n    \'hybrid\': \'Reference model\',\n    \'hybrid_sst\': \'Reference model + SST\',\n    \'unet\': \'U-Net\',\n    \'blend\': \'Reference model + U-Net\',\n    \'blend_reference\': \'Reference model + U-Net\',\n    \'blend_sst\': \'Reference model + SST + U-Net\',\n    \'climatology\': \'Climatology\',\n    \'direct_unet\': \'Direct U-Net\',\n    \'residual_unet\': \'Residual U-Net correction\',\n    \'direct_blend\': \'Reference model + direct U-Net\',\n    \'unet_corrected\': \'Bias-corrected U-Net\',\n    \'blend_corrected\': \'Reference model + bias-corrected U-Net\',\n}\n\n\ndef model_label(value):\n    """Leave unknown names intact so unrelated table values are never relabelled."""\n    return MODEL_LABELS.get(value, value)\n\n\ndef display_frame(frame):\n    """Return a presentation copy; preserve the original data and numeric values."""\n    result = frame.copy()\n    for column in (\'model\', \'modelo\', \'Model\', \'reference\', \'comparator\'):\n        if column in result:\n            result[column] = result[column].map(model_label)\n    return result\n', 'research/common/synthetic_fixture.py': '"""Small synthetic fields for tests; never operational inputs."""\nfrom types import SimpleNamespace\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nfrom .inputs import library\n\n\ndef fixture():\n    m = library()\n    dates = pd.date_range(\'1976-06-01\', \'2009-01-01\', freq=\'MS\')\n    rng = np.random.default_rng(71)\n    coords = dict(time=dates, lat=np.linspace(-60, 15, 9), lon=np.linspace(-90, -25, 13))\n    def field():\n        return xr.DataArray(rng.uniform(1, 5, (len(dates), 9, 13)).astype(\'float32\'), dims=(\'time\', \'lat\', \'lon\'), coords=coords)\n    rain, seas, cfs = field(), field(), field()\n    origins = m.origem_mensal(dates, 1)\n    seas = seas.assign_coords(time_origem=(\'time\', origins.values))\n    cfs = cfs.assign_coords(time_origem=(\'time\', origins.values), inicializacao_mais_recente=(\'time\', origins.values))\n    atmosphere = {name: field() for name in m.VARIAVEIS}\n    m.INDICES_OC = pd.DataFrame(rng.normal(size=(len(dates), 4)), index=dates, columns=m.INDICES_NOMES)\n    return SimpleNamespace(lib=m, rain=rain, seas=seas, cfs=cfs, atmosphere=atmosphere)\n', 'research/lagged_sources/library.py': 'from pathlib import Path\nfrom datetime import datetime, timezone\nfrom time import perf_counter\nimport base64, gzip, io, json, hashlib, gc, platform, importlib.metadata\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nimport lightgbm as lgb\nfrom sklearn.decomposition import PCA\nfrom threadpoolctl import threadpool_limits\nfrom IPython.display import display, Markdown, Image, FileLink\nDATASET, SYSTEM, SCHEMA = (\'seasonal-monthly-single-levels\', \'51\', \'worcap_seas5_v1\')\nPASTA_SEAS5_MANUAL = PASTA_CFSV2_MANUAL = None\nPONTOS_POR_MES = 5000\nANOS_TREINO = 30\nARVORES = 300\nSEMENTE = 42\nCORTE_FINAL = \'2022-12-01\'\nFIM_DESENVOLVIMENTO = \'2020-12-01\'\nPARAMETROS = {\'objective\': \'regression\', \'metric\': \'rmse\', \'learning_rate\': 0.05, \'num_leaves\': 31, \'min_data_in_leaf\': 150, \'lambda_l2\': 10.0, \'feature_fraction\': 0.9, \'bagging_fraction\': 0.8, \'bagging_freq\': 1, \'num_threads\': 4, \'seed\': SEMENTE, \'deterministic\': True, \'force_col_wise\': True, \'verbosity\': -1}\nBLOCOS = [{\'nome\': \'H1\', \'corte\': \'2006-12-01\', \'inicio\': \'2007-01-01\'}, {\'nome\': \'H2\', \'corte\': \'2008-12-01\', \'inicio\': \'2009-01-01\'}, {\'nome\': \'H3\', \'corte\': \'2010-12-01\', \'inicio\': \'2011-01-01\'}, {\'nome\': \'H4\', \'corte\': \'2012-12-01\', \'inicio\': \'2013-01-01\'}, {\'nome\': \'A\', \'corte\': \'2014-12-01\', \'inicio\': \'2015-01-01\'}, {\'nome\': \'B\', \'corte\': \'2016-12-01\', \'inicio\': \'2017-01-01\'}, {\'nome\': \'C\', \'corte\': \'2018-12-01\', \'inicio\': \'2019-01-01\'}]\nVARIAVEIS = [\'cloud_cover\', \'geopotential_850\', \'rel_hum_850\', \'shum_850\', \'surface_pressure\', \'t2\', \'temperature_850\', \'u_850\', \'v_850\']\nINDICES_NOMES = [\'nino34\', \'nino12\', \'tna\', \'tsa\']\nFONTES_INDICES = {\'fonte\': \'NOAA Physical Sciences Laboratory\', \'captura_utc\': \'2026-09-21T23:00:40.933121+00:00\', \'unidade\': \'degC\', \'tipo\': \'monthly sea surface temperature anomaly indices\', \'interpretacao_temporal\': \'Retrospective series with observation month strictly T-1. These are not archived real-time vintages.\', \'inicio\': \'1976-12-01\', \'fim\': \'2024-11-01\', \'meses\': 576, \'colunas\': [\'time_origem\', \'nino34\', \'nino12\', \'tna\', \'tsa\'], \'fontes\': [{\'indice\': \'nino34\', \'url\': \'https://psl.noaa.gov/data/timeseries/month/data/nino34.long.anom.data\', \'arquivo_bruto\': \'nino34.data\', \'sha256_bruto\': \'46e501b91f624ef85b525d73fbb427f9d04d30411ba6e5c7cff579b35fb4ff01\', \'inicio_declarado\': 1870, \'fim_declarado\': 2026, \'rodape\': \'-99.99\\n  NINA34\\n 5N-5S 170W-120W \\n HadISST \\n  Anomaly from 1981-2010\\n https://psl.noaa.gov/data/timeseries/month/\\n  units=degC\'}, {\'indice\': \'nino12\', \'url\': \'https://psl.noaa.gov/data/timeseries/month/data/nino12.long.anom.data\', \'arquivo_bruto\': \'nino12.data\', \'sha256_bruto\': \'b13ad5ad085cd13e70ec5d82662b509d522204e38a8eefd133f14be270e51928\', \'inicio_declarado\': 1870, \'fim_declarado\': 2026, \'rodape\': \'-99.99\\n  NINA12\\n 0N-10S 270W-280E \\n HadISST \\n  Anomaly from 1981-2010\\n https://psl.noaa.gov/data/timeseries/month/\\n  units=degC\'}, {\'indice\': \'tna\', \'url\': \'https://psl.noaa.gov/data/correlation/tna.data\', \'arquivo_bruto\': \'tna.data\', \'sha256_bruto\': \'387c4125f377e57bcb683a849aefcd76da7d7f3cdcb2ac68ab086ceadf300e8b\', \'inicio_declarado\': 1948, \'fim_declarado\': 2026, \'rodape\': \'-99.99\\n  TNAa\\n Tropical Northern Atlantic Index\\n SST Anom 5.5N-23.5N; 15W-57.5W\\n  Using HadISST1.1: Climo 1991-2020\\n  NOAA/PSL\\n  https://psl.noaa.gov/data/timeseries/month/\'}, {\'indice\': \'tsa\', \'url\': \'https://psl.noaa.gov/data/correlation/tsa.data\', \'arquivo_bruto\': \'tsa.data\', \'sha256_bruto\': \'700c0dffda0f19c1254ba482808b345eaa0538f5bf09fc3bc025f46a5fbe3a69\', \'inicio_declarado\': 1948, \'fim_declarado\': 2026, \'rodape\': \'-99.99\\n  TSAa\\n  NOAA/PSL\\n  https://psl.noaa.gov/data/timeseries/month/\'}], \'sha256_csv\': \'b6f444fbec48681c1e5b46a5c22aa197ad91c7fbbdaff82c3ed3a9c63f7c15a4\', \'transformacao_no_modelo\': \'Monthly recentering within each fold, using only the 360 training origins; no standardization or moving averages.\'}\nINDICES_CSV_GZIP_BASE64 = \'H4sIAAAAAAACCm1cy44tuY3cz7ecGuj9+JqBF4bRC7eBmf5/zElGBMkse1N90YpSSqL4CJKqv/7459//51//+8c//v7Pz59//PmvPuw/tX3++vNvn7/+72//Ve9eP7X9lPop/73G90ebn5/vP8vzc+wHsL+jABwDXBsywOwEcIZpv1xHIJqm6EDg//aDMfv3IGI8iB9fwYl/O2QSUrp9wQabAfslZOE7fX1/FMyBvVQCNhcyPz5Pw3oJOADg09PGlv2s+sblFLbPadMvW1dtQNTCA7v4PUMsWyeXUe1IK1eAM507nxik8kVgzH4bX8RxHBfLttUV++3asWoiJBdbXbctN0xHAMVSbPq9n3/a904hQFLpmz+/37AdX00hoeAg5ohl3EvIIgQSOzgJ+84+hGxBTlwe3MLeCTmEQBprxKkOrfYKgt/EinCAhEA0dor2m50nYzIkpBJS7cBGTOUInmt5jqP1tG5b63XRlOkAXMZxCdAMWNzxY52VAEoGJ4nLyL1NIgYRGxKW4vXCcdeW+zFlsw35DbuuK+VR6/0oTIXCaAmSyVeDpDDpmt9QFbtjhQt9joXjUpTz3ONGnX/WiXGpSW+P9cEFg6ZxgspjtJtbW2izVuCm64ZIqVLPxTrFJUHTVPwUhgCcondfPwRmwn4A3UVl6iMd8nHKASsrxa3caARMFyU/MGkFOS6btXWWWIDZtGecJgufhQ3AbepaglSDdxEQaOIiRKqBY6z49wYQEFcNDML00VRPQniauKu00XHzH0RzA/xTeXt7hdo/iEqJPL9ikHbdEONEK0US9qLH5d+dkK7LTdP8EdqMzoOQ5cIme/X7fTTHfO23PaKtMISTiBWW4EdWEj+nJnHD1cNO44xxw2pIB5/HTWyQ0SZE0oEVx2DrZkGAcOGUnoRzP9SFB6FzhV/lhT/mB4kIsxXCb3FLWsiGl6fFhs2sPBAJpybTiQVbvPBA3sKBf2rdHc8DkdLAh9bQ/SUEtWbN8OQG3pUAqs1tH+0Zerc1w05xzUreaWsrtF+3u5+FV9nayIUP/p5zk/WFw+EnIJX2aH2jIuztTusBVAL2p3MXAPgMDYCv+ek0IfuGQnWKpD33q32vfwQbpRLAGb73t/F2mDpUjXdsYj2b6BbCnbienaKoj7A74iaGI4vj08cHFR9fsRjhAVAQ6wHM5eZha4Idetafw/4xx5g/ceJOdapjT2a8h4LMgtMMXeM2qSAWUsnbzGQluxSkwhDjt9tyd/IgdLevuQHdfgFG6McpEbf1Yi6KCE0xe7oNjxbC3I/QDoYM1f3OGESMVxijIPjDEPRByHQ997XqxGCe9BnZrotwjT5L7vdBuOlKurHMSekz541QMPzEqkRILrADWIIhFudww3XS3TWd7YsIl8sJK2mxaO1ENCJwqDOpCbRgJsncOHc4qNIIcenW8E3TJD2JkGjuisgRUzV9R7K555kE7vHUMKIzhPOcfKWALyCFEAlnwwfOEIBFZw9E0lm41YMmMu3HpXMN0YLM+HdcbXaIQCE6IC4fmL7TCJfrmeFY4OJuuJ7CHbve0ABrO4+leBArxAOp4WRBBHAdV4gHC8GFOsUiKyIkHviLM3iCCmFWSGfU+MChzSJE0hk79rtHqM4K6cBgQ4GM/FathK6lriC8izECEccdnFzttgOpHGdsjJOgLzUsJ1BwfItH9qvnVTqFDJN+qYoE0N3fx/pAcy796AMQr69wF7jmk1E4AeSg3/XrfkEssEPbfct3l5V0CSdyOD6cCFdylCC5Z7uTv+Zlp4/jFEXnOb9ZYjDQyuFN/zy0C4TPk8MHw+PgE4zxfPv07vNxGL0quq9cGwRQH5tWH5OjYW5dx2/xqwnJyN7gt52+T2xf8aXdwuDu6zoJNH6BKCuYO8Or7TQMpCCYO311A0FS2JCI+7i8w9r9IWDKX077iUDRTpiARQDi9EvHLl7jnL2aQak0KyUU1il7NUZSedftpH2fl4jH3VfR8eZ64IS92SlVOmzzUnUSUYmA51jJmBZB5ECgAHeHO0MILsr+QFJaaW0Pn8XZn7XCOkY4CPMj0s7dOH0xm12IGPJ12XKPcFOJtx9zHzAxsGQwHjcMFMNVpbHSd/bLyg0x44SQ90CQNJZuMC74Ta79xERwQAsIdx3jdzTkCJn9nvJ+ogyEvPkEbgADvGe7Nzg8LVCPn+sQkVinq8tx73KDxSsbGLkEcww38fgV89t9L5pierrC+UzZnn+6weRLiRiXi9FH5DhwNTaJ5w/V7hZ3HDCEK7GxrjnoOpgQOX4yrQEg14ET3Yk+V04hHl+Ox0CYovDIncannAPZoJ1GjcSKckye07NQ6dYQCkOSuCNmH26NJFfxQ+v7NcXwDJOHc7IzRMycjPUEoQRfg51Mp5vLY8tbgyU2Z4lrvxZJgYzui0QG0rKst7o8UgIBpmVyF+7KbZoW7GBeAuSIx0c6PVb+hLyJLRJmZUT2/DZ35StpwfCcym3uyjHDiLDFEhm3uSs3T8cYD9ZFnyBPHM+Pne7FOQSQKLYnWjhdF6LyUgVjn89JbucsFf78BmOHzlB/riePbktMMeUd5npBPNOV8vMwckb8b0tJ4BkkEHRMu40k8EhFCYTym5Dm+VNPcpEi2Zl21xHoaOIuEHx3FTHbTdMUub/bo26yPYajOR4EjJQFwaG35rz49gixnignZ5MtyLuJvj+X7yQK1vWJ7RnSt0Fo+sTxaFiqijtqPvz20JDlZsuumUUBt0ciePkXLI9gCaXb3VzBpDM9O0VDb3dRoNSCK7OLYr07woWMsNrKKxLhfijZ/518SGLuJSU393LCdYeLo1g403L1qBOhbPCIiH2X8BHDBcI83g2TZNWDO37RD1CYtcNgDBcJ7iMYDOgD/MyI5HzyVVhp515eDIRRVRT17nibrZlS8JVnKrNlUTms94ig685gIOXj7KV80jjlerYbb7OyuJoz1CMX0KrC5DvDgWzkhulNmeK408UBgYU3avrCennK/mFeFs54RnI+ygegEpr/vKIoC9QbrQ4Rb46ebEXlHG6uaHVrZGUbD8rNFW6sKRrDG37G4yxMYsSDZMwQiaOv6bkiHKxF2jdT9OXBYPNg/a7fuS2rj9kPS6DdRNC7MQLbrPFGSDTT88LMCu2KEZubyPljRypV2ajh1CL2y3Qj5LePWBH4riSUEr7WjI/v9L7KhDMZ1sJJXCqwSsgy8P4XQt7hL3IfgBcuNlInnclLd264YjsVTLpbRhYhJhGapEWcD9uwChHJdMliIJY5jQhqiiUhLGUAQ30qx6erYqOlRxb4cpxU/bv2jpokzPA9HCdX/4ZYHfwtFVJukPU7niTxiQpF0wcu8txf9Ry0V9XZyN2ei/9+doCG1vAh2zPxw9LYTSVRHZDn4Z/ZVSxlDvueyMI/SfIJmqSc7j2eg/9aifb4UDOH+vbx2Grc1zDu/Akf/hwNCoiwMxxWJeQ8w7BuItk3VddRpSgKJywNdFNpfXuew3IWiEFSXf08H71Dnn5ofqmCEdkelUwejSfeUXVoXmCCopzI7+LCtCpY5f4jvSu11eWEOiZyTkXMpadCiCYZ1dk5i5eNCM/u2kYSIy1CyDqdbEuPLzWT853Yqm38ELGihvDzrjsLIYGclDc0RBfipCQzi/tMBxFwX1PcLQDczn1VQyJIOcrq3htSGSkfUc8nDj3EYraa1c+GNGgrpYRYYNmQemcGuhHyTt2P8HGOUNIEYTiCeRTbJxEjMiKy1mZhvqoPwPwPZRmRCEI84z7j0OFjvtEKIPvliNWOQ4MOiDuOVCBA0FkEuaGO7hAA/4aABnHPgcsx/sOxhT+fESDitmm5Uaxqv+f6RgVfSCq1I6zHLei0rYS4S09pFTF2QpJTl9bJmRAxws/5TlBpJmC+HOEarufYcC6115zvQvoFCEVa1aP70uJAUp0d5gfdTOW1zptcdkVCFVG8PhJOHSH+cD9VOhH15fZhL1ntJqK9GhzM1DGtYZcxldkZoQwnd9+QD4iWM0msMDIoAECpE89dtCSSKLDX4b0tYHW4p1Ffb9EYMdAdw/HlXucn9e1pm0HWZ8owmuNoApCDkN3ky6ddkoMgRZpodOMulYM3T44P0bdwBlEQM/qw10hF6ahFQUa0CcG2nGc8KPo9nodqvocg6IuE101PLwR073Fy9avgqgSMV+PGLR6N4t721EFnEW2f4QS1CleOG6SROT9NsjOttIYyuKXB8ZM60izPbxzBt3FzXhLhx0GIYuPigjC+FtKjqM3Pe6fWVXiBU1ocbqnTqzMPq68HO2/Lw1c1kmG85ZwofnWEdU/MvBYvpIH+YAWJmC+mJ7w9UN+YubXprtRnKsTKR7xuZKQvAeotDQulkjYBFMJSN6c3ZXKcUljbQwKqJ8Ylhd2l+eR4Gq8pl2miUL4ewzzGXRX9sjr5DM/I6K6PhND8IgcXZwXGhQhVCS4O22o3fDJOxPjISRxjWyyZagLl171WdcgEOb48zyoVYtJDG5ADT1RnvhHn1d8Ed1XDJCU2DsK/UhGuAxFtcpPlM0/FcyNRMB8RkF5SbSBacu8qXS6ynS8ikXGEgKDrasQCwufw5Jz2RkCPwF+tlyXuw3q57R9VKVc4ifW7RQ7anaKQFfmR641HiO40Q+RHFIBGaBdl8u5NrHTKHL7pQu8T/APDrg9PiRZmARECh7Ntv5Hs0trkG0zRUDQ4vrZoezeuNne42Mvx5nUYXrZ+wgUnzl2nJyzVjEjESDlHXiVvDARipmiyIg2cY6+dfIPNwXReBNg71AJByB0pGBDkpNC3MlztKTLaUaktdqn7iHZeTuKcA4yCvObGbdlBOuaJUis9+iFEhAF9L6uH48WWz7/RjhKWH+7uBO1A8FGX16YhvRO0A46qS5AIHFIB3dRiFrcoiMDOuwdLlnZ/5C9y2/vw3Pf8yF0kYm4JSROtrQX27ryMlYpWzX3ueUWzSsaF8uRW9xKMqfoFT43uKrr5x3yK9moi443oHiYmZl5KpGzq9vDmvuxU1Emmx2CJmCM1ecRkjNwDkR6HKCge0EgCZmqGr9uDmDE4vjxR7A1Q153bjSrg/chzb4+eotMdRcIlSRcNX4+VPUvY07hn0pc71fKRR7kRxF4F7JAj9yYzdXwD6BZ8frsWz6Hbfs2t2uGOxWEZuaNW/wuvx2GmpqiE40PCMgfHRwoKpiXzJ5pEMDzzW4l+1QbUNL5+XjVZ5fJ9+p3a9mgBN+IiAs6rQcRy7+pRBEB2aUY7ed+6XbWEWVqTeQhGXY1HEEYJxu96HNsIcJOkHK3U0RZZk0EqHjQgRmsEtMhxKHy2A9N4f3n74RFwPwRIAdipNv2GCjCj71TPGIrb75oINuZdHtfYNaupk93OOFKgRWs8r+R/Kw4oBNxXU6OHYrS5NbFrNAHv+opIag1RlH+z7KYstYYwSnonxvdLttVEr2+PRAz7uxchLborPSzQiyQgfpUyUukFmteSVbrOANmXopXMaP2P/A0ffwCxEk+/15OnUKBg2eZDTrw+wfVvmVJIaKwycPym3zf178NzUbW5adKTk09q8QOg5vcx1iY09LIEgHcDz0pEzqboqYDRghh5FFh7evBRcl1JAUrtKZYanohAxss/MlLgXsk03UjXRLNBUOxAhju7mlg2e9FGZLKF2K8p+npnh2rqZGcN2g2+b/XGqywlqRhRc6uRguIxB+IQUV/vK2akwoxf1NTIrmTI66nHFzJeQlEEQBrTiXg/w2FXzHTeXXNFfEdfLmKlqlletK/NV6c6EDM9ydwjPxPD+ErPyFQTcuubquH9o146JvUuASfpWOLWvoubSAhSM13+twbrRvpzRM6QJykNOc4TCjtPOC7avbUGVSee8Zl5RoSqzqFqMG9w2ZWeJUwCXkmo/AoIe5i5xOQdRnEE03t3LADwnuT0+2rEfQpcNfo4jafU6dW9rxNuSh+3MCPTy3tWPizHGWPXB1jdQ5Xt/DJDM6p7VqQ7YouQwfTq3tzAqJSh32Z9zpaHvJhHh3V5dW8+PPI0j7A2h/nble9xPeLmcPfKZlXzgIhEXd43BXbnqTHc/vW+/fYDkVrl8PK8ldp8I/5Kte4241aUDDg5eY8S3RSLqanQbZa9TXGEyu9H78GM2J9d1ADUVwGheEUftzdVuKe7e4swEIGm8nb30mM6v1TcVhtTajIHoIexq6ofJQ2N2rZZYouge4Ql+1fiaSq3CYO8c95JycepvFaNF+VwWGOJi2H/+/fzv728lUqf9waQ9e6J0AKiXtTctUEQnMFlcIZ7rtY8dtshgzNZ0PMbY1OcVCrSixTlAAVor2zS6Km5dROSE05VtDO5rvyefEWdVbSSkPl6qTpWvIQq+tDKHbCoKe7mBvH8biQ8oQCNgEiHe0spwo5KwE13Qp2vadyf+A/PJJtJ6RrX64wSz5Pz/P4447EFyGOo6v4dv7m91n3Wdq2JB+RipJ5G0DhdwklXjylTAobnmr0+uZXxr8GoVzzTLJ7xr8Go14pq33Lmd38lwxEqXbcbQam9ZU/jjePXC1w/YtbFDVdwakSB20vkMBvxdhyvbpqzpsP11dffRqjRGXYeCbToOl8jWk1MYpPj7eV0u/Tecs0tOs57JPunH2ALZj2uVA3BtYbfamAmEfN3ApzQ1Uh3bGlsK6/80g8dlxk2B5xXXJ5Yp47gvijfmU75OEPu/fghf4GP4SEEoVPXhNTRbHPLZewwj8ySPjajJXZdorbF6KYR4XMMf2hFQtOJeKeYmM6rXkFriWLjRuG9y1KM0RLFHjJyWedaotgl/vqIOeOpnaS/qOBcxjzi0Tfe7YJ1e7lta6/39fc22k0NcgYIj31oOXW9OEN9dwWg8lkV0LRMsbenVHASdi8SwcajMQsq0T43CHB6fbyMbtzSslItsWs0sCDixJ/TaESM1ytKxvVVRr619IKpEaHwuhKwXss8Hnp07WMnz6sXi56ZbKndHPVA+2W0LQjgWfH4CypMvgAQKY8bT4dOWkN0eizvTbIrgaub+DV6IpB48tpsS/R6H/8zL4iwLgHt9ZdTqqcCLEnQMrvuclnDi8et59f6DXRzNYUfLVrMx8Yzc7aSDf22HurPJ3S/H6bcLX5q3alDe152fwNgXhdf20nMYk48RNTxdX/Et57Jiz8TNC/QujcQzOez1j50blq5P89n5yGLTLtzODKvFN2xP2Vikw/PvG42QXNj93JYrQf2evLo6QQOdThtqCotWJ90pdyDOF9ljfgO0+LKlmizbFJl0bRxfOU/ZIS/wXKVem1Bm/nswFOz/v2TX12hAhdnl55+1+pmeXlPQRu/Exnmm5a3RrSR8hjR8rBoA/4ffzwna7xKAAA=\'\nINDICES_CSV_SHA256 = \'b6f444fbec48681c1e5b46a5c22aa197ad91c7fbbdaff82c3ed3a9c63f7c15a4\'\nSCHEMA_CFSV2 = \'worcap_cfsv2_pentad_media_auditada_v1\'\nPOLITICA_CFSV2 = \'ensemble_completo_ou_excecao_201908_m17_auditada_v2\'\nFONTE_CFSV2 = \'https://iridl.ldeo.columbia.edu/SOURCES/.Models/.NMME/.NCEP-CFSv2/.HINDCAST/.PENTAD_SAMPLES_FULL/.prec/\'\nFEATURE_SEAS5 = \'seas5_tp_prevista_media_T_emitida_Tmenos1\'\nFEATURE_SEAS5_ANOM = \'seas5_tp_anomalia_mensal_treino_fold\'\nFEATURE_CFSV2 = \'cfsv2_tp_prevista_media_T_emitida_Tmenos1\'\nFEATURE_CFSV2_ANOM = \'cfsv2_tp_anomalia_mensal_treino_fold\'\nNOMES_MODELOS = [\'arvores_seas5\', \'arvores_multissistema\']\nFASES_CFSV2 = dict(desenvolvimento=(\'1982-02-01\', \'2020-12-01\', 467), somente_ajuste_final=(\'2021-01-01\', \'2022-12-01\', 24), teste=(\'2023-01-01\', \'2024-12-01\', 24))\nMOS_ALPHA = 0.1\nMOS_FRACAO = 0.25\n\ndef exigir(condicao, mensagem):\n    if not bool(condicao):\n        raise ValueError(mensagem)\n\ndef sha256(arquivo):\n    digest = hashlib.sha256()\n    with Path(arquivo).open(\'rb\') as stream:\n        for bloco in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(bloco)\n    return digest.hexdigest()\n\ndef salvar_json(nome, conteudo):\n    caminho = SAIDA / nome\n    caminho.write_text(json.dumps(conteudo, indent=2, ensure_ascii=False, default=str, allow_nan=False), encoding=\'utf-8\')\n    return caminho\n\ndef localizar_dados(pasta_manual=None):\n    obrigatorios = (\'treino_tp.nc\', \'treino_tp_alvo.nc\', \'teste_features.nc\', \'sample_submission.csv\')\n    if pasta_manual is not None:\n        candidatas = [Path(pasta_manual)]\n    elif Path(\'/kaggle/input\').is_dir():\n        candidatas = sorted({p.parent for p in Path(\'/kaggle/input\').rglob(obrigatorios[0])})\n    else:\n        candidatas = [Path.cwd() / \'data\' / \'raw\', Path.cwd().parent / \'data\' / \'raw\']\n    validas = sorted({p.resolve() for p in candidatas if all(((p / nome).is_file() for nome in obrigatorios))})\n    exigir(len(validas) == 1, f\'Exactly one folder containing the four official files is required. Attach the competition data or set PASTA_DADOS_MANUAL. Complete folders found: {validas}\')\n    return validas[0]\n\ndef conferir_campo(campo, referencia, datas):\n    exigir(set(campo.dims) == {\'time\', \'lat\', \'lon\'}, \'Unexpected atmospheric dimensions.\')\n    for eixo in (\'lat\', \'lon\'):\n        exigir(np.array_equal(campo[eixo].values, referencia[eixo].values), f\'Misaligned atmospheric grid: {eixo}.\')\n    exigir(pd.DatetimeIndex(campo.time.values).equals(pd.DatetimeIndex(datas)), \'Misaligned atmospheric dates.\')\n\ndef ajustar_clima_atmosfera(campos, origens):\n    origens = pd.DatetimeIndex(origens)\n    resultado = {}\n    for nome in VARIAVEIS:\n        campo = campos[nome]\n        indices = pd.DatetimeIndex(campo.time.values).get_indexer(origens)\n        exigir((indices >= 0).all(), \'Missing atmospheric fitting months.\')\n        mapas = []\n        for mes in range(1, 13):\n            ids = indices[origens.month == mes]\n            exigir(len(ids) > 0, \'Calendar month with no atmospheric history.\')\n            valores = campo.values[ids]\n            exigir(np.isfinite(valores).all(), f\'{nome}: nonfinite training values.\')\n            mapas.append(valores.mean(axis=0, dtype=np.float64).astype(np.float32))\n        resultado[nome] = np.stack(mapas)\n    return resultado\n\ndef reconstruir_chuva(clima_tp, anomalias, peso):\n    peso = float(peso)\n    exigir(np.isfinite(peso) and peso >= 0, \'The weight must be finite and nonnegative.\')\n    meses = pd.DatetimeIndex(anomalias.time.values).month.to_numpy() - 1\n    valores = np.maximum(0, clima_tp.values[meses] + peso * anomalias.values)\n    exigir(np.isfinite(valores).all(), \'Rainfall reconstruction produced invalid values.\')\n    return xr.DataArray(valores.astype(np.float32), dims=anomalias.dims, coords=anomalias.coords, name=\'tp_mm_day\', attrs={\'units\': \'mm/day\'})\n\ndef auditar_chuva_desenv():\n    with xr.open_dataset(PASTA / \'treino_tp.nc\') as ds, xr.open_dataset(PASTA / \'treino_tp_alvo.nc\') as da:\n        bruto, alvo = (ds.tp, da.tp_alvo)\n        esperado = pd.date_range(\'1940-01-01\', CORTE_FINAL, freq=\'MS\')\n        exigir(set(bruto.dims) == set(alvo.dims) == {\'time\', \'lat\', \'lon\'}, \'Unexpected precipitation dimensions.\')\n        bruto, alvo = (bruto.transpose(\'time\', \'lat\', \'lon\'), alvo.transpose(\'time\', \'lat\', \'lon\'))\n        exigir(bruto.shape == alvo.shape == (996, 301, 261), \'Unexpected grid or calendar.\')\n        exigir(pd.DatetimeIndex(bruto.time.values).equals(esperado), \'Official calendar mismatch.\')\n        exigir(bruto.attrs.get(\'units\') == alvo.attrs.get(\'units\') == \'mm/day\', \'Rainfall must use mm/day; no conversion is applied.\')\n        for eixo in [\'time\', \'lat\', \'lon\']:\n            exigir(np.array_equal(bruto[eixo].values, alvo[eixo].values), f\'Misaligned target at {eixo}.\')\n        for eixo in [\'lat\', \'lon\']:\n            exigir(np.all(np.diff(bruto[eixo].values) == 0.25), f\'Invalid grid: {eixo}.\')\n        chuva = bruto.sel(time=slice(None, FIM_DESENVOLVIMENTO)).load()\n        exigir(np.isfinite(chuva.values).all() and (chuva.values >= 0).all(), \'Invalid historical precipitation.\')\n        for i in range(0, chuva.sizes[\'time\'] - 1, 24):\n            j = min(i + 24, chuva.sizes[\'time\'] - 1)\n            exigir(np.array_equal(alvo.isel(time=slice(i, j)).values, chuva.values[i + 1:j + 1]), \'tp_alvo[M] does not match tp[M+1].\')\n    return chuva\n\ndef carregar_atmosfera(referencia, inicio, fim):\n    resultado, metadados = ({}, [])\n    esperado_completo = pd.date_range(\'1940-01-01\', CORTE_FINAL, freq=\'MS\')\n    esperado_recorte = pd.date_range(inicio, fim, freq=\'MS\')\n    for nome in VARIAVEIS:\n        with xr.open_dataset(PASTA / f\'treino_{nome}.nc\') as ds:\n            bruto = ds[nome].transpose(\'time\', \'lat\', \'lon\')\n            conferir_campo(bruto, referencia, esperado_completo)\n            campo = bruto.sel(time=slice(inicio, fim)).astype(\'float32\').load()\n            conferir_campo(campo, referencia, esperado_recorte)\n            exigir(np.isfinite(campo.values).all(), f\'{nome}: nonfinite atmospheric data.\')\n            resultado[nome] = campo\n            metadados.append(dict(variavel=nome, atributos=dict(bruto.attrs), inicio=str(inicio), fim=str(fim), minimo=float(campo.min()), maximo=float(campo.max())))\n        print(\'Loaded:\', nome, campo.shape, flush=True)\n    return (resultado, metadados)\n\ndef carregar_indices_incorporados():\n    dados = gzip.decompress(base64.b64decode(INDICES_CSV_GZIP_BASE64))\n    exigir(hashlib.sha256(dados).hexdigest() == INDICES_CSV_SHA256, \'Embedded NOAA snapshot hash mismatch.\')\n    tabela = pd.read_csv(io.BytesIO(dados), parse_dates=[\'time_origem\'])\n    exigir(tabela.columns.tolist() == [\'time_origem\'] + INDICES_NOMES, \'Ocean index columns differ from the reference.\')\n    tabela = tabela.set_index(\'time_origem\')\n    conferir_indices(tabela)\n    esperado = pd.date_range(FONTES_INDICES[\'inicio\'], FONTES_INDICES[\'fim\'], freq=\'MS\')\n    exigir(tabela.index.equals(esperado), \'The NOAA snapshot has missing or altered dates.\')\n    return (tabela, dados)\n\ndef conferir_indices(tabela):\n    exigir(tabela.columns.tolist() == INDICES_NOMES, \'The order of the four ocean indices changed.\')\n    datas = pd.DatetimeIndex(tabela.index)\n    exigir(len(datas) > 0 and datas.is_unique and datas.is_monotonic_increasing, \'NOAA dates are empty, duplicated or unordered.\')\n    exigir(datas.equals(pd.date_range(datas.min(), datas.max(), freq=\'MS\')), \'NOAA dates must be consecutive first-of-month timestamps.\')\n    exigir(np.isfinite(tabela.to_numpy(dtype=np.float64)).all(), \'Missing or infinite NOAA index. Do not interpolate or fill from future months.\')\n\ndef auditar_dataset(ds, esperado=None):\n    exigir(ds.attrs.get(\'schema\') == SCHEMA, \'Unknown SEAS5 artifact version.\')\n    exigir(ds.attrs.get(\'dataset\') == DATASET and str(ds.attrs.get(\'system\')) == SYSTEM, \'Invalid SEAS5 provenance.\')\n    exigir(ds.attrs.get(\'product_type\') == \'monthly_mean\' and int(ds.attrs.get(\'leadtime_month\')) == 2, \'Incorrect product or lead time.\')\n    exigir(ds.seas5_tp_media.attrs.get(\'units\') == \'mm/day\', \'SEAS5 must use mm/day after conversion.\')\n    exigir(ds.seas5_tp_media.dims == (\'time\', \'lat\', \'lon\'), \'Unexpected SEAS5 dimensions.\')\n    datas = pd.DatetimeIndex(ds.time.values)\n    exigir(datas.is_unique and datas.is_monotonic_increasing, \'Duplicate or unordered targets.\')\n    exigir(datas.equals(pd.date_range(datas.min(), datas.max(), freq=\'MS\')), \'Missing SEAS5 target months.\')\n    if esperado is not None:\n        exigir(datas.equals(pd.DatetimeIndex(esperado)), \'SEAS5 dates do not match the request.\')\n    exigir(pd.DatetimeIndex(ds.time_origem.values).equals((datas.to_period(\'M\') - 1).to_timestamp()), \'SEAS5 must be initialized exactly in T-1.\')\n    exigir(np.isin(ds.n_membros.values, [25, 51]).all(), \'Incomplete ensemble member set.\')\n    for eixo in [\'lat\', \'lon\']:\n        v = ds[eixo].values\n        exigir(len(v) > 1 and np.all(np.diff(v) == 1), f\'Incorrect native one-degree grid: {eixo}.\')\n    exigir(ds.lat.min() <= -60 and ds.lat.max() >= 15 and (ds.lon.min() <= -90) and (ds.lon.max() >= -25), \'The input domain does not cover the official grid. Do not extrapolate.\')\n    exigir(np.isfinite(ds.seas5_tp_media.values).all() and (ds.seas5_tp_media.values >= 0).all(), \'Invalid SEAS5 values.\')\n\ndef localizar_seas5():\n    if PASTA_SEAS5_MANUAL:\n        candidatos = [Path(PASTA_SEAS5_MANUAL) / \'seas5_51_manifesto.json\']\n    else:\n        candidatos = []\n        for raiz in [Path(\'/kaggle/input\'), Path(\'/kaggle/working/worcap_SEAS5_dados\'), Path.cwd() / \'outputs/worcap_SEAS5_dados\']:\n            if raiz.is_dir():\n                candidatos.extend(raiz.rglob(\'seas5_51_manifesto.json\'))\n    candidatos = sorted(set((p.resolve() for p in candidatos if p.is_file())))\n    exigir(len(candidatos) == 1, \'Attach the prepared SEAS5 dataset or set PASTA_SEAS5_MANUAL. Exactly one SEAS5 manifest is required.\')\n    caminho = candidatos[0]\n    m = json.loads(caminho.read_text(encoding=\'utf-8\'))\n    exigir(m.get(\'schema\') == SCHEMA and m.get(\'system\') == SYSTEM and (m.get(\'dataset\') == DATASET), \'Incompatible SEAS5 manifest.\')\n    exigir(m.get(\'leadtime_month\') == 2 and m.get(\'paramId\') == 172228 and (m.get(\'product_type\') == \'monthly_mean\') and (m.get(\'variable\') == \'total_precipitation\'), \'SEAS5 product does not match the experiment.\')\n    return (caminho.parent, m)\n\ndef carregar_seas5(uso, referencia):\n    item = MANIFESTO_SEAS5[\'arquivos\'][uso]\n    caminho = PASTA_SEAS5 / item[\'arquivo\']\n    exigir(caminho.is_file() and sha256(caminho) == item[\'sha256\'], f\'Missing or altered SEAS5: {uso}.\')\n    with xr.open_dataset(caminho) as ds:\n        bruto = ds.load()\n    auditar_dataset(bruto, pd.date_range(item[\'inicio\'], item[\'fim\'], freq=\'MS\'))\n    exigir(bruto.attrs.get(\'uso\') == uso, \'SEAS5 file partition mismatch.\')\n    grade = bruto.seas5_tp_media.interp(lat=referencia.lat, lon=referencia.lon, method=\'linear\').astype(np.float32)\n    exigir(np.isfinite(grade.values).all() and (grade.values >= 0).all(), \'Invalid SEAS5 interpolation.\')\n    for coord in [\'lat\', \'lon\']:\n        exigir(np.array_equal(grade[coord].values, referencia[coord].values), \'Misaligned SEAS5 grid.\')\n    grade.attrs.update(units=\'mm/day\', system=SYSTEM, leadtime_month=2)\n    return grade\n\ndef valores_seas5(seas, destino, pontos=None):\n    destino = pd.Timestamp(destino)\n    exigir(pd.DatetimeIndex(seas.time.values).is_unique, \'Duplicate SEAS5 targets.\')\n    i = pd.DatetimeIndex(seas.time.values).get_indexer([destino])[0]\n    exigir(i >= 0, f\'Missing SEAS5 prediction for {destino.date()}.\')\n    origem = pd.Timestamp(seas.time_origem.values[i])\n    exigir(origem == (destino.to_period(\'M\') - 1).to_timestamp() and origem < destino, \'SEAS5 initialization is incorrect or beyond the cutoff.\')\n    v = seas.values[i].reshape(-1)\n    if pontos is not None:\n        v = v[pontos]\n    exigir(np.isfinite(v).all(), \'Nonfinite SEAS5 feature.\')\n    return v\n\ndef amostrar_m5_documentado(chuva, atmosfera, seas, corte):\n    destinos = calendario_pareado(corte)\n    origens = (destinos.to_period(\'M\') - 1).to_timestamp()\n    ref_origens, ref_alvos = calendario_pares(atmosfera[VARIAVEIS[0]].time.values, corte, 30)\n    clima, somas, contagens = estatisticas_climatologia(chuva, corte, 30)\n    exigir(np.all(contagens == 30), \'Rainfall climatology must contain exactly 30 years.\')\n    ca = ajustar_clima_atmosfera(atmosfera, ref_origens)\n    ca[\'_clima_indices\'] = ajustar_clima_indices(INDICES_OC, ref_origens, corte)\n    exigir(set(destinos).issubset(set(ref_alvos)), \'Example outside the reference window.\')\n    ngrade = chuva.sizes[\'lat\'] * chuva.sizes[\'lon\']\n    n = min(PONTOS_POR_MES, ngrade)\n    X = np.empty((len(destinos) * n, 28), dtype=np.float32)\n    y = np.empty(len(X), dtype=np.float32)\n    ids = np.empty((len(destinos), n), dtype=np.int32)\n    for i, t in enumerate(destinos):\n        pontos = np.random.default_rng(np.random.SeedSequence([SEMENTE, t.year, t.month])).choice(ngrade, size=n, replace=False)\n        ids[i] = pontos\n        basicas = matriz_mes(atmosfera, ca, clima, t, pontos)\n        obs = chuva.sel(time=t).values.reshape(-1)[pontos]\n        basicas[:, 22] = ((somas[t.month - 1].reshape(-1)[pontos] - obs.astype(np.float64)) / 29).astype(np.float32)\n        sl = slice(i * n, (i + 1) * n)\n        X[sl, :27] = basicas\n        X[sl, 27] = valores_seas5(seas, t, pontos)\n        y[sl] = obs - basicas[:, 22]\n    exigir(np.isfinite(X).all() and np.isfinite(y).all(), \'Nonfinite training data.\')\n    exigir((origens < destinos).all() and destinos.max() <= pd.Timestamp(corte), \'Temporal cutoff violated.\')\n    return (X, y, ids, destinos, clima, ca, somas, contagens)\n\ndef valores_anomalia_seas5(seas, clima_seas, destino, pontos=None):\n    for eixo in [\'lat\', \'lon\']:\n        exigir(np.array_equal(seas[eixo].values, clima_seas[eixo].values), \'Misaligned SEAS5 climatology.\')\n    bruto = valores_seas5(seas, destino, pontos)\n    referencia = clima_seas.sel(month=pd.Timestamp(destino).month).values.reshape(-1)\n    if pontos is not None:\n        referencia = referencia[pontos]\n    anomalia = (bruto - referencia).astype(np.float32)\n    exigir(np.isfinite(anomalia).all(), \'Nonfinite SEAS5 anomaly.\')\n    return anomalia\n\ndef amostrar_arvores_seas5(chuva, atmosfera, seas, corte):\n    x5, y, ids, destinos, clima, ca, somas, contagens = amostrar_m5_documentado(chuva, atmosfera, seas, corte)\n    clima_seas = ajustar_clima_seas5(seas, corte)\n    X = np.empty((len(x5), 29), dtype=np.float32)\n    X[:, :28] = x5\n    n = ids.shape[1]\n    for i, t in enumerate(destinos):\n        X[i * n:(i + 1) * n, 28] = valores_anomalia_seas5(seas, clima_seas, t, ids[i])\n    exigir(np.array_equal(X[:, :28], x5), \'The 28 base SEAS5 features changed.\')\n    exigir(np.isfinite(X).all() and np.isfinite(y).all(), \'Nonfinite training data.\')\n    del x5\n    return (X, y, ids, destinos, clima, ca, somas, contagens, clima_seas)\n\ndef validar_manifesto_cfsv2(m):\n    exigir(m.get(\'schema\') == SCHEMA_CFSV2 and m.get(\'modelo\') == \'NCEP-CFSv2\' and (m.get(\'fonte\') == FONTE_CFSV2), \'Incompatible CFSv2 manifest.\')\n    exigir(m.get(\'lead_iri\') == 1.5 and m.get(\'unidade\') == \'mm/day\' and (m.get(\'chuva_observada_utilizada\') is False), \'Incorrect CFSv2 product or temporal alignment.\')\n    exigir(m.get(\'meses\') == 515 and m.get(\'inicio_alvos\') == \'1982-02-01\' and (m.get(\'fim_alvos\') == \'2024-12-01\'), \'CFSv2 coverage mismatch.\')\n    exigir(m.get(\'politica_membros\') == POLITICA_CFSV2 and m.get(\'excecoes_permitidas\') == {\'2019-08-01\': [17]}, \'CFSv2 ensemble policy mismatch.\')\n    aplicadas = m.get(\'excecoes_aplicadas\', [])\n    exigir(isinstance(aplicadas, list) and len(aplicadas) <= 1, \'Unspecified CFSv2 exceptions.\')\n    if aplicadas:\n        r = aplicadas[0]\n        exigir(r.get(\'time_origem\') == \'2019-08-01\' and r.get(\'time_alvo\') == \'2019-09-01\' and (r.get(\'membros\') == 23) and (r.get(\'membros_esperados\') == 24) and (r.get(\'membros_indisponiveis_ids\') == [17]), \'Incorrect CFSv2 exception.\')\n        a = r.get(\'auditoria_excecao\', {})\n        exigir(a.get(\'membros_indisponiveis_ids\') == [17] and np.isfinite(a.get(\'maximo_delta_media_mm_day\', np.nan)) and (0 <= a[\'maximo_delta_media_mm_day\'] <= 1e-05), \'Exception missing the individual-member mean check.\')\n    for fase, (inicio, fim, n) in FASES_CFSV2.items():\n        item = m.get(\'arquivos\', {}).get(fase, {})\n        exigir(item.get(\'arquivo\') == f\'cfsv2_{fase}.nc\' and item.get(\'inicio_alvos\') == inicio and (item.get(\'fim_alvos\') == fim) and (item.get(\'meses\') == n), \'Incorrect CFSv2 partition.\')\n\ndef localizar_cfsv2():\n    if PASTA_CFSV2_MANUAL:\n        candidatos = [Path(PASTA_CFSV2_MANUAL) / \'cfsv2_manifesto.json\']\n    else:\n        candidatos = []\n        for raiz in [Path(\'/kaggle/input\'), Path(\'/kaggle/working/worcap_CFSv2_dados\'), Path.cwd() / \'outputs/worcap_CFSv2_dados\']:\n            if raiz.is_dir():\n                candidatos.extend(raiz.rglob(\'cfsv2_manifesto.json\'))\n    candidatos = sorted(set((p.resolve() for p in candidatos if p.is_file())))\n    exigir(len(candidatos) == 1, \'Attach the prepared CFSv2 output or set PASTA_CFSV2_MANUAL. Exactly one CFSv2 manifest is required.\')\n    caminho = candidatos[0]\n    m = json.loads(caminho.read_text(encoding=\'utf-8\'))\n    validar_manifesto_cfsv2(m)\n    return (caminho.parent, m)\n\ndef auditar_cfsv2(ds, uso, m):\n    inicio, fim, n = FASES_CFSV2[uso]\n    datas = pd.date_range(inicio, fim, freq=\'MS\')\n    origens = (datas.to_period(\'M\') - 1).to_timestamp()\n    exigir(ds.attrs.get(\'schema\') == SCHEMA_CFSV2 and ds.attrs.get(\'modelo\') == \'NCEP-CFSv2\' and (ds.attrs.get(\'fonte\') == FONTE_CFSV2) and (ds.attrs.get(\'lead_iri\') == 1.5), \'CFSv2 NetCDF provenance mismatch.\')\n    exigir(ds.attrs.get(\'politica_membros\') == POLITICA_CFSV2, \'CFSv2 NetCDF is missing the corrected ensemble policy.\')\n    exigir(ds.cfsv2_tp_media.dims == (\'time\', \'lat\', \'lon\') and ds.cfsv2_tp_media.attrs.get(\'units\') == \'mm/day\', \'Invalid CFSv2 dimensions or units.\')\n    exigir(pd.DatetimeIndex(ds.time.values).equals(datas), \'Missing, extra or unordered CFSv2 dates.\')\n    exigir(pd.DatetimeIndex(ds.time_origem.values).equals(origens), \'CFSv2 nominal issue month is not T-1.\')\n    exigir(np.array_equal(ds.lat.values, np.arange(-61, 17)) and np.array_equal(ds.lon.values, np.arange(-91, -23)), \'CFSv2 native grid mismatch.\')\n    for var in [\'n_membros\', \'inicializacao_mais_antiga\', \'inicializacao_mais_recente\', \'fase_fonte\', \'time_origem\']:\n        exigir(ds[var].dims == (\'time\',), f\'Invalid CFSv2 coordinate: {var}.\')\n    dmin = pd.DatetimeIndex(ds.inicializacao_mais_antiga.values)\n    dmax = pd.DatetimeIndex(ds.inicializacao_mais_recente.values)\n    exigir(not dmin.hasnans and (not dmax.hasnans) and (dmin <= dmax).all() and (dmax < datas).all() and (dmin >= origens - pd.Timedelta(days=40)).all() and (dmax <= origens + pd.Timedelta(days=7)).all(), \'CFSv2 initializations are outside the allowed window.\')\n    esperados = np.where(origens.month == 11, 28, 24)\n    if m[\'excecoes_aplicadas\'] and pd.Timestamp(\'2019-09-01\') in datas:\n        esperados[datas == pd.Timestamp(\'2019-09-01\')] = 23\n    exigir(np.array_equal(ds.n_membros.values, esperados), \'CFSv2 member counts differ from the manifest.\')\n    exigir(np.array_equal(ds.fase_fonte.values, (origens >= pd.Timestamp(\'2011-04-01\')).astype(np.int8)), \'Incorrect CFSv2 hindcast/real-time transition.\')\n    v = ds.cfsv2_tp_media.values\n    exigir(np.isfinite(v).all() and (v >= 0).all(), \'Invalid CFSv2 predicted rainfall.\')\n\ndef carregar_cfsv2(uso, referencia):\n    item = MANIFESTO_CFSV2[\'arquivos\'][uso]\n    path = PASTA_CFSV2 / item[\'arquivo\']\n    exigir(path.is_file() and sha256(path) == item[\'sha256\'], f\'Missing or altered CFSv2: {uso}.\')\n    with xr.open_dataset(path) as d:\n        bruto = d.load()\n    auditar_cfsv2(bruto, uso, MANIFESTO_CFSV2)\n    grade = bruto.cfsv2_tp_media.interp(lat=referencia.lat, lon=referencia.lon, method=\'linear\').astype(np.float32)\n    exigir(np.isfinite(grade.values).all() and (grade.values >= 0).all(), \'Invalid CFSv2 interpolation; do not extrapolate.\')\n    for eixo in [\'lat\', \'lon\']:\n        exigir(np.array_equal(grade[eixo].values, referencia[eixo].values), \'Misaligned CFSv2 grid.\')\n    for coord in [\'time_origem\', \'n_membros\', \'inicializacao_mais_antiga\', \'inicializacao_mais_recente\', \'fase_fonte\']:\n        grade = grade.assign_coords({coord: (\'time\', bruto[coord].values)})\n    grade.attrs.update(units=\'mm/day\', lead_iri=1.5)\n    salvar_json(f\'auditoria_CFSv2_{uso}.json\', dict(arquivo=str(path), sha256=item[\'sha256\'], meses=bruto.sizes[\'time\'], minimo=float(grade.min()), maximo=float(grade.max()), membros_usados=np.unique(bruto.n_membros.values).tolist(), interpolacao=\'bilinear without extrapolation\'))\n    return grade\n\ndef ajustar_clima_previsao(previsoes, corte, nome):\n    treino = calendario_pareado(corte)\n    datas = pd.DatetimeIndex(previsoes.time.values)\n    exigir(datas.is_unique and datas.is_monotonic_increasing and treino.isin(datas).all(), f\'Incomplete or unordered dates: {nome}.\')\n    sel = previsoes.sel(time=treino)\n    exigir(pd.DatetimeIndex(sel.time_origem.values).equals((treino.to_period(\'M\') - 1).to_timestamp()), f\'Incorrect origin in climatology {nome}.\')\n    v = sel.values\n    exigir(np.isfinite(v).all() and (v >= 0).all(), f\'Invalid values in climatology {nome}.\')\n    n = np.array([(treino.month == m).sum() for m in range(1, 13)], dtype=np.int16)\n    exigir((n > 0).all() and n.max() <= 30, \'Forecast climatology exceeds 30 years or has a missing calendar month.\')\n    medias = np.stack([v[treino.month == m].mean(axis=0, dtype=np.float64) for m in range(1, 13)]).astype(np.float32)\n    return xr.DataArray(medias, dims=(\'month\', \'lat\', \'lon\'), coords={\'month\': np.arange(1, 13), \'lat\': previsoes.lat, \'lon\': previsoes.lon, \'n_meses\': (\'month\', n)}, name=f\'climatologia_{nome}\', attrs=dict(units=\'mm/day\', inicio=str(treino[0].date()), corte=str(pd.Timestamp(corte).date()), meses_treino=len(treino), anos_min=int(n.min()), anos_max=int(n.max()), referencia=\'training-month predictors only; one ensemble mean per year and month\'))\n\ndef ajustar_clima_seas5(seas, corte):\n    return ajustar_clima_previsao(seas, corte, \'SEAS5\')\n\ndef valores_cfsv2(cfs, destino, pontos=None):\n    destino = pd.Timestamp(destino)\n    exigir(pd.Timestamp(cfs.inicializacao_mais_recente.sel(time=destino).values) < destino, \'A real CFSv2 initialization extends into the target month.\')\n    return valores_seas5(cfs, destino, pontos)\n\ndef valores_anomalia_cfsv2(cfs, clima_cfs, destino, pontos=None):\n    for eixo in [\'lat\', \'lon\']:\n        exigir(np.array_equal(cfs[eixo], clima_cfs[eixo]), \'Misaligned CFSv2 climatology.\')\n    bruto = valores_cfsv2(cfs, destino, pontos)\n    ref = clima_cfs.sel(month=pd.Timestamp(destino).month).values.ravel()\n    if pontos is not None:\n        ref = ref[pontos]\n    anom = (bruto - ref).astype(np.float32)\n    exigir(np.isfinite(anom).all(), \'Invalid CFSv2 anomaly.\')\n    return anom\n\n# Both tree components receive exactly the same sampled pixels and residual targets.\ndef amostrar_arvores_multissistema(chuva, atmosfera, seas, cfs, corte):\n    x6, y, ids, destinos, clima, ca, somas, contagens, cs = amostrar_arvores_seas5(chuva, atmosfera, seas, corte)\n    cc = ajustar_clima_previsao(cfs, corte, \'CFSv2\')\n    X = np.empty((len(x6), 31), dtype=np.float32)\n    X[:, :29] = x6\n    n = ids.shape[1]\n    for i, t in enumerate(destinos):\n        sl = slice(i * n, (i + 1) * n)\n        X[sl, 29] = valores_cfsv2(cfs, t, ids[i])\n        X[sl, 30] = valores_anomalia_cfsv2(cfs, cc, t, ids[i])\n    exigir(np.array_equal(X[:, :29], x6), \'The 29 control features changed.\')\n    exigir(np.isfinite(X).all() and np.isfinite(y).all(), \'Invalid training matrix.\')\n    return (X, y, ids, destinos, clima, ca, somas, contagens, cs, cc)\n\ndef prever_par(modelo, nome, atmosfera, seas, cfs, ca, clima, cs, cc, destinos):\n    destinos = pd.DatetimeIndex(destinos)\n    exigir(nome in NOMES_MODELOS, \'Unknown model.\')\n    exigir((destinos > pd.Timestamp(cs.attrs[\'corte\'])).all() and cs.attrs[\'corte\'] == cc.attrs[\'corte\'], \'Inference must follow the training cutoff.\')\n    out = np.empty((len(destinos), clima.sizes[\'lat\'], clima.sizes[\'lon\']), dtype=np.float32)\n    for i, t in enumerate(destinos):\n        x = np.column_stack([matriz_mes(atmosfera, ca, clima, t), valores_seas5(seas, t), valores_anomalia_seas5(seas, cs, t)]).astype(np.float32)\n        if nome == \'arvores_multissistema\':\n            x = np.column_stack([x, valores_cfsv2(cfs, t), valores_anomalia_cfsv2(cfs, cc, t)]).astype(np.float32)\n        exigir(x.shape[1] == (29 if nome == \'arvores_seas5\' else 31) and np.isfinite(x).all(), \'Invalid inference.\')\n        out[i] = modelo.predict(x).reshape(out.shape[1:])\n    exigir(np.isfinite(out).all(), \'Invalid predictions.\')\n    return xr.DataArray(out, dims=(\'time\', \'lat\', \'lon\'), coords={\'time\': destinos, \'lat\': clima.lat, \'lon\': clima.lon}, attrs={\'units\': \'mm/day\'})\n\n# Fit a local two-predictor ridge with training-only monthly means and scales.\ndef mos_fit(chuva, seas, cfs, corte):\n    dates = calendario_pareado(corte)\n    r = chuva.sel(time=dates).transpose(\'time\', \'lat\', \'lon\').values.reshape(len(dates), -1)\n    s = seas.sel(time=dates).transpose(\'time\', \'lat\', \'lon\').values.reshape(len(dates), -1)\n    c = cfs.sel(time=dates).transpose(\'time\', \'lat\', \'lon\').values.reshape(len(dates), -1)\n    for pre in [seas, cfs]:\n        assert pd.DatetimeIndex(pre.sel(time=dates).time_origem.values).equals((dates.to_period(\'M\') - 1).to_timestamp())\n    assert (pd.DatetimeIndex(cfs.sel(time=dates).inicializacao_mais_recente.values) < dates).all()\n    baseline = estatisticas_climatologia(chuva, corte, 30)[0]\n    shape = baseline.shape[1:]\n    ng = r.shape[1]\n    coef = np.zeros((2, ng), np.float64)\n    mu = np.zeros((2, 12, ng), np.float64)\n    for lo in range(0, ng, 4096):\n        hi = min(lo + 4096, ng)\n        y = r[:, lo:hi].astype(np.float64)\n        x = s[:, lo:hi].astype(np.float64)\n        z = c[:, lo:hi].astype(np.float64)\n        assert np.isfinite(y).all() and np.isfinite(x).all() and np.isfinite(z).all()\n        for m in range(1, 13):\n            ix = dates.month == m\n            mu[0, m - 1, lo:hi] = x[ix].mean(0)\n            mu[1, m - 1, lo:hi] = z[ix].mean(0)\n            x[ix] -= mu[0, m - 1, lo:hi]\n            z[ix] -= mu[1, m - 1, lo:hi]\n            y[ix] -= y[ix].mean(0)\n        sx = np.sqrt(np.mean(x * x, axis=0))\n        sz = np.sqrt(np.mean(z * z, axis=0))\n        sx = np.where(sx > 1e-12, sx, 1.0)\n        sz = np.where(sz > 1e-12, sz, 1.0)\n        x /= sx\n        z /= sz\n        a = np.mean(x * x, axis=0) + MOS_ALPHA\n        b = np.mean(x * z, axis=0)\n        d = np.mean(z * z, axis=0) + MOS_ALPHA\n        u = np.mean(x * y, axis=0)\n        v = np.mean(z * y, axis=0)\n        det = a * d - b * b\n        assert (det > 0).all()\n        coef[0, lo:hi] = (d * u - b * v) / det / sx\n        coef[1, lo:hi] = (a * v - b * u) / det / sz\n    assert np.isfinite(coef).all()\n    return dict(coef=coef.reshape(2, *shape), mu=mu.reshape(2, 12, *shape), baseline=baseline.values, lat=baseline.lat.values, lon=baseline.lon.values, corte=str(pd.Timestamp(corte).date()), inicio=str(dates[0].date()), meses=len(dates))\n\ndef mos_predict(fit, seas, cfs, dates):\n    dates = pd.DatetimeIndex(dates)\n    assert (dates > pd.Timestamp(fit[\'corte\'])).all()\n    for pre in [seas, cfs]:\n        assert np.array_equal(pre.lat, fit[\'lat\']) and np.array_equal(pre.lon, fit[\'lon\'])\n    out = []\n    for t in dates:\n        s = valores_seas5(seas, t).reshape(fit[\'baseline\'].shape[1:]).astype(np.float64)\n        c = valores_cfsv2(cfs, t).reshape(s.shape).astype(np.float64)\n        anom = fit[\'coef\'][0] * (s - fit[\'mu\'][0, t.month - 1]) + fit[\'coef\'][1] * (c - fit[\'mu\'][1, t.month - 1])\n        out.append(np.maximum(0, fit[\'baseline\'][t.month - 1].astype(np.float64) + anom))\n    return np.stack(out)\n\ndef matriz_mes(campos, clima_atmos, clima_tp, destino, pontos=None):\n    """Build features at the specified source month. The full training rainfall climatology is used here; the sampler replaces its training column with the leave-one-out value."""\n    destino = pd.Timestamp(destino)\n    origem = origem_mensal([destino], LAG_ATMOSFERA)[0]\n    nlat = clima_tp.sizes[\'lat\']\n    nlon = clima_tp.sizes[\'lon\']\n    pontos = np.arange(nlat * nlon) if pontos is None else np.asarray(pontos)\n    exigir(pontos.ndim == 1 and np.issubdtype(pontos.dtype, np.integer), \'Invalid indices.\')\n    exigir(((pontos >= 0) & (pontos < nlat * nlon)).all(), \'Point outside the grid.\')\n    X = np.empty((len(pontos), len(FEATURES)), dtype=np.float32)\n    for j, nome in enumerate(VARIAVEIS):\n        campo = campos[nome]\n        i = pd.DatetimeIndex(campo.time.values).get_indexer([origem])[0]\n        exigir(i >= 0, f\'Origin {origem.date()} missing in {nome}.\')\n        valores = campo.values[i].reshape(-1)[pontos]\n        clima_origem = clima_atmos[nome][origem.month - 1].reshape(-1)[pontos]\n        X[:, j] = valores\n        X[:, j + len(VARIAVEIS)] = valores - clima_origem\n    X[:, 18] = clima_tp.lat.values[pontos // nlon]\n    X[:, 19] = clima_tp.lon.values[pontos % nlon]\n    X[:, 20] = np.sin(2 * np.pi * destino.month / 12)\n    X[:, 21] = np.cos(2 * np.pi * destino.month / 12)\n    X[:, 22] = clima_tp.values[destino.month - 1].reshape(-1)[pontos]\n    exigir(origem < destino, \'Temporal violation: input origin is not earlier than its target.\')\n    valores_oceano = valores_indices_mes(INDICES_OC, clima_atmos[\'_clima_indices\'], destino)\n    X[:, 23:] = valores_oceano[None, :]\n    exigir(np.isfinite(X).all(), \'Features contain missing or infinite values.\')\n    return X\nSST_COMPONENTES = 8\nSST_NOME = \'ersstv5_4graus_198201_202411.nc\'\nSST_HASH = \'908034f6418833302ea432f25850ad2982d1114b747b7b298e26a350eaf2891d\'\n\ndef sst_auditar_arquivo(path):\n    with xr.open_dataset(path) as ds:\n        exigir(\'sst\' in ds and ds.sst.dims == (\'time\', \'lat\', \'lon\'), \'Incorrect SST variable or dimensions.\')\n        exigir(ds.sst.attrs.get(\'units\') == \'degC\', \'SST must use degrees Celsius.\')\n        exigir(\'ERSSTv5\' in str(ds.attrs.get(\'title\', \'\')) and str(ds.attrs.get(\'product_version\', \'\')) == \'Version 5\', \'Product differs from ERSSTv5.\')\n        t = pd.DatetimeIndex(ds.time.values)\n        exigir(t.equals(pd.date_range(\'1982-01-01\', \'2024-11-01\', freq=\'MS\')), \'Incomplete or different SST calendar.\')\n        exigir(np.array_equal(ds.lat.values, np.arange(60, -61, -4)) and np.array_equal(ds.lon.values, np.arange(0, 360, 4)), \'SST must use the 60S-60N subset on a four-degree grid.\')\n        sst = ds.sst.astype(\'float32\').load()\n        atributos = {k: str(v) for k, v in ds.attrs.items()}\n    v = sst.values\n    validos = v[np.isfinite(v)]\n    exigir(validos.size > 0 and (not np.isinf(v).any()) and (validos.min() >= -1.801) and (validos.max() <= 45), "SST values outside the product\'s physical range.")\n    return (sst, dict(arquivo=Path(path).name, sha256=sha256(path), bytes=Path(path).stat().st_size, inicio_origens=str(t[0].date()), fim_origens=str(t[-1].date()), meses=len(t), grade=[31, 90], minimo=float(validos.min()), maximo=float(validos.max()), atributos=atributos, referencia=\'https://psl.noaa.gov/data/gridded/data.noaa.ersst.v5.html\', origem=\'NOAA PSL / ERSSTv5; retrospective snapshot, without verified historical operational vintages\'))\n\n# Fit the ocean mask, monthly anomalies and PCA using training SST only.\ndef sst_ajustar_pca(sst, alvos_treino):\n    alvos_treino = pd.DatetimeIndex(alvos_treino)\n    exigir(alvos_treino.is_unique and alvos_treino.equals(pd.date_range(alvos_treino.min(), alvos_treino.max(), freq=\'MS\')), \'Invalid PCA fitting calendar.\')\n    origens = origem_mensal(alvos_treino, LAG_SST)\n    treino = sst.sel(time=origens).transpose(\'time\', \'lat\', \'lon\')\n    bruto = treino.values.reshape(len(origens), -1).astype(np.float64)\n    mascara = np.isfinite(bruto).all(axis=0)\n    exigir(mascara.any(), \'No complete training ocean grid cells.\')\n    mensal = np.full((12, bruto.shape[1]), np.nan, dtype=np.float64)\n    contagens = np.array([(origens.month == m).sum() for m in range(1, 13)])\n    exigir((contagens >= 2).all(), \'Insufficient monthly SST climatology.\')\n    for m in range(1, 13):\n        mensal[m - 1, mascara] = bruto[origens.month == m][:, mascara].mean(axis=0)\n    anom = bruto[:, mascara] - mensal[origens.month - 1][:, mascara]\n    ativos = np.flatnonzero(mascara)\n    mascara[ativos[np.std(anom, axis=0) <= 1e-08]] = False\n    latitudes = np.repeat(sst.lat.values, sst.sizes[\'lon\'])\n    pesos = np.sqrt(np.cos(np.deg2rad(latitudes[mascara]))).astype(np.float64)\n    exigir(int(mascara.sum()) > SST_COMPONENTES and len(origens) > SST_COMPONENTES, \'Insufficient dimensions for PCA.\')\n    matriz = (bruto[:, mascara] - mensal[origens.month - 1][:, mascara]) * pesos\n    exigir(np.isfinite(matriz).all(), \'Invalid PCA matrix.\')\n    with threadpool_limits(limits=1):\n        pca = PCA(n_components=SST_COMPONENTES, svd_solver=\'full\').fit(matriz)\n    return dict(mascara=mascara, clima_mensal=mensal[:, mascara], pesos_area=pesos, centro=pca.mean_, componentes=pca.components_, variancia_explicada=pca.explained_variance_ratio_, contagens_mensais=contagens, origens_treino=origens.values, lat=sst.lat.values.copy(), lon=sst.lon.values.copy())\n\ndef sst_transformar(sst, estado, alvos):\n    alvos = pd.DatetimeIndex(alvos)\n    origens = origem_mensal(alvos, LAG_SST)\n    exigir(np.array_equal(sst.lat, estado[\'lat\']) and np.array_equal(sst.lon, estado[\'lon\']), \'SST grid changed.\')\n    v = sst.sel(time=origens).values.reshape(len(origens), -1).astype(np.float64)[:, estado[\'mascara\']]\n    exigir(np.isfinite(v).all(), \'Missing SST in the training-selected ocean mask; do not impute from the future.\')\n    x = (v - estado[\'clima_mensal\'][origens.month - 1]) * estado[\'pesos_area\'] - estado[\'centro\']\n    with threadpool_limits(limits=1):\n        scores = x @ estado[\'componentes\'].T\n    exigir(scores.shape == (len(alvos), SST_COMPONENTES) and np.isfinite(scores).all(), \'Invalid PCA scores.\')\n    return scores.astype(np.float32)\n\ndef pesquisa_prever_sst(modelo, nome, atmosfera, seas, cfs, ca, clima, cs, cc, datas, scores):\n    datas = pd.DatetimeIndex(datas)\n    exigir((datas > pd.Timestamp(cs.attrs[\'corte\'])).all(), \'Inference overlaps training.\')\n    exigir(scores.shape == (len(datas), 8), \'Misaligned SST scores.\')\n    out = np.empty((len(datas), clima.sizes[\'lat\'], clima.sizes[\'lon\']), dtype=np.float32)\n    expected = (FEATURES_SEAS5 if nome == \'arvores_seas5\' else FEATURES_MULTISSISTEMA) + SST_FEATURES\n    exigir(modelo.feature_name() == expected, \'Incorrect SST feature order.\')\n    for i, t in enumerate(datas):\n        x = np.column_stack([matriz_mes(atmosfera, ca, clima, t), valores_seas5(seas, t), valores_anomalia_seas5(seas, cs, t)]).astype(np.float32)\n        if nome == \'arvores_multissistema\':\n            x = np.column_stack([x, valores_cfsv2(cfs, t), valores_anomalia_cfsv2(cfs, cc, t)]).astype(np.float32)\n        x = np.column_stack([x, np.broadcast_to(scores[i], (len(x), 8))]).astype(np.float32)\n        exigir(x.shape[1] == len(expected) and np.isfinite(x).all(), \'Invalid SST features.\')\n        out[i] = modelo.predict(x).reshape(out.shape[1:])\n    return xr.DataArray(out, dims=(\'time\', \'lat\', \'lon\'), coords=dict(time=datas, lat=clima.lat, lon=clima.lon), attrs=dict(units=\'mm/day\'))\n\ndef pesquisa_diagnosticos(obs, previsoes, bloco):\n    """Complete fields, without pixel subsampling or hiding errors."""\n    linhas = []\n    lat = obs.lat.values\n    regioes = {\'dominio_inteiro\': np.ones(lat.size, dtype=bool), \'lat_menor_que_menos35\': lat < -35, \'lat_menos35_a_menos15\': (lat >= -35) & (lat < -15), \'lat_maior_igual_menos15\': lat >= -15}\n    for modelo, pred in previsoes.items():\n        obs, pred = xr.align(obs, pred, join=\'exact\')\n        for i, t in enumerate(pd.DatetimeIndex(obs.time.values)):\n            erro = pred.values[i].astype(np.float64) - obs.values[i].astype(np.float64)\n            exigir(np.isfinite(erro).all(), \'Nonfinite diagnostic error.\')\n            for regiao, mask in regioes.items():\n                if not mask.any():\n                    continue\n                e = erro[mask]\n                w = np.broadcast_to(np.cos(np.deg2rad(lat[mask]))[:, None], e.shape)\n                linhas.append(dict(modelo=modelo, bloco=bloco, mes_alvo=str(t.date()), ano=t.year, mes=t.month, regiao=regiao, n=e.size, sse=float(np.square(e).sum()), soma_erro=float(e.sum()), soma_erro_absoluto=float(np.abs(e).sum()), sse_area=float((w * np.square(e)).sum()), peso_area=float(w.sum())))\n    return pd.DataFrame(linhas)\n\ndef pesquisa_resumir(mensais, grupos):\n    cols = [\'n\', \'sse\', \'soma_erro\', \'soma_erro_absoluto\', \'sse_area\', \'peso_area\']\n    r = mensais.groupby(grupos, as_index=False)[cols].sum()\n    r[\'rmse\'] = np.sqrt(r.sse / r.n)\n    r[\'mae\'] = r.soma_erro_absoluto / r.n\n    r[\'vies\'] = r.soma_erro / r.n\n    r[\'rmse_area\'] = np.sqrt(r.sse_area / r.peso_area)\n    return r\n\'Candidate lag policy; not evidence of historical publication dates or vintages.\'\nLAG_ATMOSFERA = 4\nLAG_INDICES = 3\nLAG_SST = 2\nLAG_CHUVA_TREINO = 4\nINICIO_COMUM = pd.Timestamp(\'1982-03-01\')\nFEATURES = [v + \'_lag4\' for v in VARIAVEIS] + [v + \'_lag4_anomalia\' for v in VARIAVEIS] + [\'latitude\', \'longitude\', \'mes_alvo_sin\', \'mes_alvo_cos\', \'clima_tp_alvo\'] + [n + \'_lag3_anomalia_fold\' for n in INDICES_NOMES]\nFEATURES_SEAS5 = FEATURES + [FEATURE_SEAS5, FEATURE_SEAS5_ANOM]\nFEATURES_MULTISSISTEMA = FEATURES_SEAS5 + [FEATURE_CFSV2, FEATURE_CFSV2_ANOM]\nSST_FEATURES = [f\'sst_lag2_pc_{i:02d}_treino_fold\' for i in range(1, 9)]\n\ndef origem_mensal(alvos, lag):\n    datas = pd.DatetimeIndex(alvos)\n    exigir(lag >= 1 and datas.equals(datas.to_period(\'M\').to_timestamp()), \'Invalid dates or lag.\')\n    return (datas.to_period(\'M\') - lag).to_timestamp()\n\ndef janela_referencia(corte, anos=30):\n    corte = pd.Timestamp(corte)\n    exigir(corte.day == 1, \'The cutoff must identify a month.\')\n    return pd.date_range(end=corte, periods=12 * anos, freq=\'MS\')\n\ndef calendario_pareado(corte):\n    datas = janela_referencia(corte)\n    datas = datas[datas >= INICIO_COMUM]\n    exigir(24 <= len(datas) <= 360, \'Invalid common window.\')\n    return datas\n\ndef calendario_pares(datas_disponiveis, corte, anos=30):\n    alvos = janela_referencia(corte, anos)\n    origens = origem_mensal(alvos, LAG_ATMOSFERA)\n    disponiveis = pd.DatetimeIndex(datas_disponiveis)\n    exigir(disponiveis.is_unique and origens.isin(disponiveis).all(), \'Missing atmospheric origins.\')\n    return (origens, alvos)\n\n# Monthly rainfall sums and counts support leave-one-out training references.\ndef estatisticas_climatologia(tp, corte, anos=30):\n    datas = janela_referencia(corte, anos)\n    treino = tp.sel(time=datas).transpose(\'time\', \'lat\', \'lon\')\n    exigir(np.isfinite(treino.values).all(), \'Incomplete training rainfall.\')\n    somas = np.stack([treino.values[datas.month == m].sum(axis=0, dtype=np.float64) for m in range(1, 13)])\n    contagens = np.array([(datas.month == m).sum() for m in range(1, 13)], dtype=np.int64)\n    exigir((contagens == anos).all(), \'Climatology requires every calendar month.\')\n    clima = xr.DataArray((somas / contagens[:, None, None]).astype(np.float32), dims=(\'month\', \'lat\', \'lon\'), coords=dict(month=np.arange(1, 13), lat=tp.lat, lon=tp.lon), attrs=dict(units=\'mm/day\', ajuste_inicio=str(datas[0].date()), ajuste_fim=str(datas[-1].date())))\n    return (clima, somas, contagens)\n\ndef ajustar_clima_indices(tabela, origens_atmosfera, corte):\n    alvos = janela_referencia(corte)\n    exigir(pd.DatetimeIndex(origens_atmosfera).equals(origem_mensal(alvos, LAG_ATMOSFERA)), \'Atmospheric reference outside the protocol.\')\n    origens = origem_mensal(alvos, LAG_INDICES)\n    valores = tabela.loc[origens, INDICES_NOMES].to_numpy(dtype=np.float64)\n    exigir(np.isfinite(valores).all(), \'Incomplete training indices.\')\n    return np.stack([valores[origens.month == m].mean(axis=0, dtype=np.float64) for m in range(1, 13)])\n\ndef valores_indices_mes(tabela, clima_indices, destino):\n    origem = origem_mensal([destino], LAG_INDICES)[0]\n    valores = tabela.loc[origem, INDICES_NOMES].to_numpy(dtype=np.float64)\n    exigir(np.isfinite(valores).all(), \'Unavailable index; do not fill.\')\n    return (valores - clima_indices[origem.month - 1]).astype(np.float32)\n\ndef exigir_disponibilidade_real(registros, emitido_em, fontes_necessarias):\n    """Validate supplied availability metadata for a future production gate. Receipt records must identify the actual bytes. This check alone cannot authenticate a historical publication timestamp."""\n    emissao = pd.Timestamp(emitido_em)\n    exigir(emissao.tzinfo is not None, \'Issue time must include a timezone.\')\n    exigir(len(registros) == len(fontes_necessarias), \'Incorrect source count.\')\n    exigir({r.get(\'fonte\') for r in registros} == set(fontes_necessarias), \'Missing or duplicate source.\')\n    for r in registros:\n        for chave in [\'disponivel_em\', \'recebido_em\', \'url\', \'sha256\', \'versao\']:\n            exigir(bool(r.get(chave)), \'Unknown availability: \' + r.get(\'fonte\', \'?\') + \'/\' + chave)\n        recebido, publicado = (pd.Timestamp(r[\'recebido_em\']), pd.Timestamp(r[\'disponivel_em\']))\n        exigir(recebido.tzinfo is not None and publicado.tzinfo is not None, \'Dates must include a timezone.\')\n        exigir(publicado <= recebido <= emissao, \'A source arrived after issue time or its chronology is invalid.\')\n        exigir(len(r[\'sha256\']) == 64 and set(r[\'sha256\']) <= set(\'0123456789abcdef\'), \'Invalid SHA-256.\')\n    return True\n\ndef calendario_emissoes(datas, corte):\n    datas = pd.DatetimeIndex(datas)\n    limite = origem_mensal([datas[0]], LAG_CHUVA_TREINO)[0]\n    exigir(pd.Timestamp(corte) <= limite, \'Training rainfall is too recent for the first issue date.\')\n    return pd.DataFrame(dict(time_alvo=datas, limite_emissao=datas - pd.Timedelta(nanoseconds=1), atmosfera=origem_mensal(datas, LAG_ATMOSFERA), indices=origem_mensal(datas, LAG_INDICES), sst=origem_mensal(datas, LAG_SST), inicializacao_sazonal=origem_mensal(datas, 1), ultimo_alvo_treino=pd.Timestamp(corte), disponibilidade_historica_comprovada=False))\n\ndef auditar_invariancia_defasada(chuva, atmosfera, seas, cfs, sst, corte, datas, clima, ca, cs, cc, pca):\n    """Mutate small copies of real data; values beyond each cutoff must not affect T."""\n    primeiro = pd.Timestamp(datas[0])\n    pontos = np.array([0, 1, chuva.sizes[\'lon\']], dtype=np.int32)\n    ch = chuva.isel(lat=slice(0, 2), lon=slice(0, 2)).copy(deep=True)\n    at = {k: v.isel(lat=slice(0, 2), lon=slice(0, 2)).copy(deep=True) for k, v in atmosfera.items()}\n    alvos_ref = janela_referencia(corte)\n    orig_at = origem_mensal(alvos_ref, LAG_ATMOSFERA)\n    c0, _, _ = estatisticas_climatologia(ch, corte)\n    a0 = ajustar_clima_atmosfera(at, orig_at)\n    a0[\'_clima_indices\'] = ajustar_clima_indices(INDICES_OC, orig_at, corte)\n    x0 = matriz_mes(at, a0, c0, primeiro)\n    ch.values[pd.DatetimeIndex(ch.time.values) > pd.Timestamp(corte)] = 123456\n    c1, _, _ = estatisticas_climatologia(ch, corte)\n    exigir(np.array_equal(c0.values, c1.values), \'Post-cutoff rainfall changed the climatology.\')\n    for v in at.values():\n        v.values[pd.DatetimeIndex(v.time.values) > origem_mensal([primeiro], LAG_ATMOSFERA)[0]] = 7654321\n    a1 = ajustar_clima_atmosfera(at, orig_at)\n    a1[\'_clima_indices\'] = a0[\'_clima_indices\']\n    exigir(all((np.array_equal(a0[k], a1[k]) for k in a0)), \'Future data changed atmospheric climatology.\')\n    exigir(np.array_equal(x0, matriz_mes(at, a1, c1, primeiro)), \'Recent or future atmospheric data changed T.\')\n    ni = INDICES_OC.copy(deep=True)\n    ni.loc[ni.index > origem_mensal([primeiro], LAG_INDICES)[0], :] = 99999\n    ci = ajustar_clima_indices(ni, orig_at, corte)\n    exigir(np.array_equal(ci, a0[\'_clima_indices\']), \'Future indices changed the fit.\')\n    exigir(np.array_equal(valores_indices_mes(ni, ci, primeiro), x0[0, 23:]), \'A future index changed the feature.\')\n    for pre, nome in [(seas, \'SEAS5\'), (cfs, \'CFSv2\')]:\n        pequeno = pre.isel(lat=slice(0, 2), lon=slice(0, 2)).copy(deep=True)\n        antes = ajustar_clima_previsao(pequeno, corte, nome)\n        pequeno.values[pd.DatetimeIndex(pequeno.time.values) > pd.Timestamp(corte)] = 12345\n        exigir(np.array_equal(antes, ajustar_clima_previsao(pequeno, corte, nome)), \'Future data changed seasonal climatology.\')\n    copia = sst.copy(deep=True)\n    copia.values[pd.DatetimeIndex(copia.time.values) > origem_mensal([primeiro], LAG_SST)[0]] = 12345\n    novo = sst_ajustar_pca(copia, calendario_pareado(corte))\n    for k in [\'mascara\', \'clima_mensal\', \'centro\', \'componentes\', \'origens_treino\']:\n        exigir(np.array_equal(novo[k], pca[k]), \'Future SST changed PCA: \' + k)\n    exigir(np.array_equal(sst_transformar(copia, novo, [primeiro]), sst_transformar(sst, pca, [primeiro])), \'SST more recent than T-2 changed the prediction for T.\')\n    return dict(chuva_pos_corte_invariante=True, atmosfera_apos_Tmenos4_invariante=True, indices_apos_Tmenos3_invariantes=True, sst_apos_Tmenos2_invariante=True, climas_sazonais_somente_treino=True, publicacao_historica_comprovada=False)\n', 'research/lagged_sources/ocean_indices.csv': 'time_origem,nino34,nino12,tna,tsa\n1976-07-01,0.150000000,2.020000000,-0.640000000,-0.930000000\n1976-08-01,0.310000000,1.810000000,-0.270000000,-0.680000000\n1976-09-01,0.570000000,1.230000000,-0.240000000,-0.470000000\n1976-10-01,0.860000000,0.760000000,-0.250000000,-0.580000000\n1976-11-01,0.880000000,0.440000000,-0.270000000,-0.630000000\n1976-12-01,0.640000000,0.250000000,-0.600000000,-0.470000000\n1977-01-01,0.840000000,0.290000000,-0.400000000,-0.530000000\n1977-02-01,0.550000000,-0.140000000,-0.400000000,-0.270000000\n1977-03-01,0.400000000,-0.380000000,-0.270000000,-0.340000000\n1977-04-01,-0.250000000,-0.680000000,-0.250000000,-0.340000000\n1977-05-01,-0.030000000,-0.450000000,-0.280000000,-0.390000000\n1977-06-01,0.360000000,0.080000000,-0.200000000,-0.410000000\n1977-07-01,0.450000000,0.030000000,-0.420000000,-0.400000000\n1977-08-01,0.250000000,-0.520000000,-0.620000000,-0.190000000\n1977-09-01,0.440000000,-0.500000000,-0.610000000,-0.120000000\n1977-10-01,0.890000000,-0.190000000,-0.660000000,-0.310000000\n1977-11-01,1.030000000,-0.290000000,-0.570000000,-0.270000000\n1977-12-01,1.090000000,-0.560000000,-0.440000000,-0.340000000\n1978-01-01,0.720000000,-0.070000000,-0.130000000,-0.520000000\n1978-02-01,0.510000000,-0.350000000,0.020000000,-0.560000000\n1978-03-01,0.060000000,-0.770000000,0.090000000,-0.800000000\n1978-04-01,-0.370000000,-0.370000000,0.100000000,-0.960000000\n1978-05-01,-0.500000000,-0.540000000,-0.130000000,-0.990000000\n1978-06-01,-0.450000000,-0.860000000,-0.390000000,-0.780000000\n1978-07-01,-0.480000000,-0.400000000,-0.600000000,-0.330000000\n1978-08-01,-0.620000000,-0.640000000,-0.660000000,-0.400000000\n1978-09-01,-0.600000000,-0.300000000,-0.590000000,-0.500000000\n1978-10-01,-0.340000000,-0.430000000,-0.350000000,-0.610000000\n1978-11-01,-0.150000000,0.040000000,-0.300000000,-0.610000000\n1978-12-01,0.070000000,0.230000000,-0.480000000,-0.430000000\n1979-01-01,0.050000000,0.230000000,-0.080000000,-0.490000000\n1979-02-01,0.090000000,-0.580000000,0.100000000,-0.510000000\n1979-03-01,0.130000000,-0.250000000,-0.010000000,-0.650000000\n1979-04-01,0.170000000,0.510000000,0.140000000,-0.300000000\n1979-05-01,-0.090000000,0.270000000,0.150000000,-0.270000000\n1979-06-01,0.040000000,0.760000000,0.180000000,-0.210000000\n1979-07-01,-0.200000000,0.450000000,0.010000000,-0.120000000\n1979-08-01,0.210000000,0.500000000,-0.250000000,0.070000000\n1979-09-01,0.480000000,1.020000000,-0.270000000,0.140000000\n1979-10-01,0.320000000,0.670000000,-0.320000000,-0.170000000\n1979-11-01,0.350000000,0.120000000,-0.200000000,-0.120000000\n1979-12-01,0.690000000,-0.340000000,-0.030000000,-0.280000000\n1980-01-01,0.550000000,-0.100000000,0.150000000,-0.480000000\n1980-02-01,0.330000000,-0.250000000,0.230000000,-0.610000000\n1980-03-01,0.090000000,0.020000000,0.020000000,-0.610000000\n1980-04-01,0.200000000,-0.000000000,0.400000000,-0.420000000\n1980-05-01,0.130000000,0.250000000,0.500000000,-0.380000000\n1980-06-01,0.370000000,0.210000000,0.230000000,-0.390000000\n1980-07-01,0.020000000,-0.440000000,-0.080000000,-0.310000000\n1980-08-01,-0.230000000,-0.440000000,-0.040000000,-0.160000000\n1980-09-01,-0.120000000,-0.110000000,-0.170000000,-0.040000000\n1980-10-01,-0.110000000,-0.520000000,-0.380000000,-0.250000000\n1980-11-01,0.100000000,-0.680000000,-0.280000000,-0.430000000\n1980-12-01,0.360000000,-1.130000000,-0.310000000,-0.860000000\n1981-01-01,-0.460000000,-1.290000000,0.030000000,-0.610000000\n1981-02-01,-0.450000000,-0.830000000,-0.010000000,-0.730000000\n1981-03-01,-0.020000000,-0.190000000,0.450000000,-0.990000000\n1981-04-01,-0.170000000,-0.310000000,0.180000000,-0.830000000\n1981-05-01,-0.110000000,-0.220000000,0.160000000,-0.750000000\n1981-06-01,-0.150000000,-0.060000000,-0.060000000,-0.530000000\n1981-07-01,-0.430000000,-0.560000000,-0.120000000,-0.480000000\n1981-08-01,-0.180000000,-0.550000000,-0.240000000,-0.070000000\n1981-09-01,-0.070000000,-0.240000000,-0.230000000,0.050000000\n1981-10-01,-0.030000000,-0.520000000,-0.390000000,0.120000000\n1981-11-01,-0.270000000,-0.550000000,-0.180000000,0.080000000\n1981-12-01,0.070000000,-0.170000000,-0.020000000,-0.160000000\n1982-01-01,-0.040000000,-0.120000000,-0.060000000,-0.300000000\n1982-02-01,-0.130000000,-0.350000000,-0.240000000,-0.470000000\n1982-03-01,-0.020000000,-0.720000000,-0.230000000,-0.560000000\n1982-04-01,0.240000000,-0.510000000,-0.320000000,-0.660000000\n1982-05-01,0.650000000,0.080000000,-0.240000000,-0.710000000\n1982-06-01,0.920000000,0.270000000,-0.400000000,-0.760000000\n1982-07-01,0.640000000,0.600000000,-0.540000000,-0.700000000\n1982-08-01,0.930000000,1.090000000,-0.640000000,-0.770000000\n1982-09-01,1.390000000,2.010000000,-0.660000000,-0.460000000\n1982-10-01,2.000000000,2.280000000,-0.770000000,-0.330000000\n1982-11-01,2.070000000,3.080000000,-0.770000000,-0.460000000\n1982-12-01,2.330000000,3.200000000,-0.790000000,-0.430000000\n1983-01-01,2.430000000,2.730000000,-0.610000000,-0.010000000\n1983-02-01,2.220000000,2.030000000,-0.190000000,0.110000000\n1983-03-01,1.690000000,2.340000000,0.280000000,-0.480000000\n1983-04-01,1.120000000,3.360000000,0.290000000,-0.560000000\n1983-05-01,1.120000000,4.100000000,-0.030000000,-0.800000000\n1983-06-01,0.620000000,4.560000000,0.100000000,-0.760000000\n1983-07-01,-0.110000000,3.930000000,-0.050000000,-0.560000000\n1983-08-01,-0.130000000,3.110000000,-0.330000000,-0.280000000\n1983-09-01,-0.500000000,2.070000000,-0.560000000,-0.110000000\n1983-10-01,-1.030000000,1.020000000,-0.580000000,-0.310000000\n1983-11-01,-1.130000000,0.560000000,-0.260000000,-0.120000000\n1983-12-01,-0.950000000,0.220000000,-0.060000000,0.120000000\n1984-01-01,-0.800000000,-0.290000000,-0.300000000,0.350000000\n1984-02-01,-0.530000000,-0.770000000,-0.380000000,0.380000000\n1984-03-01,-0.480000000,-0.410000000,-0.250000000,0.440000000\n1984-04-01,-0.600000000,-0.250000000,-0.340000000,0.190000000\n1984-05-01,-0.640000000,-1.110000000,-0.360000000,-0.080000000\n1984-06-01,-0.900000000,-0.920000000,-0.610000000,0.140000000\n1984-07-01,-0.400000000,-0.540000000,-0.630000000,0.290000000\n1984-08-01,-0.400000000,-0.520000000,-0.620000000,0.570000000\n1984-09-01,-0.350000000,-0.360000000,-0.620000000,0.690000000\n1984-10-01,-0.830000000,-0.610000000,-0.920000000,0.360000000\n1984-11-01,-1.180000000,-0.180000000,-0.770000000,0.130000000\n1984-12-01,-1.530000000,-0.500000000,-0.790000000,-0.010000000\n1985-01-01,-0.890000000,-0.770000000,-0.680000000,-0.020000000\n1985-02-01,-0.910000000,-1.130000000,-0.550000000,0.250000000\n1985-03-01,-0.960000000,-0.590000000,-0.770000000,-0.210000000\n1985-04-01,-0.980000000,-1.380000000,-0.810000000,-0.160000000\n1985-05-01,-0.900000000,-1.410000000,-0.910000000,-0.100000000\n1985-06-01,-0.760000000,-1.250000000,-0.540000000,-0.270000000\n1985-07-01,-0.610000000,-1.040000000,-0.320000000,0.020000000\n1985-08-01,-0.490000000,-1.020000000,-0.420000000,-0.100000000\n1985-09-01,-0.570000000,-0.830000000,-0.540000000,-0.130000000\n1985-10-01,-0.640000000,-0.820000000,-0.570000000,0.050000000\n1985-11-01,-0.430000000,-0.920000000,-0.390000000,0.060000000\n1985-12-01,-0.400000000,-0.750000000,-0.540000000,0.030000000\n1986-01-01,-0.790000000,-0.590000000,-0.860000000,-0.140000000\n1986-02-01,-0.640000000,-0.350000000,-0.800000000,0.200000000\n1986-03-01,-0.510000000,-0.840000000,-0.760000000,0.040000000\n1986-04-01,-0.410000000,-0.860000000,-0.800000000,-0.030000000\n1986-05-01,-0.470000000,-0.820000000,-0.740000000,-0.080000000\n1986-06-01,-0.080000000,-0.900000000,-0.840000000,0.100000000\n1986-07-01,0.160000000,-0.140000000,-0.650000000,-0.060000000\n1986-08-01,0.240000000,0.170000000,-0.740000000,0.010000000\n1986-09-01,0.590000000,-0.130000000,-0.340000000,0.160000000\n1986-10-01,0.900000000,0.140000000,-0.630000000,-0.080000000\n1986-11-01,1.040000000,0.280000000,-0.910000000,-0.350000000\n1986-12-01,0.980000000,0.530000000,-0.980000000,-0.240000000\n1987-01-01,1.190000000,0.770000000,-0.570000000,-0.320000000\n1987-02-01,1.170000000,1.020000000,-0.350000000,-0.190000000\n1987-03-01,1.240000000,1.150000000,0.060000000,-0.180000000\n1987-04-01,0.890000000,1.510000000,0.170000000,-0.270000000\n1987-05-01,0.910000000,1.350000000,0.170000000,-0.060000000\n1987-06-01,1.240000000,1.060000000,0.370000000,0.110000000\n1987-07-01,1.340000000,1.190000000,0.210000000,0.250000000\n1987-08-01,1.480000000,0.910000000,0.220000000,0.240000000\n1987-09-01,1.530000000,1.310000000,0.020000000,0.170000000\n1987-10-01,1.360000000,1.500000000,0.020000000,0.190000000\n1987-11-01,1.290000000,0.900000000,0.050000000,0.450000000\n1987-12-01,1.050000000,0.890000000,0.220000000,0.130000000\n1988-01-01,0.690000000,0.090000000,-0.090000000,0.300000000\n1988-02-01,0.350000000,-0.270000000,-0.200000000,0.390000000\n1988-03-01,0.290000000,-0.520000000,0.120000000,0.430000000\n1988-04-01,-0.490000000,-0.900000000,0.220000000,0.280000000\n1988-05-01,-1.050000000,-1.080000000,0.080000000,0.370000000\n1988-06-01,-1.460000000,-1.960000000,0.100000000,0.420000000\n1988-07-01,-1.540000000,-1.540000000,-0.000000000,0.200000000\n1988-08-01,-1.440000000,-1.590000000,-0.310000000,0.300000000\n1988-09-01,-1.330000000,-1.430000000,-0.320000000,0.160000000\n1988-10-01,-2.090000000,-1.260000000,-0.630000000,0.150000000\n1988-11-01,-2.180000000,-0.650000000,-0.540000000,-0.050000000\n1988-12-01,-1.980000000,-0.970000000,-0.520000000,-0.070000000\n1989-01-01,-1.950000000,-0.280000000,-0.670000000,0.080000000\n1989-02-01,-1.370000000,0.050000000,-0.660000000,-0.040000000\n1989-03-01,-1.330000000,-0.450000000,-0.820000000,0.000000000\n1989-04-01,-1.110000000,-0.490000000,-1.040000000,-0.020000000\n1989-05-01,-0.800000000,-1.160000000,-0.860000000,-0.010000000\n1989-06-01,-0.640000000,-0.730000000,-0.420000000,0.000000000\n1989-07-01,-0.470000000,-0.440000000,-0.030000000,0.000000000\n1989-08-01,-0.580000000,-0.460000000,0.050000000,0.190000000\n1989-09-01,-0.380000000,-0.470000000,-0.320000000,0.060000000\n1989-10-01,-0.400000000,-0.290000000,-0.300000000,0.060000000\n1989-11-01,-0.300000000,-0.380000000,-0.350000000,-0.240000000\n1989-12-01,-0.130000000,-0.430000000,-0.070000000,-0.510000000\n1990-01-01,0.020000000,-0.330000000,-0.330000000,-0.680000000\n1990-02-01,0.360000000,-0.270000000,-0.280000000,-0.140000000\n1990-03-01,0.200000000,-0.410000000,-0.200000000,-0.030000000\n1990-04-01,0.260000000,-0.330000000,-0.100000000,0.040000000\n1990-05-01,0.320000000,-0.230000000,-0.070000000,-0.400000000\n1990-06-01,0.000000000,-0.050000000,-0.270000000,-0.230000000\n1990-07-01,0.150000000,-0.870000000,-0.220000000,-0.370000000\n1990-08-01,0.170000000,-0.610000000,-0.040000000,-0.330000000\n1990-09-01,0.120000000,-0.280000000,0.020000000,-0.220000000\n1990-10-01,0.130000000,-0.730000000,-0.010000000,-0.130000000\n1990-11-01,0.080000000,-0.550000000,0.020000000,-0.010000000\n1990-12-01,0.320000000,-0.440000000,-0.120000000,-0.040000000\n1991-01-01,0.510000000,-0.090000000,-0.390000000,-0.210000000\n1991-02-01,0.320000000,-0.100000000,-0.290000000,-0.200000000\n1991-03-01,0.100000000,0.000000000,-0.370000000,-0.210000000\n1991-04-01,0.210000000,-0.680000000,-0.540000000,-0.000000000\n1991-05-01,0.440000000,-0.010000000,-0.620000000,0.140000000\n1991-06-01,0.650000000,0.240000000,-0.670000000,0.130000000\n1991-07-01,0.620000000,0.600000000,-0.670000000,-0.200000000\n1991-08-01,0.430000000,0.000000000,-0.620000000,-0.520000000\n1991-09-01,0.220000000,0.160000000,-0.520000000,-0.510000000\n1991-10-01,0.910000000,0.420000000,-0.530000000,-0.590000000\n1991-11-01,1.140000000,0.640000000,-0.460000000,-0.520000000\n1991-12-01,1.620000000,0.450000000,-0.470000000,-0.340000000\n1992-01-01,1.610000000,0.360000000,-0.240000000,-0.610000000\n1992-02-01,1.620000000,0.440000000,-0.140000000,-0.530000000\n1992-03-01,1.500000000,0.920000000,-0.340000000,-0.740000000\n1992-04-01,1.410000000,1.720000000,-0.440000000,-0.880000000\n1992-05-01,1.270000000,1.830000000,-0.390000000,-1.010000000\n1992-06-01,0.530000000,0.710000000,-0.360000000,-1.110000000\n1992-07-01,0.270000000,-0.050000000,-0.290000000,-0.860000000\n1992-08-01,-0.150000000,-0.240000000,-0.560000000,-0.860000000\n1992-09-01,-0.140000000,-0.500000000,-0.580000000,-0.700000000\n1992-10-01,-0.350000000,-0.260000000,-0.480000000,-0.740000000\n1992-11-01,-0.140000000,-0.250000000,-0.610000000,-0.970000000\n1992-12-01,0.040000000,-0.310000000,-0.520000000,-0.630000000\n1993-01-01,0.280000000,0.030000000,-0.500000000,-0.590000000\n1993-02-01,0.420000000,0.200000000,-0.430000000,-0.390000000\n1993-03-01,0.470000000,0.530000000,-0.380000000,-0.440000000\n1993-04-01,0.920000000,0.830000000,-0.220000000,-0.110000000\n1993-05-01,0.930000000,1.130000000,-0.250000000,-0.320000000\n1993-06-01,0.640000000,0.880000000,-0.180000000,-0.340000000\n1993-07-01,0.330000000,0.320000000,-0.440000000,-0.210000000\n1993-08-01,0.160000000,0.210000000,-0.450000000,-0.160000000\n1993-09-01,0.260000000,0.120000000,-0.500000000,0.300000000\n1993-10-01,0.360000000,0.320000000,-0.560000000,0.470000000\n1993-11-01,0.280000000,-0.230000000,-0.650000000,0.570000000\n1993-12-01,0.190000000,-0.290000000,-0.700000000,0.240000000\n1994-01-01,0.040000000,-0.070000000,-0.830000000,-0.010000000\n1994-02-01,-0.160000000,-0.330000000,-0.780000000,-0.140000000\n1994-03-01,-0.060000000,-1.130000000,-0.760000000,-0.100000000\n1994-04-01,0.030000000,-1.220000000,-0.620000000,-0.130000000\n1994-05-01,0.140000000,-0.910000000,-0.700000000,-0.230000000\n1994-06-01,0.270000000,-0.590000000,-0.670000000,-0.490000000\n1994-07-01,0.160000000,-0.820000000,-0.670000000,-0.520000000\n1994-08-01,0.520000000,-0.900000000,-0.630000000,-0.370000000\n1994-09-01,0.400000000,-0.050000000,-0.620000000,-0.300000000\n1994-10-01,0.900000000,0.650000000,-0.500000000,-0.270000000\n1994-11-01,1.140000000,0.590000000,-0.340000000,-0.110000000\n1994-12-01,1.210000000,0.680000000,-0.420000000,-0.070000000\n1995-01-01,1.100000000,0.910000000,-0.300000000,0.070000000\n1995-02-01,0.870000000,0.240000000,-0.220000000,0.340000000\n1995-03-01,0.490000000,-0.570000000,-0.210000000,0.390000000\n1995-04-01,0.270000000,-0.990000000,-0.040000000,0.290000000\n1995-05-01,0.030000000,-1.010000000,0.100000000,0.240000000\n1995-06-01,0.080000000,-0.530000000,0.230000000,0.330000000\n1995-07-01,0.030000000,-0.250000000,0.310000000,0.090000000\n1995-08-01,-0.380000000,-0.520000000,0.250000000,-0.260000000\n1995-09-01,-0.570000000,-0.280000000,0.030000000,-0.190000000\n1995-10-01,-0.720000000,-0.410000000,-0.060000000,-0.270000000\n1995-11-01,-0.780000000,-0.260000000,0.200000000,-0.060000000\n1995-12-01,-0.720000000,-0.690000000,0.170000000,-0.260000000\n1996-01-01,-0.650000000,-0.610000000,0.490000000,-0.150000000\n1996-02-01,-0.660000000,-0.320000000,0.220000000,0.160000000\n1996-03-01,-0.480000000,-0.070000000,0.070000000,0.120000000\n1996-04-01,-0.340000000,-1.280000000,0.150000000,0.340000000\n1996-05-01,-0.400000000,-0.810000000,0.160000000,0.430000000\n1996-06-01,-0.120000000,-1.070000000,-0.060000000,0.560000000\n1996-07-01,-0.140000000,-0.980000000,-0.150000000,0.410000000\n1996-08-01,-0.300000000,-0.460000000,-0.360000000,0.150000000\n1996-09-01,-0.340000000,-0.530000000,-0.220000000,-0.010000000\n1996-10-01,-0.280000000,-0.800000000,-0.340000000,-0.100000000\n1996-11-01,-0.300000000,-0.790000000,-0.280000000,-0.060000000\n1996-12-01,-0.430000000,-1.030000000,-0.150000000,-0.330000000\n1997-01-01,-0.430000000,-0.560000000,0.060000000,-0.550000000\n1997-02-01,-0.240000000,-0.030000000,0.160000000,-0.600000000\n1997-03-01,-0.060000000,0.820000000,-0.000000000,-0.820000000\n1997-04-01,0.340000000,0.980000000,0.120000000,-0.810000000\n1997-05-01,0.870000000,2.160000000,0.230000000,-0.890000000\n1997-06-01,1.150000000,3.170000000,0.290000000,-0.980000000\n1997-07-01,1.600000000,3.630000000,0.060000000,-0.530000000\n1997-08-01,1.940000000,3.980000000,-0.170000000,-0.290000000\n1997-09-01,2.100000000,4.050000000,-0.120000000,0.060000000\n1997-10-01,2.290000000,4.000000000,0.160000000,0.470000000\n1997-11-01,2.420000000,4.120000000,0.270000000,0.600000000\n1997-12-01,2.300000000,4.270000000,0.150000000,0.760000000\n1998-01-01,2.420000000,3.550000000,0.190000000,0.380000000\n1998-02-01,2.080000000,2.880000000,0.590000000,0.470000000\n1998-03-01,1.490000000,2.880000000,0.590000000,0.340000000\n1998-04-01,0.900000000,3.000000000,0.430000000,0.230000000\n1998-05-01,0.680000000,3.010000000,0.390000000,0.200000000\n1998-06-01,-0.390000000,2.000000000,0.420000000,0.350000000\n1998-07-01,-0.730000000,1.310000000,0.290000000,0.590000000\n1998-08-01,-0.830000000,0.940000000,0.330000000,0.400000000\n1998-09-01,-0.820000000,0.310000000,0.130000000,0.260000000\n1998-10-01,-1.190000000,0.120000000,0.070000000,-0.010000000\n1998-11-01,-1.230000000,-0.210000000,0.120000000,0.140000000\n1998-12-01,-1.510000000,-0.140000000,0.000000000,-0.150000000\n1999-01-01,-1.530000000,-0.230000000,-0.350000000,-0.200000000\n1999-02-01,-1.410000000,0.050000000,-0.380000000,-0.220000000\n1999-03-01,-0.920000000,0.300000000,-0.380000000,-0.020000000\n1999-04-01,-0.810000000,-0.780000000,-0.280000000,0.140000000\n1999-05-01,-0.870000000,-0.430000000,-0.090000000,0.180000000\n1999-06-01,-0.950000000,-0.550000000,-0.100000000,0.180000000\n1999-07-01,-0.840000000,-0.650000000,-0.100000000,0.380000000\n1999-08-01,-0.980000000,-0.510000000,0.010000000,0.350000000\n1999-09-01,-0.840000000,-0.970000000,0.010000000,0.090000000\n1999-10-01,-1.030000000,-0.760000000,-0.180000000,0.050000000\n1999-11-01,-1.410000000,-1.110000000,-0.180000000,0.150000000\n1999-12-01,-1.540000000,-1.060000000,-0.020000000,0.100000000\n2000-01-01,-1.790000000,-0.890000000,-0.420000000,-0.120000000\n2000-02-01,-1.530000000,-0.470000000,-0.280000000,0.120000000\n2000-03-01,-1.260000000,-0.560000000,-0.090000000,0.050000000\n2000-04-01,-0.800000000,0.460000000,-0.170000000,0.060000000\n2000-05-01,-0.800000000,-0.290000000,-0.270000000,-0.050000000\n2000-06-01,-0.750000000,-0.650000000,-0.400000000,-0.110000000\n2000-07-01,-0.570000000,-0.570000000,-0.270000000,-0.100000000\n2000-08-01,-0.360000000,-0.590000000,-0.190000000,-0.000000000\n2000-09-01,-0.390000000,-0.340000000,-0.360000000,-0.130000000\n2000-10-01,-0.550000000,-0.490000000,-0.420000000,-0.120000000\n2000-11-01,-0.750000000,-0.700000000,-0.430000000,-0.110000000\n2000-12-01,-0.920000000,-0.550000000,-0.490000000,-0.270000000\n2001-01-01,-0.880000000,-0.760000000,-0.380000000,-0.170000000\n2001-02-01,-0.630000000,-0.330000000,-0.320000000,-0.100000000\n2001-03-01,-0.480000000,0.350000000,-0.010000000,-0.060000000\n2001-04-01,-0.300000000,0.420000000,-0.110000000,0.070000000\n2001-05-01,-0.300000000,-0.640000000,-0.140000000,0.120000000\n2001-06-01,-0.110000000,-0.860000000,-0.010000000,0.040000000\n2001-07-01,0.010000000,-0.700000000,0.020000000,-0.110000000\n2001-08-01,-0.070000000,-0.770000000,0.000000000,-0.060000000\n2001-09-01,-0.280000000,-1.200000000,0.130000000,-0.040000000\n2001-10-01,-0.260000000,-1.140000000,0.060000000,-0.030000000\n2001-11-01,-0.280000000,-0.940000000,0.310000000,-0.020000000\n2001-12-01,-0.460000000,-1.120000000,0.370000000,-0.250000000\n2002-01-01,-0.140000000,-0.940000000,0.530000000,-0.190000000\n2002-02-01,0.000000000,-0.300000000,0.350000000,-0.330000000\n2002-03-01,0.110000000,0.620000000,0.210000000,-0.060000000\n2002-04-01,0.140000000,0.330000000,-0.180000000,-0.000000000\n2002-05-01,0.210000000,0.180000000,-0.400000000,0.150000000\n2002-06-01,0.680000000,-0.400000000,-0.380000000,0.020000000\n2002-07-01,0.570000000,-0.440000000,-0.330000000,0.220000000\n2002-08-01,0.700000000,-0.260000000,-0.380000000,-0.190000000\n2002-09-01,0.820000000,0.040000000,-0.310000000,-0.230000000\n2002-10-01,1.160000000,0.570000000,-0.070000000,-0.090000000\n2002-11-01,1.410000000,0.780000000,-0.090000000,-0.250000000\n2002-12-01,1.410000000,0.690000000,-0.090000000,0.080000000\n2003-01-01,0.980000000,0.100000000,0.020000000,0.220000000\n2003-02-01,0.640000000,-0.210000000,-0.170000000,0.300000000\n2003-03-01,0.480000000,-0.490000000,-0.110000000,0.280000000\n2003-04-01,-0.030000000,-0.900000000,-0.150000000,0.110000000\n2003-05-01,-0.520000000,-1.350000000,-0.140000000,0.020000000\n2003-06-01,-0.190000000,-0.910000000,-0.020000000,-0.210000000\n2003-07-01,0.140000000,-0.580000000,0.090000000,0.140000000\n2003-08-01,0.050000000,0.080000000,0.260000000,0.300000000\n2003-09-01,0.150000000,-0.430000000,0.280000000,0.340000000\n2003-10-01,0.460000000,-0.070000000,0.380000000,0.310000000\n2003-11-01,0.390000000,0.290000000,0.170000000,0.360000000\n2003-12-01,0.320000000,0.310000000,0.200000000,0.300000000\n2004-01-01,0.260000000,-0.030000000,0.250000000,0.070000000\n2004-02-01,0.170000000,-0.200000000,0.340000000,-0.050000000\n2004-03-01,-0.100000000,-0.340000000,0.200000000,-0.360000000\n2004-04-01,0.060000000,-0.450000000,0.300000000,-0.570000000\n2004-05-01,0.100000000,-0.960000000,-0.030000000,-0.470000000\n2004-06-01,0.140000000,-0.690000000,0.020000000,-0.390000000\n2004-07-01,0.410000000,-0.700000000,0.180000000,-0.180000000\n2004-08-01,0.660000000,-0.440000000,0.350000000,0.020000000\n2004-09-01,0.670000000,-0.100000000,0.210000000,0.180000000\n2004-10-01,0.730000000,0.110000000,0.200000000,-0.080000000\n2004-11-01,0.620000000,0.390000000,0.320000000,0.060000000\n2004-12-01,0.710000000,0.270000000,0.240000000,0.170000000\n2005-01-01,0.560000000,0.030000000,0.220000000,0.280000000\n2005-02-01,0.260000000,-0.610000000,0.200000000,0.220000000\n2005-03-01,0.280000000,-1.150000000,0.530000000,0.260000000\n2005-04-01,0.280000000,-0.360000000,0.650000000,-0.020000000\n2005-05-01,0.300000000,0.220000000,0.810000000,-0.400000000\n2005-06-01,0.220000000,-0.150000000,0.670000000,-0.570000000\n2005-07-01,-0.010000000,-0.060000000,0.570000000,-0.570000000\n2005-08-01,-0.040000000,-0.300000000,0.410000000,-0.190000000\n2005-09-01,-0.080000000,-0.610000000,0.350000000,-0.230000000\n2005-10-01,-0.150000000,-1.050000000,0.120000000,-0.220000000\n2005-11-01,-0.440000000,-1.410000000,0.090000000,-0.260000000\n2005-12-01,-0.750000000,-1.190000000,0.260000000,-0.290000000\n2006-01-01,-0.980000000,-0.690000000,0.160000000,-0.300000000\n2006-02-01,-0.710000000,0.330000000,0.010000000,-0.040000000\n2006-03-01,-0.730000000,-0.270000000,0.020000000,0.060000000\n2006-04-01,-0.300000000,-0.890000000,0.260000000,-0.000000000\n2006-05-01,-0.110000000,-0.390000000,0.230000000,-0.110000000\n2006-06-01,0.090000000,-0.190000000,0.280000000,0.100000000\n2006-07-01,0.030000000,0.180000000,0.090000000,0.050000000\n2006-08-01,0.370000000,0.510000000,0.200000000,0.130000000\n2006-09-01,0.620000000,0.780000000,0.310000000,0.130000000\n2006-10-01,0.760000000,1.200000000,0.310000000,0.010000000\n2006-11-01,0.980000000,0.910000000,0.400000000,-0.100000000\n2006-12-01,1.100000000,0.620000000,0.180000000,0.050000000\n2007-01-01,0.590000000,0.570000000,0.210000000,-0.090000000\n2007-02-01,0.120000000,0.060000000,0.380000000,-0.060000000\n2007-03-01,-0.150000000,-0.650000000,0.170000000,-0.310000000\n2007-04-01,-0.160000000,-1.050000000,0.240000000,-0.070000000\n2007-05-01,-0.390000000,-1.470000000,0.010000000,-0.060000000\n2007-06-01,-0.160000000,-1.400000000,-0.080000000,0.120000000\n2007-07-01,-0.370000000,-0.940000000,-0.180000000,-0.020000000\n2007-08-01,-0.570000000,-1.360000000,-0.330000000,-0.040000000\n2007-09-01,-1.040000000,-1.340000000,-0.270000000,0.120000000\n2007-10-01,-1.410000000,-1.650000000,-0.190000000,-0.110000000\n2007-11-01,-1.580000000,-1.950000000,-0.070000000,-0.080000000\n2007-12-01,-1.610000000,-1.630000000,-0.210000000,-0.060000000\n2008-01-01,-1.790000000,-0.800000000,-0.100000000,-0.140000000\n2008-02-01,-1.700000000,-0.160000000,-0.000000000,0.090000000\n2008-03-01,-1.170000000,0.330000000,0.120000000,0.250000000\n2008-04-01,-0.890000000,0.500000000,-0.060000000,0.330000000\n2008-05-01,-0.640000000,-0.320000000,0.070000000,0.470000000\n2008-06-01,-0.440000000,0.260000000,0.150000000,0.390000000\n2008-07-01,-0.040000000,0.470000000,0.170000000,0.400000000\n2008-08-01,-0.040000000,0.670000000,0.120000000,0.310000000\n2008-09-01,-0.280000000,0.320000000,0.200000000,0.130000000\n2008-10-01,-0.300000000,-0.170000000,0.010000000,0.050000000\n2008-11-01,-0.370000000,-0.210000000,-0.040000000,0.130000000\n2008-12-01,-0.900000000,-0.330000000,-0.030000000,0.150000000\n2009-01-01,-1.000000000,-0.170000000,-0.170000000,0.110000000\n2009-02-01,-0.710000000,-0.310000000,-0.550000000,0.080000000\n2009-03-01,-0.720000000,-0.850000000,-0.490000000,0.200000000\n2009-04-01,-0.250000000,0.000000000,-0.470000000,0.210000000\n2009-05-01,0.170000000,0.170000000,-0.430000000,0.440000000\n2009-06-01,0.490000000,0.540000000,-0.190000000,0.280000000\n2009-07-01,0.690000000,0.700000000,0.170000000,0.140000000\n2009-08-01,0.620000000,0.660000000,0.070000000,0.040000000\n2009-09-01,0.680000000,0.220000000,-0.030000000,0.040000000\n2009-10-01,0.960000000,0.220000000,0.200000000,0.230000000\n2009-11-01,1.490000000,0.140000000,0.200000000,0.110000000\n2009-12-01,1.810000000,0.490000000,0.250000000,0.430000000\n2010-01-01,1.430000000,0.360000000,0.550000000,0.460000000\n2010-02-01,1.180000000,0.040000000,0.900000000,0.410000000\n2010-03-01,1.070000000,-0.040000000,1.120000000,0.540000000\n2010-04-01,0.560000000,0.510000000,1.250000000,0.370000000\n2010-05-01,-0.150000000,0.390000000,1.150000000,0.270000000\n2010-06-01,-0.620000000,0.140000000,0.870000000,0.540000000\n2010-07-01,-0.890000000,-1.050000000,0.740000000,0.350000000\n2010-08-01,-1.330000000,-1.230000000,0.630000000,0.290000000\n2010-09-01,-1.560000000,-1.290000000,0.370000000,0.080000000\n2010-10-01,-1.650000000,-1.530000000,0.350000000,0.210000000\n2010-11-01,-1.570000000,-1.390000000,0.390000000,0.220000000\n2010-12-01,-1.630000000,-1.030000000,0.670000000,0.150000000\n2011-01-01,-1.700000000,-0.610000000,0.730000000,0.120000000\n2011-02-01,-1.260000000,0.070000000,0.350000000,0.320000000\n2011-03-01,-0.980000000,-0.430000000,0.250000000,0.380000000\n2011-04-01,-0.740000000,-0.050000000,0.170000000,0.180000000\n2011-05-01,-0.530000000,0.250000000,0.200000000,-0.040000000\n2011-06-01,-0.250000000,0.360000000,0.350000000,-0.110000000\n2011-07-01,-0.230000000,0.270000000,0.150000000,0.020000000\n2011-08-01,-0.660000000,-0.200000000,0.150000000,0.000000000\n2011-09-01,-0.760000000,-1.010000000,-0.010000000,0.090000000\n2011-10-01,-0.930000000,-0.710000000,0.020000000,0.060000000\n2011-11-01,-1.090000000,-0.800000000,-0.100000000,-0.230000000\n2011-12-01,-1.050000000,-0.680000000,-0.060000000,-0.740000000\n2012-01-01,-0.930000000,-0.390000000,-0.090000000,-0.660000000\n2012-02-01,-0.610000000,0.400000000,-0.190000000,-0.780000000\n2012-03-01,-0.480000000,-0.020000000,-0.410000000,-0.430000000\n2012-04-01,-0.290000000,0.480000000,-0.330000000,-0.640000000\n2012-05-01,-0.180000000,0.430000000,-0.110000000,-0.380000000\n2012-06-01,0.140000000,0.990000000,0.070000000,-0.270000000\n2012-07-01,0.440000000,0.820000000,0.020000000,-0.290000000\n2012-08-01,0.660000000,0.270000000,0.190000000,-0.280000000\n2012-09-01,0.440000000,0.040000000,0.340000000,-0.130000000\n2012-10-01,0.230000000,-0.390000000,0.350000000,-0.190000000\n2012-11-01,0.330000000,-0.460000000,0.440000000,-0.160000000\n2012-12-01,-0.130000000,-0.610000000,0.260000000,-0.090000000\n2013-01-01,-0.420000000,-0.700000000,0.190000000,0.050000000\n2013-02-01,-0.400000000,-0.320000000,0.220000000,-0.020000000\n2013-03-01,-0.140000000,-0.380000000,0.550000000,-0.090000000\n2013-04-01,-0.080000000,-1.320000000,0.370000000,0.040000000\n2013-05-01,-0.280000000,-1.440000000,0.440000000,0.130000000\n2013-06-01,-0.330000000,-1.440000000,0.070000000,-0.030000000\n2013-07-01,-0.280000000,-1.360000000,0.060000000,-0.030000000\n2013-08-01,-0.290000000,-0.760000000,0.220000000,0.050000000\n2013-09-01,-0.090000000,-0.770000000,0.230000000,-0.120000000\n2013-10-01,-0.240000000,-0.160000000,0.230000000,-0.180000000\n2013-11-01,-0.020000000,-0.500000000,0.020000000,-0.260000000\n2013-12-01,-0.090000000,-0.250000000,-0.020000000,-0.160000000\n2014-01-01,-0.420000000,0.370000000,-0.120000000,-0.230000000\n2014-02-01,-0.450000000,-0.670000000,-0.250000000,-0.050000000\n2014-03-01,-0.070000000,-0.510000000,-0.500000000,-0.130000000\n2014-04-01,0.280000000,-0.250000000,-0.360000000,-0.080000000\n2014-05-01,0.450000000,0.740000000,-0.270000000,0.140000000\n2014-06-01,0.480000000,1.170000000,-0.290000000,0.150000000\n2014-07-01,0.130000000,1.040000000,-0.130000000,-0.090000000\n2014-08-01,0.140000000,0.670000000,-0.100000000,0.050000000\n2014-09-01,0.370000000,0.150000000,0.130000000,0.110000000\n2014-10-01,0.480000000,0.340000000,0.130000000,-0.060000000\n2014-11-01,0.890000000,0.570000000,0.040000000,-0.500000000\n2014-12-01,0.770000000,0.140000000,0.050000000,-0.380000000\n2015-01-01,0.590000000,-0.160000000,-0.080000000,0.100000000\n2015-02-01,0.570000000,-0.620000000,-0.060000000,0.150000000\n2015-03-01,0.480000000,-0.010000000,-0.270000000,-0.110000000\n2015-04-01,0.900000000,0.640000000,-0.400000000,0.050000000\n2015-05-01,1.040000000,1.590000000,-0.130000000,0.110000000\n2015-06-01,1.280000000,2.190000000,-0.310000000,0.070000000\n2015-07-01,1.560000000,2.320000000,-0.120000000,-0.190000000\n2015-08-01,1.870000000,2.080000000,0.090000000,-0.310000000\n2015-09-01,2.010000000,2.480000000,0.350000000,-0.190000000\n2015-10-01,2.210000000,2.380000000,0.510000000,0.060000000\n2015-11-01,2.570000000,2.210000000,0.180000000,0.190000000\n2015-12-01,2.560000000,2.090000000,0.130000000,0.440000000\n2016-01-01,2.560000000,1.820000000,0.250000000,0.470000000\n2016-02-01,2.110000000,1.090000000,0.280000000,0.320000000\n2016-03-01,1.600000000,1.100000000,0.240000000,0.250000000\n2016-04-01,1.050000000,0.290000000,0.170000000,0.080000000\n2016-05-01,0.450000000,0.450000000,0.180000000,0.010000000\n2016-06-01,0.060000000,0.520000000,0.150000000,0.370000000\n2016-07-01,-0.250000000,-0.060000000,0.100000000,0.370000000\n2016-08-01,-0.480000000,0.190000000,0.150000000,0.330000000\n2016-09-01,-0.460000000,0.250000000,0.010000000,0.110000000\n2016-10-01,-0.750000000,-0.030000000,0.120000000,0.240000000\n2016-11-01,-0.630000000,-0.080000000,0.150000000,0.100000000\n2016-12-01,-0.510000000,0.020000000,0.230000000,0.410000000\n2017-01-01,-0.340000000,0.330000000,0.240000000,0.250000000\n2017-02-01,-0.010000000,0.930000000,0.060000000,-0.180000000\n2017-03-01,-0.090000000,1.560000000,-0.030000000,-0.060000000\n2017-04-01,0.220000000,0.490000000,0.310000000,0.090000000\n2017-05-01,0.300000000,0.250000000,0.380000000,0.120000000\n2017-06-01,0.220000000,0.200000000,0.350000000,0.260000000\n2017-07-01,0.220000000,-0.460000000,0.320000000,0.100000000\n2017-08-01,-0.180000000,-0.760000000,0.330000000,0.390000000\n2017-09-01,-0.560000000,-1.280000000,0.150000000,0.120000000\n2017-10-01,-0.520000000,-1.320000000,0.030000000,0.190000000\n2017-11-01,-0.840000000,-1.440000000,0.220000000,0.020000000\n2017-12-01,-0.850000000,-1.540000000,0.250000000,-0.090000000\n2018-01-01,-0.860000000,-1.250000000,0.030000000,0.290000000\n2018-02-01,-0.730000000,-0.430000000,-0.270000000,-0.270000000\n2018-03-01,-0.730000000,-1.300000000,-0.130000000,-0.180000000\n2018-04-01,-0.360000000,-0.810000000,-0.330000000,-0.030000000\n2018-05-01,-0.120000000,-0.460000000,-0.600000000,-0.070000000\n2018-06-01,0.120000000,-0.470000000,-0.720000000,0.150000000\n2018-07-01,0.270000000,-0.080000000,-0.480000000,0.120000000\n2018-08-01,0.050000000,-0.540000000,-0.320000000,0.210000000\n2018-09-01,0.300000000,0.120000000,-0.040000000,0.210000000\n2018-10-01,0.840000000,0.300000000,-0.130000000,0.310000000\n2018-11-01,1.000000000,0.690000000,-0.320000000,0.210000000\n2018-12-01,0.970000000,0.790000000,-0.330000000,0.400000000\n2019-01-01,0.510000000,0.670000000,-0.170000000,0.410000000\n2019-02-01,0.620000000,0.660000000,-0.040000000,0.410000000\n2019-03-01,0.810000000,-0.180000000,-0.150000000,0.670000000\n2019-04-01,0.670000000,0.360000000,-0.170000000,0.340000000\n2019-05-01,0.610000000,0.240000000,-0.080000000,0.310000000\n2019-06-01,0.660000000,-0.110000000,0.060000000,0.270000000\n2019-07-01,0.410000000,-0.360000000,0.090000000,0.240000000\n2019-08-01,0.190000000,-0.290000000,0.090000000,0.220000000\n2019-09-01,0.110000000,-0.680000000,0.200000000,0.330000000\n2019-10-01,0.550000000,-0.710000000,0.010000000,0.250000000\n2019-11-01,0.740000000,0.020000000,-0.050000000,0.870000000\n2019-12-01,0.510000000,-0.190000000,-0.040000000,0.800000000\n2020-01-01,0.640000000,-0.120000000,0.010000000,0.950000000\n2020-02-01,0.480000000,-0.030000000,0.320000000,0.570000000\n2020-03-01,0.360000000,-0.440000000,0.580000000,0.310000000\n2020-04-01,0.490000000,0.030000000,0.460000000,0.410000000\n2020-05-01,-0.120000000,-0.240000000,0.320000000,0.530000000\n2020-06-01,-0.210000000,-0.430000000,0.470000000,0.290000000\n2020-07-01,-0.040000000,-1.090000000,0.380000000,0.290000000\n2020-08-01,-0.420000000,-0.430000000,0.250000000,0.300000000\n2020-09-01,-0.660000000,-0.850000000,0.150000000,0.090000000\n', 'research/lagged_sources/official_hashes.json': '[\n  {\n    "nome": "treino_tp.nc",\n    "sha256": "012bebbe0e38aefdf4f9e2432c3ca22f71d94ea1a1b5421da9d9e5048632cab0",\n    "bytes": 186685299\n  },\n  {\n    "nome": "treino_tp_alvo.nc",\n    "sha256": "915f67f2627e37399830d1dd92b209ce1160082b0645662675eab17df957abe4",\n    "bytes": 186509303\n  },\n  {\n    "nome": "teste_features.nc",\n    "sha256": "70bcb5c9bb9c011c8901660834b42dc57ff7e54a4f34161d489ec5e9f05c1f23",\n    "bytes": 39057097\n  },\n  {\n    "nome": "sample_submission.csv",\n    "sha256": "34fe449ddb309b0579a87786f76049863833d82c26f183d18e943dc76f1e4c8c",\n    "bytes": 48145117\n  },\n  {\n    "nome": "treino_cloud_cover.nc",\n    "sha256": "607ee1210615a0ca8a5c954737617e668a5f016eb881554cb1468fda74b4fde1",\n    "bytes": 190610590\n  },\n  {\n    "nome": "treino_geopotential_850.nc",\n    "sha256": "ca3baca0177e2e90b42901803f348a4b4affe88bee202b76af3f293fa7035c17",\n    "bytes": 159707312\n  },\n  {\n    "nome": "treino_rel_hum_850.nc",\n    "sha256": "90f62b44b1b461259e7539249914e26cf09993f3bcd572ba4871931e45264b19",\n    "bytes": 180398147\n  },\n  {\n    "nome": "treino_shum_850.nc",\n    "sha256": "35cf8071b68d1e924ad87e0f8bd801d0a66d1d78ae75fc3cb623bde85c3f47bb",\n    "bytes": 175651693\n  },\n  {\n    "nome": "treino_surface_pressure.nc",\n    "sha256": "f00a5e44a03ea923ac0f693b975ef71cd347f8ad3d07692c602e3ac9a25c10fb",\n    "bytes": 160228025\n  },\n  {\n    "nome": "treino_t2.nc",\n    "sha256": "7d8c53b438289cddf87a11414ad10c8167f52ff33c6eff459a3288975079ef23",\n    "bytes": 175952066\n  },\n  {\n    "nome": "treino_temperature_850.nc",\n    "sha256": "1db7b4fdeba87b6a25f7fb6f1e40abec50964234dd9cc918cd34f902b8e41bac",\n    "bytes": 172474226\n  },\n  {\n    "nome": "treino_u_850.nc",\n    "sha256": "73c898f27bea25cf2ab0914bbe5c29acc9794a93f8b6b3affa39854bebccf761",\n    "bytes": 188451602\n  },\n  {\n    "nome": "treino_v_850.nc",\n    "sha256": "7bab583972e5718bb0e0fe0fe6a056bd56c968af1f1497205182f7621e26c241",\n    "bytes": 197354831\n  }\n]', 'research/sst_extension/__init__.py': '"""Fixed SST/PCA contribution on an additional retrospective time block."""\n', 'research/sst_extension/data.py': '"""Reuse the archived input bytes and training calendar without neural dependencies."""\nfrom pathlib import Path\nfrom types import SimpleNamespace\nimport importlib.metadata\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nfrom research.common.inputs import ROOT, library, require, sha256, write_json, find_unique, preflight\n\nINDEX = \'competition/metadata/NOAA/indices_noaa.csv\'\nINDEX_SHA256 = \'b6f444fbec48681c1e5b46a5c22aa197ad91c7fbbdaff82c3ed3a9c63f7c15a4\'\n\n\ndef calendar(plan):\n    train = pd.date_range(plan[\'training_start\'], plan[\'training_end\'], freq=\'MS\')\n    targets = pd.date_range(plan[\'evaluation_start\'], plan[\'evaluation_end\'], freq=\'MS\')\n    require(len(train) == 360 and len(targets) == 24, \'Expected 360 fitting and 24 evaluation months.\')\n    require(train[-1] == targets[0] - pd.DateOffset(months=4), \'The four-month label gap changed.\')\n    return train, targets\n\n\ndef index_snapshot(m, train, targets):\n    path = ROOT / INDEX\n    require(sha256(path) == INDEX_SHA256, \'Archived NOAA snapshot changed.\')\n    full = pd.read_csv(path, parse_dates=[\'time_origem\']).set_index(\'time_origem\')\n    old = pd.read_csv(ROOT / \'research/lagged_sources/ocean_indices.csv\', parse_dates=[\'time_origem\']).set_index(\'time_origem\')\n    shared = old.index.intersection(full.index)\n    # The full archive starts five months later in 1976; none are needed by this 1990+ window.\n    require(len(shared) == 526 and full.columns.equals(old.columns)\n            and np.array_equal(full.loc[shared].values, old.loc[shared].values), \'NOAA overlap differs.\')\n    m.conferir_indices(full)\n    required = pd.date_range(train[0] - pd.DateOffset(months=3), targets[-1] - pd.DateOffset(months=3), freq=\'MS\')\n    require(required.isin(full.index).all(), \'Missing NOAA source months; no interpolation allowed.\')\n    return full.loc[required].copy(), dict(sha256=INDEX_SHA256, shared_months=len(shared), overlap_exact=True,\n             required_start=str(required[0].date()), required_end=str(required[-1].date()),\n             older_unshared_months=old.index.difference(full.index).strftime(\'%Y-%m\').tolist(),\n             all_required_months_present=True)\n\n\ndef check(plan, official=None, seas5=None, cfsv2=None, sst=None):\n    paths, audit = preflight(official, seas5, cfsv2, final=True)\n    pca_versions = {name: importlib.metadata.version(name) for name in [\'scikit-learn\', \'scipy\']}\n    require(pca_versions == {\'scikit-learn\': \'1.6.1\', \'scipy\': \'1.16.3\'},\n            \'Use the archived PCA environment: scikit-learn 1.6.1, scipy 1.16.3. Found \' + str(pca_versions))\n    audit[\'pca_versions\'] = pca_versions\n    m = library()\n    train, targets = calendar(plan)\n    _, audit[\'indices\'] = index_snapshot(m, train, targets)\n    paths[\'sst\'] = find_unique(m.SST_NOME, sst) / m.SST_NOME\n    require(sha256(paths[\'sst\']) == m.SST_HASH,\n            \'SST snapshot differs. Attach ersstv5_4graus_198201_202411.nc from the original SST preparation.\')\n    field, metadata = m.sst_auditar_arquivo(paths[\'sst\'])\n    needed = m.origem_mensal(train.append(targets), 2)\n    require(needed.isin(pd.DatetimeIndex(field.time.values)).all(), \'Incomplete SST origins.\')\n    audit.update(sst=metadata, sst_used=True, new_download=False)\n    print(\'Input files, versions, NOAA overlap and SST snapshot checked. No models fitted.\', flush=True)\n    return paths, audit\n\n\ndef load(paths, audit, output, plan):\n    train, targets = calendar(plan)\n    m = library()\n    m.PASTA, m.PASTA_SEAS5_MANUAL, m.PASTA_CFSV2_MANUAL = paths[\'official\'], paths[\'seas5\'], paths[\'cfsv2\']\n    m.PASTA_SEAS5, m.MANIFESTO_SEAS5 = m.localizar_seas5()\n    m.PASTA_CFSV2, m.MANIFESTO_CFSV2 = m.localizar_cfsv2()\n    m.SAIDA = Path(output)\n    # Decode only training rainfall. Evaluation labels are opened separately after freezing.\n    with xr.open_dataset(paths[\'official\'] / \'treino_tp.nc\') as ds:\n        require(ds.tp.attrs.get(\'units\') == \'mm/day\', \'Unexpected rainfall units.\')\n        rain = ds.tp.sel(time=train).transpose(\'time\', \'lat\', \'lon\').load()\n    require(pd.DatetimeIndex(rain.time.values).equals(train), \'Training rainfall dates changed.\')\n    require(np.array_equal(rain.lat, np.arange(-60, 15.25, .25))\n            and np.array_equal(rain.lon, np.arange(-90, -24.75, .25)), \'Different rainfall grid.\')\n    require(np.isfinite(rain.values).all() and (rain.values >= 0).all(), \'Invalid rainfall.\')\n    atmosphere, _ = m.carregar_atmosfera(rain, train[0] - pd.DateOffset(months=4), targets[-1] - pd.DateOffset(months=4))\n    m.INDICES_OC, indices = index_snapshot(m, train, targets)\n    write_json(Path(output) / \'indices.json\', indices)\n    phases = [\'desenvolvimento\', \'somente_ajuste_final\']\n    seas = xr.concat([m.carregar_seas5(p, rain) for p in phases], dim=\'time\').sel(time=slice(train[0], targets[-1]))\n    cfs = xr.concat([m.carregar_cfsv2(p, rain) for p in phases], dim=\'time\').sel(time=slice(train[0], targets[-1]))\n    sst, _ = m.sst_auditar_arquivo(paths[\'sst\'])\n    sst = sst.sel(time=slice(None, targets[-1] - pd.DateOffset(months=2)))\n    audit = dict(audit, rainfall_decoded_for_fit=[str(train[0].date()), str(train[-1].date())],\n                 evaluation_rainfall_decoded=False, test_partition_read=False)\n    return SimpleNamespace(lib=m, rain=rain, atmosphere=atmosphere, seas=seas, cfs=cfs, audit=audit), sst\n', 'research/sst_extension/experiment.py': '"""One paired, fixed SST experiment; freeze both predictions before opening labels."""\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom time import perf_counter\nimport gc\nimport hashlib\nimport html\nimport importlib.metadata\nimport json\nimport platform\nimport shutil\nimport zipfile\nimport numpy as np\nimport pandas as pd\nfrom research.common.presentation import display_frame\nimport xarray as xr\nfrom research.common.inputs import ROOT, library, require, sha256, write_json\nfrom .data import calendar, check, load, INDEX\n\nPROTOCOL = Path(__file__).with_name(\'protocol.json\')\nREFERENCE = \'research/temporal_extension/evidence/kaggle_354026629/evaluation/global.csv\'\nMODELS = (\'hybrid\', \'hybrid_sst\', \'climatology\')\n\n\ndef utcnow():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef protocol():\n    plan = json.loads(PROTOCOL.read_text())\n    require(plan[\'id\'] == \'sst_temporal_extension_2021_2022_v1\', \'Unexpected SST protocol.\')\n    require([plan[k] for k in [\'training_start\', \'training_end\', \'evaluation_start\', \'evaluation_end\']]\n            == [\'1990-10-01\', \'2020-09-01\', \'2021-01-01\', \'2022-12-01\'], \'Experiment calendar changed.\')\n    require(plan[\'weights\'] == [.375, .375, .25] and plan[\'residual_scales\'] == [.9, .875]\n            and plan[\'pca\'][\'components\'] == 8 and not plan[\'weight_search\']\n            and not plan[\'hyperparameter_search\'], \'Keep the prespecified candidate fixed.\')\n    require([plan[k] for k in [\'trees\', \'leaves\', \'learning_rate\', \'seed\', \'sampled_points_per_month\', \'ridge_alpha\']]\n            == [300, 31, .05, 42, 5000, .1], \'Recorded training parameters changed.\')\n    require(plan[\'lags\'] == dict(atmosphere=4, indices=3, sst=2, seasonal_initialization=1, training_rainfall=4)\n            and plan[\'pca\'] == dict(components=8, solver=\'full\', area_weight=\'sqrt(cos(latitude))\',\n                                   standardize_variance=False, fit_mask_climatology_center_on_training_only=True),\n            \'Recorded source lags or PCA preprocessing changed.\')\n    m = library()\n    require((m.ARVORES, m.SEMENTE, m.PONTOS_POR_MES, m.ANOS_TREINO, m.MOS_ALPHA, m.SST_COMPONENTES)\n            == (300, 42, 5000, 30, .1, 8), \'Training configuration changed.\')\n    expected = {\'objective\': \'regression\', \'metric\': \'rmse\', \'learning_rate\': .05, \'num_leaves\': 31,\n                \'min_data_in_leaf\': 150, \'lambda_l2\': 10., \'feature_fraction\': .9,\n                \'bagging_fraction\': .8, \'bagging_freq\': 1, \'num_threads\': 4, \'seed\': 42,\n                \'deterministic\': True, \'force_col_wise\': True, \'verbosity\': -1}\n    require(m.PARAMETROS == expected and (m.LAG_ATMOSFERA, m.LAG_INDICES, m.LAG_SST, m.LAG_CHUVA_TREINO)\n            == (4, 3, 2, 4), \'Fixed parameters or lags changed.\')\n    return plan\n\n\ndef source_files():\n    names = [INDEX, REFERENCE, \'research/common/inputs.py\', \'research/common/presentation.py\', \'research/lagged_sources/library.py\',\n             \'research/lagged_sources/ocean_indices.csv\', \'research/lagged_sources/official_hashes.json\',\n             \'research/sst_extension/protocol.json\']\n    names += [p.relative_to(ROOT).as_posix() for p in Path(__file__).parent.glob(\'*.py\')]\n    return sorted(set(names))\n\n\ndef array_hash(value):\n    return hashlib.sha256(np.ascontiguousarray(value).tobytes()).hexdigest()\n\n\ndef fit_pair(data, sst, cutoff, targets, folder):\n    """Shared sampling and ridge; eight PCs are the only added predictors."""\n    folder = Path(folder)\n    folder.mkdir(parents=True, exist_ok=True)\n    started = perf_counter()\n    m = data.lib\n    train = data.rain.sel(time=slice(None, cutoff))\n    X, y, ids, dates, climate, ca, _, _, cs, cc = m.amostrar_arvores_multissistema(\n        train, data.atmosphere, data.seas, data.cfs, cutoff)\n    require(dates[-1] == cutoff and dates.max() <= targets[0] - pd.DateOffset(months=4), \'Training crosses the label cutoff.\')\n    before = dict(X=array_hash(X), y=array_hash(y), ids=array_hash(ids))\n    pca = m.sst_ajustar_pca(sst, dates)\n    train_pc, forecast_pc = m.sst_transformar(sst, pca, dates), m.sst_transformar(sst, pca, targets)\n    np.savez_compressed(folder / \'pca.npz\', **pca)\n    np.savez_compressed(folder / \'sample.npz\', ids=ids, time=dates.values)\n    np.savez_compressed(folder / \'climate_atmosphere_indices.npz\', **ca)\n    for name, field in [(\'rainfall\', climate), (\'seas5\', cs), (\'cfsv2\', cc)]:\n        field.to_netcdf(folder / (\'climate_\' + name + \'.nc\'))\n    pd.DataFrame(forecast_pc, index=targets, columns=m.SST_FEATURES).to_csv(folder / \'forecast_pcs.csv\', index_label=\'target\')\n    ridge = m.mos_fit(train, data.seas, data.cfs, cutoff)\n    np.savez_compressed(folder / \'ridge.npz\', **ridge)\n    ridge_predictions = m.mos_predict(ridge, data.seas, data.cfs, targets)\n    timings, predictions = [], {}\n    count = 0\n    for treatment in [\'hybrid\', \'hybrid_sst\']:\n        components = []\n        for name, base, scale in [(\'arvores_seas5\', m.FEATURES_SEAS5, .9),\n                                  (\'arvores_multissistema\', m.FEATURES_MULTISSISTEMA, .875)]:\n            count += 1\n            begin = perf_counter()\n            features = list(base)\n            matrix = np.ascontiguousarray(X[:, :len(base)])\n            if treatment == \'hybrid_sst\':\n                matrix = np.column_stack([matrix, np.repeat(train_pc, ids.shape[1], axis=0)]).astype(\'float32\')\n                features += m.SST_FEATURES\n            require(np.array_equal(matrix[:, :len(base)], X[:, :len(base)]), \'Base features differ between treatments.\')\n            print(f\'[{count}/4] {treatment}: {len(features)} features, {m.ARVORES} trees; fit and inference...\', flush=True)\n            dataset = m.lgb.Dataset(matrix, label=y, feature_name=features)\n            model = m.lgb.train(m.PARAMETROS, dataset, num_boost_round=m.ARVORES)\n            require(model.feature_name() == features, \'Feature order differs.\')\n            model.save_model(str(folder / f\'{treatment}_{name}.txt\'))\n            del dataset, matrix\n            gc.collect()\n            args = (model, name, data.atmosphere, data.seas, data.cfs, ca, climate, cs, cc, targets)\n            anomalies = m.pesquisa_prever_sst(*args, forecast_pc) if treatment == \'hybrid_sst\' else m.prever_par(*args)\n            components.append(m.reconstruir_chuva(climate, anomalies, scale).values.astype(\'float64\'))\n            timings.append(dict(model=treatment, component=name, seconds=perf_counter() - begin))\n            print(f\'[{count}/4] complete in {timings[-1]["seconds"]:.1f} s.\', flush=True)\n            del model, anomalies\n        predictions[treatment] = .75 * (.5 * components[0] + .5 * components[1]) + .25 * ridge_predictions\n    require(before == dict(X=array_hash(X), y=array_hash(y), ids=array_hash(ids)), \'Shared training arrays changed.\')\n    predictions[\'climatology\'] = climate.values[targets.month - 1].astype(\'float64\')\n    coords = dict(time=targets, lat=data.rain.lat, lon=data.rain.lon)\n    ds = xr.Dataset({name: xr.DataArray(value, dims=(\'time\', \'lat\', \'lon\'), coords=coords,\n                       attrs=dict(units=\'mm/day\')) for name, value in predictions.items()},\n                    attrs=dict(training_cutoff=str(cutoff.date()), nature=\'retrospective, historical vintages unverified\'))\n    require(all(np.isfinite(ds[n].values).all() and (ds[n].values >= 0).all() for n in MODELS), \'Invalid forecasts.\')\n    ds.to_netcdf(folder / \'predictions.nc\', engine=\'h5netcdf\', encoding={n: dict(zlib=True, complevel=4) for n in MODELS})\n    with xr.open_dataset(folder / \'predictions.nc\') as saved:\n        for name in MODELS:\n            np.testing.assert_array_equal(ds[name].values, saved[name].values)\n    fit_audit = dict(training_start=str(dates[0].date()), training_end=str(dates[-1].date()),\n        months=len(dates), examples=len(y), paired_array_hashes=before, shared_ridge=True,\n        pca_training_origins=[str(pd.Timestamp(pca[\'origens_treino\'][i]).date()) for i in [0, -1]],\n        pca_components=8, pca_explained_variance_ratio=pca[\'variancia_explicada\'].tolist(),\n        parameters=m.PARAMETROS, trees=m.ARVORES, weights=[.375, .375, .25], scales=[.9, .875],\n        evaluation_labels_read=False, timings=timings, seconds=perf_counter() - started)\n    write_json(folder / \'fit.json\', fit_audit)\n    return ds, fit_audit\n\n\ndef seal(output):\n    files = {p.relative_to(output).as_posix(): sha256(p) for p in sorted(output.rglob(\'*\'))\n             if p.is_file() and p.name != \'frozen.json\' and \'__pycache__\' not in p.parts}\n    write_json(output / \'frozen.json\', dict(frozen_at=utcnow(), files=files, forecast_issued=False,\n               evaluation_labels_read=False))\n\n\ndef verify_frozen(output):\n    output = Path(output)\n    record = json.loads((output / \'frozen.json\').read_text())\n    require(\'models/predictions.nc\' in record[\'files\'] and \'signature.json\' in record[\'files\'], \'Incomplete frozen record.\')\n    for name, digest in record[\'files\'].items():\n        path = (output / name).resolve()\n        require(path.is_relative_to(output.resolve()), \'Artifact path leaves output directory.\')\n        require(path.is_file() and sha256(path) == digest, \'Changed or missing frozen artifact: \' + name)\n    return record\n\n\ndef freeze(output, official=None, seas5=None, cfsv2=None, sst=None):\n    plan = protocol()\n    paths, audit = check(plan, official, seas5, cfsv2, sst)\n    output = Path(output).resolve()\n    code = {name: sha256(ROOT / name) for name in source_files()}\n    signature = dict(plan=plan, inputs=audit, code=code, python=platform.python_version(),\n                     versions={n: importlib.metadata.version(n) for n in [\'numpy\', \'lightgbm\', \'scikit-learn\', \'scipy\', \'xarray\']})\n    if output.exists():\n        require((output / \'frozen.json\').is_file(),\n                \'An unfinished run exists. Preserve it and select a new OUTPUT directory; it is not overwritten.\')\n        verify_frozen(output)\n        require(json.loads((output / \'signature.json\').read_text()) == signature, \'Code, configuration, inputs or environment changed.\')\n        print(\'Existing frozen predictions verified; no refitting.\', flush=True)\n        return output\n    for path in [ROOT / \'research\', ROOT / \'competition\', paths[\'official\'], paths[\'seas5\'], paths[\'cfsv2\'], paths[\'sst\'].parent]:\n        require(not output.is_relative_to(path.resolve()), \'Output must be separate from source code and inputs.\')\n    output.mkdir(parents=True)\n    write_json(output / \'signature.json\', signature)\n    for name in code:\n        dest = output / \'code\' / name\n        dest.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copy2(ROOT / name, dest)\n        require(sha256(dest) == code[name], \'Code changed while snapshotting.\')\n    write_json(output / \'plan.json\', plan)\n    data, field = load(paths, audit, output, plan)\n    train, targets = calendar(plan)\n    write_json(output / \'input_audit.json\', data.audit)\n    data.lib.calendario_emissoes(targets, train[-1]).to_csv(output / \'source_calendar.csv\', index=False)\n    fit_pair(data, field, train[-1], targets, output / \'models\')\n    seal(output)\n    print(\'Both forecasts frozen. Evaluation rainfall has not been decoded.\', flush=True)\n    return output\n\n\ndef metrics(predictions, truth, m):\n    require(set(predictions.data_vars) == set(MODELS), \'Incomplete model set.\')\n    truth, predictions = xr.align(truth, predictions, join=\'exact\')\n    require(truth.dims == (\'time\', \'lat\', \'lon\') and truth.attrs.get(\'units\') == \'mm/day\', \'Unexpected observations.\')\n    require(np.isfinite(truth.values).all() and (truth.values >= 0).all(), \'Invalid observations.\')\n    for name in MODELS:\n        require(predictions[name].dims == truth.dims and predictions[name].attrs.get(\'units\') == \'mm/day\'\n                and np.isfinite(predictions[name].values).all() and (predictions[name].values >= 0).all(), \'Invalid prediction field.\')\n    detail = m.pesquisa_diagnosticos(truth, {n: predictions[n] for n in MODELS}, \'D_2021_2022\')\n    whole = detail[detail.regiao == \'dominio_inteiro\']\n    tables = dict(monthly_metrics=detail)\n    for name, frame, keys in [(\'global\', whole, [\'modelo\']), (\'years\', whole, [\'modelo\', \'ano\']),\n                               (\'regions\', detail, [\'modelo\', \'regiao\']), (\'calendar\', whole, [\'modelo\', \'mes\'])]:\n        tables[name] = m.pesquisa_resumir(frame, keys)\n    return tables\n\n\ndef score(output, official=None):\n    output = Path(output).resolve()\n    verify_frozen(output)\n    require(json.loads((output / \'plan.json\').read_text()) == protocol(), \'Scoring protocol differs.\')\n    sig = json.loads((output / \'signature.json\').read_text())\n    require(sig[\'code\'] == {name: sha256(ROOT / name) for name in source_files()}, \'Scoring code differs from frozen source.\')\n    from research.common.inputs import find_unique\n    path = find_unique(\'treino_tp.nc\', official) / \'treino_tp.nc\'\n    require(sha256(path) == sig[\'inputs\'][\'files\'][\'treino_tp.nc\'], \'Observation file differs from frozen signature.\')\n    _, targets = calendar(sig[\'plan\'])\n    report = output / \'evaluation\'\n    report.mkdir(exist_ok=True)\n    write_json(report / \'opened.json\', dict(opened_at=utcnow(), frozen_sha256=sha256(output / \'frozen.json\'),\n                retrospective=True, independent_holdout=False))\n    with xr.open_dataset(output / \'models/predictions.nc\') as ds:\n        predictions = ds.load()\n    require(pd.DatetimeIndex(predictions.time.values).equals(targets), \'Prediction calendar changed.\')\n    with xr.open_dataset(path) as ds:\n        truth = ds.tp.sel(time=targets).transpose(\'time\', \'lat\', \'lon\').load()\n    tables = metrics(predictions, truth, library())\n    for name, frame in tables.items():\n        frame.to_csv(report / (name + \'.csv\'), index=False)\n    g = tables[\'global\'].set_index(\'modelo\')\n    require((g.n == 1885464).all(), \'Incomplete full-grid evaluation.\')\n    archived = pd.read_csv(ROOT / REFERENCE).set_index(\'modelo\').loc[\'hybrid\']\n    baseline_differences = {key: float(g.loc[\'hybrid\', key] - archived[key]) for key in [\'rmse\', \'mae\', \'vies\', \'rmse_area\']}\n    reproduced = all(abs(v) < 1e-6 for v in baseline_differences.values())\n    monthly = tables[\'monthly_metrics\'].query("regiao == \'dominio_inteiro\'")\n    monthly = library().pesquisa_resumir(monthly, [\'modelo\', \'mes_alvo\']).pivot(index=\'mes_alvo\', columns=\'modelo\', values=\'rmse\')\n    delta = monthly.hybrid_sst - monthly.hybrid\n    summary = dict(status=\'complete\' if reproduced else \'baseline_mismatch\', complete_extension=True,\n        baseline_reproduced=reproduced, baseline_differences=baseline_differences,\n        delta={key: float(g.loc[\'hybrid_sst\', key] - g.loc[\'hybrid\', key]) for key in [\'rmse\', \'mae\', \'vies\', \'rmse_area\']},\n        months_better=int((delta < 0).sum()), months_worse=int((delta > 0).sum()), months_tied=int((delta == 0).sum()),\n        evaluation=[\'2021-01\', \'2022-12\'], independent_holdout=False, complete_seven_blocks=False,\n        historical_publication_vintages_verified=False, forecast_issued=False, automatic_promotion=False,\n        seconds_fit=json.loads((output / \'models/fit.json\').read_text())[\'seconds\'])\n    write_json(report / \'summary.json\', summary)\n    import matplotlib.pyplot as plt\n    fig, ax = plt.subplots(figsize=(10, 3.6), layout=\'constrained\')\n    ax.bar(range(len(delta)), delta, color=[\'#167d8d\' if v < 0 else \'#bd6745\' for v in delta])\n    ax.axhline(0, color=\'#263945\', lw=.8)\n    ax.set(xticks=list(range(0, 24, 3)), xticklabels=[t[:7] for t in delta.index[::3]],\n           ylabel=\'SST minus reference model RMSE (mm/day)\', title=\'Additional 2021–2022 block: negative values favor SST\')\n    ax.spines[[\'top\', \'right\']].set_visible(False)\n    fig.savefig(report / \'monthly_comparison.png\', dpi=150)\n    plt.close(fig)\n    text = \'<!doctype html><meta charset="utf-8"><title>SST temporal extension</title>\'\n    text += \'<style>body{font:16px system-ui;max-width:1050px;margin:36px auto;padding:0 20px;color:#213442}table{border-collapse:collapse}td,th{padding:8px;border-bottom:1px solid #ddd}pre{white-space:pre-wrap}</style>\'\n    text += \'<h1>Does global SST add information?</h1><p>Fixed 8-PC candidate, evaluated on 2021–2022 with the same samples and base inputs as the reference model. Four tree fits and one shared ridge; no neural training or parameter search.</p>\'\n    text += \'<p><strong>Retrospective extension; both years have been consulted before.</strong> Monthly lags do not verify historical product vintages. These results do not authorize operational promotion.</p>\'\n    text += \'<h2>Pooled errors, mm/day</h2>\' + display_frame(tables[\'global\'][[\'modelo\', \'rmse\', \'mae\', \'vies\', \'rmse_area\']]).to_html(index=False, float_format=lambda x: f\'{x:.6f}\')\n    text += \'<h2>Annual errors</h2>\' + display_frame(tables[\'years\'][[\'modelo\', \'ano\', \'rmse\', \'mae\', \'vies\']]).to_html(index=False, float_format=lambda x: f\'{x:.6f}\')\n    text += \'<h2>Audit and decision inputs</h2><pre>\' + html.escape(json.dumps(summary, indent=2)) + \'</pre>\'\n    (report / \'report.html\').write_text(text, encoding=\'utf-8\')\n    verify_frozen(output)\n    write_json(report / \'complete.json\', dict(files={p.name: sha256(p) for p in report.iterdir()\n               if p.is_file() and p.name != \'complete.json\'}, summary=summary))\n    print(json.dumps(summary, indent=2), flush=True)\n    return summary\n\n\ndef export_reports(output):\n    output = Path(output).resolve()\n    verify_frozen(output)\n    complete = json.loads((output / \'evaluation/complete.json\').read_text())\n    for name, digest in complete[\'files\'].items():\n        path = (output / \'evaluation\' / name).resolve()\n        require(path.is_relative_to(output / \'evaluation\') and sha256(path) == digest, \'Changed evaluation artifact.\')\n    destination = output.parent / \'sst_temporal_extension_reports.zip\'\n    with zipfile.ZipFile(destination, \'w\', zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(output.rglob(\'*\')):\n            if path.is_file() and path.suffix in {\'.json\', \'.csv\', \'.html\', \'.png\', \'.py\'} and \'__pycache__\' not in path.parts:\n                archive.write(path, path.relative_to(output).as_posix())\n    return destination\n', 'research/sst_extension/protocol.json': '{\n  "id": "sst_temporal_extension_2021_2022_v1",\n  "hypothesis": "Eight global SST principal components add information to the fixed lagged hybrid.",\n  "training_start": "1990-10-01",\n  "training_end": "2020-09-01",\n  "evaluation_start": "2021-01-01",\n  "evaluation_end": "2022-12-01",\n  "training_months": 360,\n  "evaluation_months": 24,\n  "lags": {"atmosphere": 4, "indices": 3, "sst": 2, "seasonal_initialization": 1, "training_rainfall": 4},\n  "pca": {"components": 8, "solver": "full", "area_weight": "sqrt(cos(latitude))", "standardize_variance": false, "fit_mask_climatology_center_on_training_only": true},\n  "trees": 300,\n  "leaves": 31,\n  "learning_rate": 0.05,\n  "seed": 42,\n  "sampled_points_per_month": 5000,\n  "ridge_alpha": 0.1,\n  "weights": [0.375, 0.375, 0.25],\n  "residual_scales": [0.9, 0.875],\n  "paired_inputs": "Identical samples, targets, base features, climatologies and ridge. Append eight PCs only to the two tree models.",\n  "models_fixed_for_block": true,\n  "weight_search": false,\n  "hyperparameter_search": false,\n  "independent_holdout": false,\n  "prior_use": "Both years were used in earlier final fits and have already been evaluated in the hybrid/U-Net temporal extension. This is an additional retrospective check, not a fresh holdout.",\n  "historical_publication_vintages_verified": false,\n  "availability_policy": "Apply the existing monthly lags; do not infer historical publication timestamps from month labels. Actual future issuance still requires timely source receipts.",\n  "forecast_issued": false,\n  "automatic_promotion": false,\n  "primary_metric": "Pooled full-grid RMSE: SST candidate minus paired hybrid.",\n  "secondary_metrics": ["MAE", "signed bias", "area-weighted RMSE", "year", "month", "latitude band"],\n  "budget": {"lightgbm_fits": 4, "shared_ridge_fits": 1, "pca_fits": 1, "neural_fits": 0, "outer_blocks": 1}\n}\n', 'research/sst_extension/test_sst.py': '"""Small synthetic checks of chronology, paired fitting and frozen artifacts."""\nfrom pathlib import Path\nfrom contextlib import redirect_stdout\nimport io\nimport json\nimport tempfile\nimport unittest\nfrom unittest.mock import patch\nimport numpy as np\nimport pandas as pd\nimport xarray as xr\nfrom research.common.synthetic_fixture import fixture\nfrom research.common import baseline\nfrom research.common.inputs import sha256, write_json\nfrom .data import calendar, index_snapshot\nfrom .experiment import protocol, fit_pair, metrics, seal, verify_frozen, score, export_reports\n\n\ndef sst_fixture(dates):\n    rng = np.random.default_rng(861)\n    values = rng.normal(size=(len(dates), 3, 7)).astype(\'float32\')\n    values[:, 0, 0] = np.nan\n    return xr.DataArray(values, dims=(\'time\', \'lat\', \'lon\'),\n                        coords=dict(time=dates, lat=[-40, 0, 40], lon=np.arange(0, 350, 50)), attrs=dict(units=\'degC\'))\n\n\nclass SSTTests(unittest.TestCase):\n    def setUp(self):\n        self.data = fixture()\n        self.data.audit = dict(synthetic=True)\n        self.m = self.data.lib\n        self.m.ARVORES, self.m.PONTOS_POR_MES = 2, 4\n        self.m.PARAMETROS = dict(self.m.PARAMETROS, num_threads=1, min_data_in_leaf=3, num_leaves=4)\n        self.cutoff = pd.Timestamp(\'2006-09-01\')\n        self.targets = pd.date_range(\'2007-01-01\', periods=24, freq=\'MS\')\n        self.training = self.m.calendario_pareado(self.cutoff)\n        self.sst = sst_fixture(pd.DatetimeIndex(self.data.rain.time.values))\n\n    def test_protocol_calendar_uses_fixed_gap_and_30_years(self):\n        plan = protocol()\n        train, targets = calendar(plan)\n        self.assertEqual(train[0], pd.Timestamp(\'1990-10-01\'))\n        self.assertEqual(self.m.origem_mensal(train, 2)[-1], pd.Timestamp(\'2020-07-01\'))\n        self.assertEqual(self.m.origem_mensal(targets, 2)[-1], pd.Timestamp(\'2022-10-01\'))\n        self.assertFalse(plan[\'independent_holdout\'])\n        with self.assertRaises(ValueError):\n            calendar(dict(plan, training_end=\'2020-10-01\'))\n\n    def test_archived_indices_match_overlap_and_cover_extension_origins(self):\n        train, targets = calendar(protocol())\n        indices, audit = index_snapshot(self.m, train, targets)\n        self.assertEqual(audit[\'shared_months\'], 526)\n        self.assertEqual(indices.index[0], pd.Timestamp(\'1990-07-01\'))\n        self.assertEqual(indices.index[-1], pd.Timestamp(\'2022-09-01\'))\n        self.assertEqual(len(audit[\'older_unshared_months\']), 5)\n        self.assertTrue(audit[\'all_required_months_present\'])\n\n    def test_pca_mask_climate_center_components_ignore_post_training_sst(self):\n        a = self.m.sst_ajustar_pca(self.sst, self.training)\n        changed = self.sst.copy(deep=True)\n        changed.loc[dict(time=slice(\'2006-08-01\', None))] = 999.\n        changed.loc[dict(time=slice(\'2006-08-01\', None), lat=0, lon=50)] = np.nan\n        b = self.m.sst_ajustar_pca(changed, self.training)\n        for key in a:\n            np.testing.assert_array_equal(a[key], b[key])\n        self.assertEqual(a[\'contagens_mensais\'].sum(), len(self.training))\n\n    def test_t_minus_two_scores_ignore_later_sst_and_reject_missing_ocean_cells(self):\n        state = self.m.sst_ajustar_pca(self.sst, self.training)\n        targets = self.targets[:1]\n        before = self.m.sst_transformar(self.sst, state, targets)\n        changed = self.sst.copy(deep=True)\n        changed.loc[dict(time=slice(\'2006-12-01\', None))] = 300.\n        np.testing.assert_array_equal(before, self.m.sst_transformar(changed, state, targets))\n        changed.loc[dict(time=\'2006-11-01\', lat=0, lon=50)] = np.nan\n        with self.assertRaisesRegex(ValueError, \'Missing SST\'):\n            self.m.sst_transformar(changed, state, targets)\n\n    def test_paired_fit_reproduces_baseline_without_outer_rainfall(self):\n        truth = self.data.rain.sel(time=self.targets).copy()\n        truth.attrs[\'units\'] = \'mm/day\'\n        self.data.rain = self.data.rain.sel(time=slice(None, self.cutoff))\n        with tempfile.TemporaryDirectory() as temp:\n            root = Path(temp)\n            state = baseline.fit(self.data, self.cutoff, root / \'reference\')\n            reference = baseline.predict(self.data, state, self.targets)\n            with redirect_stdout(io.StringIO()):\n                predictions, audit = fit_pair(self.data, self.sst, self.cutoff, self.targets, root / \'pair\')\n            xr.testing.assert_equal(reference, predictions.hybrid.rename(None))\n            self.assertEqual(audit[\'paired_array_hashes\'][\'ids\'], state[\'audit\'][\'sample_hash\'])\n            self.assertEqual(len(audit[\'timings\']), 4)\n            self.assertFalse(audit[\'evaluation_labels_read\'])\n            with xr.open_dataset(root / \'pair/predictions.nc\') as saved:\n                xr.testing.assert_equal(predictions, saved)\n            tables = metrics(predictions, truth, self.m)\n            g = tables[\'global\'].set_index(\'modelo\')\n            for name in [\'hybrid\', \'hybrid_sst\']:\n                error = predictions[name].values - truth.values.astype(\'float64\')\n                self.assertAlmostEqual(g.loc[name, \'rmse\'], np.sqrt(np.square(error).mean()), places=12)\n                self.assertAlmostEqual(g.loc[name, \'mae\'], np.abs(error).mean(), places=12)\n            self.assertEqual(len(tables[\'monthly_metrics\']), 3 * 24 * 4)\n            with self.assertRaises(ValueError):\n                metrics(predictions, truth.assign_coords(lon=truth.lon + 1), self.m)\n\n    def test_frozen_artifact_tampering_is_rejected(self):\n        with tempfile.TemporaryDirectory() as temp:\n            root = Path(temp)\n            (root / \'models\').mkdir()\n            (root / \'models/predictions.nc\').write_bytes(b\'placeholder\')\n            (root / \'signature.json\').write_text(\'{}\')\n            seal(root)\n            self.assertFalse(verify_frozen(root)[\'evaluation_labels_read\'])\n            (root / \'models/predictions.nc\').write_bytes(b\'changed\')\n            with self.assertRaisesRegex(ValueError, \'Changed or missing\'):\n                verify_frozen(root)\n\n    def test_scoring_and_report_export_from_frozen_full_grid_synthetic_maps(self):\n        # Schema-sized constant maps exercise I/O/scoring without a full-grid model fit.\n        plan = protocol()\n        _, targets = calendar(plan)\n        coords = dict(time=targets, lat=np.arange(-60, 15.25, .25), lon=np.arange(-90, -24.75, .25))\n        def field(value):\n            return xr.DataArray(np.full((24, 301, 261), value, dtype=\'float32\'),\n                                dims=(\'time\', \'lat\', \'lon\'), coords=coords, attrs=dict(units=\'mm/day\'))\n        truth = field(2.2)\n        predictions = xr.Dataset(dict(hybrid=field(2.), hybrid_sst=field(2.1), climatology=field(1.8)))\n        with tempfile.TemporaryDirectory() as temp:\n            root = Path(temp)\n            output, official = root / \'run\', root / \'official\'\n            (output / \'models\').mkdir(parents=True)\n            official.mkdir()\n            truth.to_dataset(name=\'tp\').to_netcdf(official / \'treino_tp.nc\', engine=\'h5netcdf\')\n            predictions.to_netcdf(output / \'models/predictions.nc\', engine=\'h5netcdf\')\n            metrics(predictions, truth, self.m)[\'global\'].to_csv(root / \'reference.csv\', index=False)\n            write_json(output / \'signature.json\', dict(plan=plan, code={\'reference.csv\': sha256(root / \'reference.csv\')},\n                       inputs=dict(files={\'treino_tp.nc\': sha256(official / \'treino_tp.nc\')})))\n            write_json(output / \'models/fit.json\', dict(seconds=0))\n            write_json(output / \'plan.json\', plan)\n            seal(output)\n            with patch(\'research.sst_extension.experiment.ROOT\', root), \\\n                 patch(\'research.sst_extension.experiment.REFERENCE\', \'reference.csv\'), \\\n                 patch(\'research.sst_extension.experiment.source_files\', return_value=[\'reference.csv\']):\n                with redirect_stdout(io.StringIO()):\n                    result = score(output, official)\n                self.assertTrue(result[\'complete_extension\'] and result[\'baseline_reproduced\'])\n                self.assertEqual(result[\'months_better\'], 24)\n                self.assertTrue(export_reports(output).is_file())\n                (output / \'evaluation/global.csv\').write_text(\'tampered\')\n                with self.assertRaisesRegex(ValueError, \'Changed evaluation artifact\'):\n                    export_reports(output)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', 'research/temporal_extension/evidence/kaggle_354026629/evaluation/global.csv': 'modelo,n,sse,soma_erro,soma_erro_absoluto,sse_area,peso_area,rmse,mae,vies,rmse_area\nblend,1885464,6076965.304107503,275566.076565667,2060351.1677062258,5735361.502433309,1619418.8409421304,1.7952885294286802,1.0927555061810916,0.14615292393048449,1.8819184697356335\nclimatology,1885464,6636712.305557575,93262.72383335876,2132920.31411643,6277256.8181300685,1619418.8409421304,1.8761492047197617,1.1312442529353146,0.04946407029429295,1.9688170181230564\nhybrid,1885464,6087091.083042482,241685.94897880653,2055483.674319979,5745872.031969661,1619418.8409421304,1.7967836118898637,1.0901739170410991,0.1281838046119186,1.8836420669177203\nunet,1885464,6154974.405279987,377206.4593262484,2099750.103034948,5807365.477739566,1619418.8409421304,1.8067747114937378,1.1136516544653985,0.20006028188618208,1.8936947947541105\n'}
BUNDLE_SHA256 = {'competition/metadata/NOAA/indices_noaa.csv': 'b6f444fbec48681c1e5b46a5c22aa197ad91c7fbbdaff82c3ed3a9c63f7c15a4', 'research/common/baseline.py': 'e2d5e9b633338f2223ee0183587f4fc2b297e71ec75f96e58704a99c32065476', 'research/common/inputs.py': 'd22140d4ad3db4ac3fecdca26c7ebdd1d780486b51622a1f333fce2a637e6851', 'research/common/presentation.py': '869ff90ebd68655e72452aea103841f4e209f4f2eb6b3d32172a77d4cda228a9', 'research/common/synthetic_fixture.py': 'da4de7a662c2c8c25f324c5b543ab04914d490ffde5e316c39187eb8f1640bff', 'research/lagged_sources/library.py': '4b92dd77d868d7806fd0ae37d2837abfcde36e173df75b1bd6d88760593ee9c3', 'research/lagged_sources/ocean_indices.csv': '69226b2e65f783b4d4971a254d1e325d0908447c76ec1df3bf532ecef1f0cf5b', 'research/lagged_sources/official_hashes.json': 'bd5648197c4cf25f85d89a32c701b422f1e0b788421766cc7b8aca99ea42223d', 'research/sst_extension/__init__.py': '973626224642d88e2a717b1a8571dace2592c939d1fc9160350568b2facb1533', 'research/sst_extension/data.py': '2d4a5be5f4e35f1fa9782485f50d6be99e16653348f52b08deec584e6c38d7e9', 'research/sst_extension/experiment.py': '25fefdc4483e961d444665c2e15567ca95fd41c7e44f3f93e5f88b81c80737a3', 'research/sst_extension/protocol.json': '16d6d5a26ea621bc0a702b7b31c17b7b8f1ee0a9d9d97397196d3102f08b9be3', 'research/sst_extension/test_sst.py': 'b604b786779ccb2dc69899543c31f2a3b6212a12ee7fa6ff3e01bac13cd0c0e8', 'research/temporal_extension/evidence/kaggle_354026629/evaluation/global.csv': '044c310fe49c905b01b4963704cc550c31da6a9ac0bdaaf531361e569a378f36'}
for name, content in BUNDLE.items():
    assert hashlib.sha256(content.encode('utf-8')).hexdigest() == BUNDLE_SHA256[name]
    path = CODE_ROOT/name
    if path.exists() and path.read_text(encoding='utf-8') != content:
        raise RuntimeError('Different source code already exists. Preserve this run and choose a new RUN_ROOT.')
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        path.write_text(content, encoding='utf-8', newline='\n')
sys.path.insert(0, str(CODE_ROOT))
sys.path.insert(0, str(CODE_ROOT/'research/prospective'))
print('Reviewed source bundle:', CODE_ROOT)


## 1. Small synthetic checks
These validate the implementation on tiny grids, not forecast skill.

In [ ]:
import unittest
suite = unittest.defaultTestLoader.loadTestsFromName("research.sst_extension.test_sst")
assert unittest.TextTestRunner(verbosity=2).run(suite).wasSuccessful(), "Synthetic checks failed."


## 2. Verify inputs and freeze both forecasts
CPU is sufficient. Four fixed tree fits; no neural training or parameter search. Keep each manual path as `None` for automatic discovery, or set the directory containing the corresponding files.

In [ ]:
from research.sst_extension.experiment import freeze, score, export_reports
OFFICIAL = None
SEAS5 = None
CFSV2 = None
SST = None  # Directory containing ersstv5_4graus_198201_202411.nc
OUTPUT = RUN_ROOT / "results"
freeze(OUTPUT, official=OFFICIAL, seas5=SEAS5, cfsv2=CFSV2, sst=SST)


## 3. Open evaluation rainfall and compare
The saved predictions and source hashes are verified first. These years were previously consulted; this is not an independent holdout.

In [ ]:
import pandas as pd
from research.common.presentation import display_frame
from IPython.display import display, HTML, Image, FileLink
RESULT = score(OUTPUT, official=OFFICIAL)
for name in ["global", "years", "regions"]:
    print(name)
    display(display_frame(pd.read_csv(OUTPUT / "evaluation" / (name + ".csv"))))
display(HTML((OUTPUT / "evaluation/report.html").read_text(encoding="utf-8")))
display(Image(filename=str(OUTPUT / "evaluation/monthly_comparison.png")))
REPORT = export_reports(OUTPUT)
display(FileLink(str(REPORT.relative_to(BASE))))
print("Complete extension:", RESULT["complete_extension"])
print("Baseline reproduced:", RESULT["baseline_reproduced"])
print("Download sst_temporal_extension_reports.zip and save the complete notebook output.")
assert RESULT["baseline_reproduced"], "The reference differs: inspect the reports before interpreting any SST gain."
